# NumPy 전체 정리 · 배열 · 인덱싱 · shape

**한 줄 요약** · 리스트 대신 배열로 숫자를 한꺼번에 계산하고, 위치 · 조건으로 꺼내며, 계산 전에 결과 shape를 미리 말하는 법 — 마지막에 이미지 · 시계열을 모델 입력 모양으로 바꿈

## 🎯 학습 목표

이 노트북을 끝내면 **자료를 보지 않고** 다음을 할 수 있어야 함

1. 리스트와 배열의 `* 2` 결과 차이를 말하고, 배열을 받으면 `shape` · `dtype`부터 확인한다
2. `reshape` · 슬라이싱 결과가 원본과 데이터를 공유하는지 말하고, 필요할 때 `copy()`한다
3. 조건 mask(`&` · `|` · `~`)와 `argsort`로 원하는 값 · 순위를 꺼낸다
4. `axis` · broadcasting · `@` 규칙으로 계산 결과의 shape를 실행 전에 말한다
5. 이미지를 float32 · `(1, C, H, W)`로, 시계열을 `(샘플, 시간, 특성)` 창으로 바꾼다

## 어디에 쓰나

<table width="100%" style="width:100%; table-layout:auto">
<thead>
<tr>
<th style="text-align: left;">실제 상황</th>
<th style="text-align: left;">이 권에서 배우는 것</th>
</tr>
</thead>
<tbody>
<tr>
<td style="text-align: left;">CSV에서 읽은 숫자 열로 합계 · 비율을 낼 때</td>
<td style="text-align: left;">원소별 계산 · 통계 · dtype 확인 (1 · 5장)</td>
</tr>
<tr>
<td style="text-align: left;"><code>operands could not be broadcast together</code> 에러를 만났을 때</td>
<td style="text-align: left;">오른쪽 끝부터 shape 비교 · <code>reshape(3, 1)</code> (7장)</td>
</tr>
<tr>
<td style="text-align: left;">이미지 파일을 모델에 넣기 전</td>
<td style="text-align: left;">uint8 → float32 · HWC → CHW · 배치 축 (9장)</td>
</tr>
<tr>
<td style="text-align: left;">시계열 프로젝트의 학습 데이터를 만들 때</td>
<td style="text-align: left;">슬라이딩 윈도우 · <code>(샘플, 시간, 특성)</code> (9장)</td>
</tr>
<tr>
<td style="text-align: left;">예측 점수에서 상위 클래스를 뽑을 때</td>
<td style="text-align: left;"><code>argmax</code> · <code>argsort()[::-1][:k]</code> (5 · 6장)</td>
</tr>
</tbody>
</table>

## 학습 지도

<img src="images/np_map.png" alt="NumPy 학습 지도" width="800">

**예상 학습 시간** · 개념 2시간 + 실습 1시간 (GPU · 데이터 파일 없이 바로 실행)

## 목차

<table width="100%" style="width:100%; table-layout:auto">
<thead>
<tr>
<th style="text-align: center;">장</th>
<th style="text-align: left;">제목</th>
<th style="text-align: left;">이 장의 질문</th>
</tr>
</thead>
<tbody>
<tr>
<td style="text-align: center;">0</td>
<td style="text-align: left;">준비</td>
<td style="text-align: left;">라이브러리 · 쓰는 데이터</td>
</tr>
<tr>
<td style="text-align: center;">1</td>
<td style="text-align: left;">배열 기본</td>
<td style="text-align: left;">리스트와 배열은 계산이 어떻게 다르고, 모양 · 자료형은 어떻게 확인하나?</td>
</tr>
<tr>
<td style="text-align: center;">2</td>
<td style="text-align: left;">만들기 · 모양 바꾸기</td>
<td style="text-align: left;">규칙 배열 · 빈 표를 만들고, 같은 값의 모양만 바꾸려면?</td>
</tr>
<tr>
<td style="text-align: center;">3</td>
<td style="text-align: left;">위치로 꺼내기</td>
<td style="text-align: left;">1차원 · 2차원에서 위치 · 구간을 꺼낼 때 원본은 어떻게 되나?</td>
</tr>
<tr>
<td style="text-align: center;">4</td>
<td style="text-align: left;">조건으로 꺼내기</td>
<td style="text-align: left;">위치를 모르고 조건만 알 때 값을 어떻게 꺼내나?</td>
</tr>
<tr>
<td style="text-align: center;">5</td>
<td style="text-align: left;">한꺼번에 계산</td>
<td style="text-align: left;">반복문 없이 합계 · 평균 · 원소별 계산을 하려면?</td>
</tr>
<tr>
<td style="text-align: center;">6</td>
<td style="text-align: left;">고르기 · 정렬 · 축</td>
<td style="text-align: left;">순위를 매기고, 행마다 · 열마다 계산하려면?</td>
</tr>
<tr>
<td style="text-align: center;">7</td>
<td style="text-align: left;">shape 맞추기</td>
<td style="text-align: left;">shape가 다른 배열끼리 · 행렬끼리 계산하려면?</td>
</tr>
<tr>
<td style="text-align: center;">8</td>
<td style="text-align: left;">종합 실습</td>
<td style="text-align: left;">메뉴별 이익 · 순위 · 관리 대상을 배열 연산만으로 찾으려면?</td>
</tr>
<tr>
<td style="text-align: center;">9</td>
<td style="text-align: left;">딥러닝 데이터로 연결</td>
<td style="text-align: left;">이미지 · 시계열을 모델이 받는 모양과 자료형으로 바꾸려면?</td>
</tr>
<tr>
<td style="text-align: center;">—</td>
<td style="text-align: left;">마무리 · 부록</td>
<td style="text-align: left;">오늘 배운 것 · 기억 확인 · 오답 노트 · 면접 · 요약표 · 복습일 · 원본 대응표 · 전체 코드</td>
</tr>
</tbody>
</table>

---
## 0. 준비

모든 장에서 쓰는 라이브러리를 한 번에 불러옴. 데이터 파일은 필요 없고(모든 데이터를 코드 안에서 만듦), **셀은 위에서부터 차례로 실행**(앞 섹션의 변수를 이어서 씀).

<table width="100%" style="width:100%; table-layout:auto">
<thead>
<tr>
<th style="text-align: left;">데이터</th>
<th style="text-align: left;">내용</th>
<th style="text-align: left;">처음 나오는 곳</th>
</tr>
</thead>
<tbody>
<tr>
<td style="text-align: left;"><code>cups</code> · <code>sold</code></td>
<td style="text-align: left;">메뉴별 판매 잔 수 · 지점 3곳 × 시간대 4개 표</td>
<td style="text-align: left;">1-2</td>
</tr>
<tr>
<td style="text-align: left;"><code>months</code> · <code>visitors</code></td>
<td style="text-align: left;">1~12월 · 월별 방문자</td>
<td style="text-align: left;">3-1</td>
</tr>
<tr>
<td style="text-align: left;"><code>branches</code> · <code>timeslots</code></td>
<td style="text-align: left;">지점 이름 · 시간대 이름</td>
<td style="text-align: left;">3-4</td>
</tr>
<tr>
<td style="text-align: left;"><code>menu</code> · <code>price</code></td>
<td style="text-align: left;">메뉴 이름 · 가격</td>
<td style="text-align: left;">5-2</td>
</tr>
<tr>
<td style="text-align: left;"><code>made</code> · <code>baristas</code></td>
<td style="text-align: left;">바리스타 제조 잔 수 · 이름</td>
<td style="text-align: left;">6-2 · 6-3</td>
</tr>
<tr>
<td style="text-align: left;"><code>img</code> · <code>chw</code></td>
<td style="text-align: left;">(2, 4, 3) 가짜 RGB 이미지 · 채널 우선 (3, 2, 4)</td>
<td style="text-align: left;">9-1 · 9-2</td>
</tr>
<tr>
<td style="text-align: left;"><code>temps10</code></td>
<td style="text-align: left;">10일 기온</td>
<td style="text-align: left;">9-6</td>
</tr>
<tr>
<td style="text-align: left;">seed · 실행 장치</td>
<td style="text-align: left;">난수는 9-5의 <code>default_rng(42)</code> · <code>default_rng(0)</code>만 사용 · GPU 필요 없음</td>
<td style="text-align: left;">9-5</td>
</tr>
</tbody>
</table>

In [1]:
# 라이브러리 불러오기
import warnings                                          # 경고 메시지를 잡아 출력할 때 사용 (4-3)
import numpy as np                                       # 관례: np 로 줄여 부름
from numpy.lib.stride_tricks import sliding_window_view  # 시계열 창 자르기 (9-6)


# 버전 확인
print("NumPy 버전:", np.__version__)  # 설치된 버전 기록

NumPy 버전: 2.4.6


---
## 1. 배열 기본

> **이 장의 흐름**
>
> - 01 Python 문법에서는 리스트로 값을 묶었다
> - 숫자 계산에서는 리스트가 반복문을 요구함.
> - 계산 전용 상자인 배열과 그 모양(shape) · 종류(dtype)를 먼저 익힘

**파이프라인 위치** · ③ 전처리 — 데이터를 배열로 바꾸고 모양 · 자료형 확인

### 1-1. `A` 리스트와 배열 · `np.array()` <sub>원본 01 · 강의 Part 0</sub>

> 🎯 **리스트에 `* 2`를 하면 왜 가격이 2배가 되지 않고, 배열은 무엇이 다른가?**

💡 **쉽게 말하면**

- 리스트는 **물건을 담는 상자**라 `* 2`를 하면 상자를 하나 더 이어 붙이고, 배열은 **계산기**라 안의 숫자마다 2를 곱함.

**실행 흐름**

1. 리스트
2. 배열 변환
3. `* 2` · `+` 비교
4. 리스트 반복문과 비교

**먼저 예측** · `[4500, 5000, 5500] * 2`의 길이는? — 답은 코드 실행 결과 아래에

In [2]:
price_list = [4500, 5000, 5500]     # 가격 3개 (리스트)
price_array = np.array(price_list)  # 리스트 → 배열


# 같은 * 2, 다른 결과
print("리스트 * 2:", price_list * 2, "| 길이", len(price_list * 2))  # 리스트 반복 → 6개
print("배열 * 2  :", price_array * 2,                                # 같은 * 2를 배열에
      "| 길이", len(price_array * 2))                                # 원소마다 × 2 → 3개


# 같은 +, 다른 결과
print("리스트 + 리스트:", price_list + price_list)   # 이어 붙이기
print("배열 + 배열   :", price_array + price_array)  # 같은 위치끼리 더하기


# 리스트로 '가격 2배'를 하려면 반복문 필요
print("리스트 반복문:", [p * 2 for p in price_list])  # 하나씩 꺼내 × 2 → 새 리스트
print("자료형:", type(price_list).__name__, "→", type(price_array).__name__)  # 두 자료형 이름 비교

리스트 * 2: [4500, 5000, 5500, 4500, 5000, 5500] | 길이 6
배열 * 2  : [ 9000 10000 11000] | 길이 3
리스트 + 리스트: [4500, 5000, 5500, 4500, 5000, 5500]
배열 + 배열   : [ 9000 10000 11000]
리스트 반복문: [9000, 10000, 11000]
자료형: list → ndarray


**결과** · 예측 정답은 `6` — 리스트의 `*`는 곱하기가 아니라 반복(이어 붙이기)

- `리스트 * 2: [4500, 5000, 5500, 4500, 5000, 5500] | 길이 6` — 리스트를 두 번 이어 붙임 → 6개
- `배열 * 2  : [ 9000 10000 11000] | 길이 3` — 원소마다 × 2 → 개수 그대로 (원하던 결과)
- `배열 + 배열   : [ 9000 10000 11000]` — `+`도 리스트는 이어 붙이기, 배열은 같은 위치끼리 더하기
- `리스트 반복문: [9000, 10000, 11000]` — 리스트로는 반복문을 써야 같은 결과
- `자료형: list → ndarray` — 배열의 정식 이름은 `ndarray` (N차원 배열)

**차이 · 언제 쓰나**

<table width="100%" style="width:100%; table-layout:auto">
<thead>
<tr>
<th style="text-align: left;">방법</th>
<th style="text-align: left;">차이</th>
<th style="text-align: left;">언제 쓰나</th>
</tr>
</thead>
<tbody>
<tr>
<td style="text-align: left;"><code>list * 2</code></td>
<td style="text-align: left;">리스트를 <strong>2번 이어 붙임</strong> (개수 2배)</td>
<td style="text-align: left;">같은 목록을 반복할 때 (계산 X)</td>
</tr>
<tr>
<td style="text-align: left;"><code>array * 2</code></td>
<td style="text-align: left;"><strong>원소마다</strong> × 2 (개수 그대로)</td>
<td style="text-align: left;">가격 · 픽셀 등 숫자 전체를 계산할 때 (기본 선택)</td>
</tr>
<tr>
<td style="text-align: left;"><code>[p * 2 for p in list]</code></td>
<td style="text-align: left;">반복문으로 원소별 계산, 결과는 리스트</td>
<td style="text-align: left;">숫자가 아닌 값을 하나씩 가공할 때</td>
</tr>
<tr>
<td style="text-align: left;"><code>np.array(list)</code></td>
<td style="text-align: left;">리스트 → 배열 변환</td>
<td style="text-align: left;">계산 전에 한 번 바꿔 둘 때</td>
</tr>
</tbody>
</table>

> ⚠️ **주의**
>
> - 리스트에 `* 2` · `+`를 쓰면 에러 없이 조용히 이어 붙여짐
> - 계산 전 `type()` 확인

> ✅ **한 줄 정리** · 숫자 계산은 `np.array()`로 바꾼 뒤 하고, 배열의 연산은 원소마다 적용된다.

**딥러닝 코드에서는**

- `x = np.array(pixel_list)` → `torch.from_numpy(x)`
- 모델 입력은 리스트가 아니라 배열 · 텐서

### 1-2. `A` 모양 읽기 · `ndim` · `shape` · `size` <sub>원본 02 · 강의 Part 0</sub>

> 🎯 **배열의 차원 · 모양 · 개수는 어떻게 확인하고, 셋은 무엇이 다른가?**

💡 **쉽게 말하면**

- `shape`는 **상자의 가로 · 세로 칸 수**, `size`는 **상자 안 물건의 총개수**, `ndim`은 **방향(축)이 몇 개**인지.

<img src="images/np_ndim.png" alt="축이 늘어날수록 shape의 숫자가 늘어남" width="763">

**실행 흐름**

1. 1차원 cups · 2차원 sold
2. ndim · shape · size · len 비교
3. 3차원 ndim

**먼저 예측** · 3행 4열 표의 `len()`은 3일까 12일까? — 답은 코드 실행 결과 아래에

In [3]:
cups = np.array([320, 180, 145, 60, 95])  # 메뉴 5개의 하루 판매 잔 수
sold = np.array([[42, 88, 61, 25],        # 지점 3곳(행) × 시간대 4개(열)
                 [30, 75, 94, 48],
                 [55, 102, 47, 12]])


# 1차원 · 2차원 배열의 속성 비교
for name, arr in [("cups", cups), ("sold", sold)]:           # (이름, 배열) 짝을 하나씩 꺼냄
    print(f"{name}: ndim {arr.ndim} | shape {arr.shape} | "  # 속성 4개를 한 줄로
          f"size {arr.size} | len {len(arr)}")               # f-string 두 줄 → 한 줄로 이어짐


# 괄호 안 숫자 개수 = ndim
print("np.zeros((2, 3, 4)) →", np.zeros((2, 3, 4)).ndim, "차원")  # 3차원도 같은 규칙

cups: ndim 1 | shape (5,) | size 5 | len 5
sold: ndim 2 | shape (3, 4) | size 12 | len 3
np.zeros((2, 3, 4)) → 3 차원


**결과** · 예측 정답은 `3` — `len`은 첫 번째 축(행) 길이만 셈

- `cups: ndim 1 | shape (5,) | size 5 | len 5` — 1차원은 넷 다 5, `(5,)`의 쉼표 = 원소 1개짜리 튜플
- `sold: ndim 2 | shape (3, 4) | size 12 | len 3` — size = 3 × 4 = 12, len은 행 개수 3
- `np.zeros((2, 3, 4)) → 3 차원` — shape 괄호 안 숫자 개수 = ndim

**차이 · 언제 쓰나**

<table width="100%" style="width:100%; table-layout:auto">
<thead>
<tr>
<th style="text-align: left;">방법</th>
<th style="text-align: left;">차이</th>
<th style="text-align: left;">언제 쓰나</th>
</tr>
</thead>
<tbody>
<tr>
<td style="text-align: left;"><code>arr.ndim</code></td>
<td style="text-align: left;">축(차원)이 <strong>몇 개</strong>인가</td>
<td style="text-align: left;">1차원 · 2차원 · 이미지(3차원) 구분</td>
</tr>
<tr>
<td style="text-align: left;"><code>arr.shape</code></td>
<td style="text-align: left;">축마다 <strong>길이</strong> (튜플)</td>
<td style="text-align: left;">계산 · 모델 입력 전 모양 확인 (가장 자주)</td>
</tr>
<tr>
<td style="text-align: left;"><code>arr.size</code></td>
<td style="text-align: left;">값의 <strong>총 개수</strong> = shape의 곱</td>
<td style="text-align: left;">reshape 가능 여부 판단</td>
</tr>
<tr>
<td style="text-align: left;"><code>len(arr)</code></td>
<td style="text-align: left;"><strong>첫 번째 축</strong> 길이만</td>
<td style="text-align: left;">행(샘플) 개수만 필요할 때</td>
</tr>
</tbody>
</table>

> ⚠️ **주의**
>
> - `(5,)`와 `(5, 1)`은 다른 모양(1차원 vs 2차원)
> - 같은 계산도 결과가 달라짐 (7-1)

> ✅ **한 줄 정리** · `shape`는 모양, `size`는 총 개수, `ndim`은 축 개수이며, 배열을 다룰 때는 항상 `shape`부터 확인한다.

**딥러닝 코드에서는**

- `print(x.shape)` → `(32, 3, 224, 224)` = 배치 32장 · 채널 3 · 높이 224 · 너비 224

### 1-3. `A` 자료형 통일 · `dtype` <sub>원본 03 · 강의 Part 1</sub>

> 🎯 **서로 다른 종류의 값을 섞으면 배열의 자료형은 어떻게 정해지는가?**

💡 **쉽게 말하면**

- 배열은 **한 가지 규격 상자만 쓰는 창고**.
- 섞이면 모두 담을 수 있는 큰 상자(정수 → 실수 → 문자열)로 통일.

**실행 흐름**

1. 정수 · 정수+실수 · 숫자+문자열 3개 비교
2. 문자열 배열 계산 에러
3. dtype 직접 지정

**먼저 예측** · `np.array([10, 20, "30"])`의 dtype은? — 답은 코드 실행 결과 아래에

In [4]:
ints = np.array([10, 20, 30])            # 정수만
mixed_number = np.array([10, 20, 30.5])  # 정수 + 실수 하나
mixed_text = np.array([10, 20, "30"])    # 숫자 + 문자열 하나


# 섞인 값이 어느 쪽으로 통일되는지 확인
for arr in [ints, mixed_number, mixed_text]:  # 세 배열을 차례로
    print(arr, "→", arr.dtype)                # 정수 → 실수 → 문자열 방향으로 통일


# 문자열로 통일된 배열은 숫자 계산 불가
try:
    mixed_text + 1                         # 문자열 배열 + 정수
except TypeError as e:                     # 계산 에러를 잡아 이름만
    print("계산 불가:", type(e).__name__)  # UFuncTypeError = TypeError의 한 종류


# dtype을 직접 지정해서 만들기
print("dtype 지정:", np.array([1, 2, 3], dtype=np.float32).dtype)  # 딥러닝 기본 실수형

[10 20 30] → int64
[10.  20.  30.5] → float64
['10' '20' '30'] → <U21
계산 불가: UFuncTypeError
dtype 지정: float32


**결과** · 예측 정답은 문자열 `<U21` — 하나라도 문자열이면 전부 문자열

- `[10 20 30] → int64` — 정수만 → 64비트 정수
- `[10.  20.  30.5] → float64` — 실수 하나 때문에 10도 `10.`(실수)이 됨
- `['10' '20' '30'] → <U21` — `<U21` = 최대 21글자 유니코드 문자열
- `계산 불가: UFuncTypeError` — 문자열 배열 + 숫자 → TypeError 계열 에러
- `dtype 지정: float32` — `dtype=`으로 직접 정함

**차이 · 언제 쓰나**

<table width="100%" style="width:100%; table-layout:auto">
<thead>
<tr>
<th style="text-align: left;">방법</th>
<th style="text-align: left;">차이</th>
<th style="text-align: left;">언제 쓰나</th>
</tr>
</thead>
<tbody>
<tr>
<td style="text-align: left;">정수만 → <code>int64</code></td>
<td style="text-align: left;">자동 지정</td>
<td style="text-align: left;">개수 · 번호 · 라벨</td>
</tr>
<tr>
<td style="text-align: left;">실수가 하나라도 → <code>float64</code></td>
<td style="text-align: left;">정수도 실수로 바뀜 (<code>10.</code>)</td>
<td style="text-align: left;">측정값 · 평균 · 비율</td>
</tr>
<tr>
<td style="text-align: left;">문자열이 하나라도 → <code>&lt;U</code></td>
<td style="text-align: left;">전부 문자열 → <strong>계산 불가</strong></td>
<td style="text-align: left;">메뉴 이름처럼 이름표 배열만</td>
</tr>
<tr>
<td style="text-align: left;"><code>dtype=np.float32</code> 지정</td>
<td style="text-align: left;">직접 정함, float64의 절반 메모리</td>
<td style="text-align: left;">딥러닝 입력 (PyTorch 기본 실수형)</td>
</tr>
</tbody>
</table>

> ⚠️ **주의**
>
> - CSV에서 숫자 사이에 `"-"` · `"N/A"` 같은 글자가 하나만 섞여도 열 전체가 문자열 · `dtype` 먼저 확인

> ✅ **한 줄 정리** · 배열은 하나의 dtype만 가지며, 섞이면 정수 → 실수 → 문자열 방향으로 통일된다.

**딥러닝 코드에서는**

- `x = np.array(data, dtype=np.float32)`
- PyTorch 모델의 기본 실수형이 float32

### 1-4. `A` 자료형 바꾸기 · `astype()` <sub>원본 04 · 강의 Part 1</sub>

> 🎯 **실수를 정수로 바꾸면 반올림일까, 버림일까?**

💡 **쉽게 말하면**

- `astype`은 **다른 규격 상자에 옮겨 담기**.
- 정수 상자에는 소수점 칸이 없어 소수 부분을 그냥 잘라 버림.

**실행 흐름**

1. 실수
2. astype(int)
3. 원본 확인
4. np.round 후 변환
5. 음수 비교

**먼저 예측** · `26.5`를 `astype(int)`하면? — 답은 코드 실행 결과 아래에

In [5]:
temps = np.array([21.7, 24.3, 19.8, 26.5])  # 실수 온도 4개


# 정수로 바꾼 새 배열 만들기
temps_int = temps.astype(int)                  # 새 배열 반환 → 원본 유지
print("변환 전:", temps, temps.dtype)          # 원래 값 · 자료형
print("변환 후:", temps_int, temps_int.dtype)  # 소수점이 어떻게 됐나
print("원본 유지:", temps)                     # 원본이 바뀌었나 확인


# 버림 vs 반올림
print("반올림 후 변환:", np.round(temps).astype(int))  # 반올림 → 정수 (26.5는 짝수 쪽 26)


# 음수에서 버림(0 쪽)과 내림(작은 쪽)의 차이
print("음수 -1.7 → astype:", np.array([-1.7]).astype(int),  # 음수에서 버림 결과
      "| floor:", np.floor([-1.7]))

변환 전: [21.7 24.3 19.8 26.5] float64
변환 후: [21 24 19 26] int64
원본 유지: [21.7 24.3 19.8 26.5]
반올림 후 변환: [22 24 20 26]
음수 -1.7 → astype: [-1] | floor: [-2.]


**결과** · 예측 정답은 `26` — 반올림이 아니라 버림

- `변환 후: [21 24 19 26] int64` — 21.7 → 21, 19.8 → 19 (소수점 버림)
- `원본 유지: [21.7 24.3 19.8 26.5]` — astype은 새 배열 반환 → 원본 그대로
- `반올림 후 변환: [22 24 20 26]` — 26.5는 27이 아니라 26 (.5는 짝수 쪽으로)
- `음수 -1.7 → astype: [-1] | floor: [-2.]` — 버림은 0 쪽(-1), 내림은 작은 쪽(-2)

**차이 · 언제 쓰나**

<table width="100%" style="width:100%; table-layout:auto">
<thead>
<tr>
<th style="text-align: left;">방법</th>
<th style="text-align: left;">차이</th>
<th style="text-align: left;">언제 쓰나</th>
</tr>
</thead>
<tbody>
<tr>
<td style="text-align: left;"><code>arr.astype(int)</code></td>
<td style="text-align: left;"><strong>버림</strong> (0 쪽으로), 새 배열</td>
<td style="text-align: left;">소수점이 없는 값으로 확실히 바꿀 때</td>
</tr>
<tr>
<td style="text-align: left;"><code>np.round(arr).astype(int)</code></td>
<td style="text-align: left;"><strong>반올림</strong> 후 정수 (.5는 짝수 쪽)</td>
<td style="text-align: left;">보고 · 표시용 정수</td>
</tr>
<tr>
<td style="text-align: left;"><code>np.floor(arr)</code></td>
<td style="text-align: left;"><strong>내림</strong> (작은 쪽, 음수에서 차이)</td>
<td style="text-align: left;">구간 나누기 (<code>-1.7 → -2</code>)</td>
</tr>
<tr>
<td style="text-align: left;"><code>arr.astype(np.float32)</code></td>
<td style="text-align: left;">실수형으로 바꿈</td>
<td style="text-align: left;">이미지 uint8 → 계산용 실수 (9-1)</td>
</tr>
</tbody>
</table>

> ⚠️ **주의**
>
> - `np.round`는 .5를 짝수 쪽으로 보냄 (`0.5 → 0`, `1.5 → 2`, `26.5 → 26`) — 사사오입이 아님

> ✅ **한 줄 정리** · `astype(int)`는 소수점을 버리므로, 반올림이 필요하면 `np.round()`를 먼저 하고 바꾼다.

**딥러닝 코드에서는**

- `img.astype(np.float32) / 255` (9-1) · `labels.astype(np.int64)`
- 손실 함수가 요구하는 자료형 맞추기

---
## 2. 만들기 · 모양 바꾸기

> **이 장의 흐름**
>
> - 1장에서는 값을 직접 적어 배열을 만들었다
> - 규칙적인 값 · 빈 표는 NumPy가 대신 만들고, 같은 값의 모양만 바꿀 수 있음

**파이프라인 위치** · ③ 전처리 · ④ 모델 — 번호 목록, 결과를 담을 그릇, 이미지 → 1차원 특성

### 2-1. `A` 규칙 수열 만들기 · `np.arange()` <sub>원본 05 · 강의 Part 2</sub>

> 🎯 **규칙적인 숫자 배열을 만들 때 끝값은 어떻게 정하는가?**

💡 **쉽게 말하면**

- `np.arange`는 배열판 `range`.
- **시작은 포함, 끝은 제외**라서 20까지 원하면 21을 적음.

**실행 흐름**

1. 0~9 · 6~20 · 5~50(5씩)
2. 처음 · 끝 · 개수 확인
3. stop 실수
4. 실수 간격 · linspace

**먼저 예측** · `np.arange(6, 20)`의 마지막 값은? — 답은 코드 실행 결과 아래에

In [6]:
# 0~9 · 6~20 · 5~50(5씩): stop = 원하는 끝 + 1
seq_a = np.arange(10)                                                   # arange(stop) → 0부터
seq_b = np.arange(6, 21)                                                # arange(start, stop)
seq_c = np.arange(5, 51, 5)                                             # arange(start, stop, step)
for name, s in [("seq_a", seq_a), ("seq_b", seq_b), ("seq_c", seq_c)]:  # 세 배열을 이름과 함께
    print(f"{name}: 처음 {s[0]} · 끝 {s[-1]} · size {s.size}")          # 끝값 포함 여부 확인


# 흔한 실수: stop에 끝값을 그대로 쓰면
print("arange(6, 20)의 끝:", np.arange(6, 20)[-1])  # 20 미포함 → 19


# 실수 간격 · 개수 지정
print("arange(0, 1, 0.25):", np.arange(0, 1, 0.25))  # step이 실수여도 됨, 1 미포함
print("linspace(0, 1, 5)  :", np.linspace(0, 1, 5))  # 개수 5개, 1 포함

seq_a: 처음 0 · 끝 9 · size 10
seq_b: 처음 6 · 끝 20 · size 15
seq_c: 처음 5 · 끝 50 · size 10
arange(6, 20)의 끝: 19
arange(0, 1, 0.25): [0.   0.25 0.5  0.75]
linspace(0, 1, 5)  : [0.   0.25 0.5  0.75 1.  ]


**결과** · 예측 정답은 `19` — stop 20은 포함되지 않음

- `seq_b: 처음 6 · 끝 20 · size 15` — stop = 21 → 20까지 15개
- `seq_c: 처음 5 · 끝 50 · size 10` — stop = 51, step 5 → 10개
- `arange(6, 20)의 끝: 19` — 끝값을 stop에 그대로 쓰면 하나 모자람
- `arange(0, 1, 0.25): [0.   0.25 0.5  0.75]` — 실수 step 가능, 1은 미포함
- `linspace(0, 1, 5)  : [0.   0.25 0.5  0.75 1.  ]` — 개수 5개, 끝 1 포함 → 간격 0.25

**차이 · 언제 쓰나**

<table width="100%" style="width:100%; table-layout:auto">
<thead>
<tr>
<th style="text-align: left;">방법</th>
<th style="text-align: left;">차이</th>
<th style="text-align: left;">언제 쓰나</th>
</tr>
</thead>
<tbody>
<tr>
<td style="text-align: left;"><code>range(a, b)</code></td>
<td style="text-align: left;">Python 내장, 정수만, 배열 아님</td>
<td style="text-align: left;"><code>for</code> 반복 횟수</td>
</tr>
<tr>
<td style="text-align: left;"><code>np.arange(a, b, step)</code></td>
<td style="text-align: left;">배열, <strong>stop 미포함</strong>, 실수 step 가능</td>
<td style="text-align: left;">간격이 정해진 수열</td>
</tr>
<tr>
<td style="text-align: left;"><code>np.linspace(a, b, n)</code></td>
<td style="text-align: left;"><strong>개수 n</strong> 지정, <strong>stop 포함</strong></td>
<td style="text-align: left;">0~1을 n등분 · 그래프 x축</td>
</tr>
</tbody>
</table>

> ⚠️ **주의**
>
> - 실수 step(`0.1`)은 계산 오차로 끝값 포함 여부가 흔들림
> - 실수 구간은 `linspace`

> ✅ **한 줄 정리** · `np.arange(start, stop, step)`의 stop은 포함되지 않으므로 원하는 끝값 + 1을 적는다.

**딥러닝 코드에서는**

- `np.arange(len(dataset))`
- 샘플 번호 목록을 만든 뒤 섞어서 분할 (9-5)

### 2-2. `B` 빈 표 만들기 · `np.zeros()` · `np.ones()` <sub>원본 06 · 강의 Part 2</sub>

> 🎯 **값을 채우기 전에 원하는 모양의 빈 표를 먼저 만들려면?**

💡 **쉽게 말하면**

- **빈 성적표 양식을 먼저 인쇄**해 두는 것.
- 칸 수(shape)만 정하면 0이나 1로 채워 줌.

**실행 흐름**

1. zeros(4, 3) 정수 · ones(2, 5)
2. shape · dtype
3. 괄호 실수
4. full · zeros_like

**먼저 예측** · `np.zeros(4, 3)`처럼 괄호를 하나만 쓰면? — 답은 코드 실행 결과 아래에

In [7]:
# 4주(행) × 3지점(열) 정수 0 표 · 2행 5열 1.0 표
booking = np.zeros((4, 3), dtype=int)       # shape는 괄호로 묶어 하나로 전달
ones = np.ones((2, 5))                      # dtype 생략 → float64
print("zeros:\n", booking)                  # 빈 표 모양 눈으로
print("shape:", booking.shape, ones.shape,  # 두 표의 모양
      "| dtype:", booking.dtype, ones.dtype)


# 괄호를 하나만 쓰면 에러
try:
    np.zeros(4, 3)          # 두 번째 자리 3을 dtype으로 읽음
except TypeError as e:      # 괄호 실수 에러 잡기
    print("TypeError:", e)  # 에러 문장 그대로


# 원하는 값 · 다른 배열과 같은 모양으로 채우기
print("full:", np.full(3, 7),                      # 7로 채운 3칸
      "| zeros_like(cups):", np.zeros_like(cups))  # 1-2 cups와 같은 shape

zeros:
 [[0 0 0]
 [0 0 0]
 [0 0 0]
 [0 0 0]]
shape: (4, 3) (2, 5) | dtype: int64 float64
TypeError: Cannot interpret '3' as a data type
full: [7 7 7] | zeros_like(cups): [0 0 0 0 0]


**결과** · 예측 정답은 `TypeError` — 두 번째 자리 `3`을 dtype으로 읽음

- `shape: (4, 3) (2, 5) | dtype: int64 float64` — dtype을 안 주면 기본은 실수 float64
- `TypeError: Cannot interpret '3' as a data type` — shape는 반드시 `(4, 3)` 하나로 묶어 전달
- `full: [7 7 7] | zeros_like(cups): [0 0 0 0 0]` — 원하는 값으로 채우기 · cups와 같은 모양

**차이 · 언제 쓰나**

<table width="100%" style="width:100%; table-layout:auto">
<thead>
<tr>
<th style="text-align: left;">방법</th>
<th style="text-align: left;">차이</th>
<th style="text-align: left;">언제 쓰나</th>
</tr>
</thead>
<tbody>
<tr>
<td style="text-align: left;"><code>np.zeros(shape)</code></td>
<td style="text-align: left;">0으로 채움 (기본 float64)</td>
<td style="text-align: left;">결과를 담을 빈 그릇 · 누적 합계</td>
</tr>
<tr>
<td style="text-align: left;"><code>np.ones(shape)</code></td>
<td style="text-align: left;">1로 채움</td>
<td style="text-align: left;">곱셈 초기값 · 마스크 전부 켜기</td>
</tr>
<tr>
<td style="text-align: left;"><code>np.full(shape, 값)</code></td>
<td style="text-align: left;"><strong>원하는 값</strong>으로 채움</td>
<td style="text-align: left;">기본값 · 패딩 값 채우기</td>
</tr>
<tr>
<td style="text-align: left;"><code>np.zeros_like(arr)</code></td>
<td style="text-align: left;">다른 배열과 <strong>같은 shape · dtype</strong></td>
<td style="text-align: left;">입력과 같은 모양의 결과 배열</td>
</tr>
</tbody>
</table>

> ⚠️ **주의**
>
> - `np.empty()`는 0으로 채우지 않고 메모리에 남은 값이 그대로 들어감
> - 초기화가 필요하면 `zeros`

> ✅ **한 줄 정리** · 빈 표는 `np.zeros((행, 열))`로 만들고, shape는 괄호로 한 번 더 묶는다.

**딥러닝 코드에서는**

- `preds = np.zeros(len(test_set))`에 배치별 예측을 채워 넣기 · `mask = np.ones((H, W))`

### 2-3. `A` 모양 바꾸기와 원본 · `reshape()` <sub>원본 07 · 강의 Part 3</sub>

> 🎯 **`reshape`는 원본을 바꾸는가, 새 배열을 만드는가?**

💡 **쉽게 말하면**

- `reshape`는 **같은 구슬 12개를 다른 판에 다시 꽂는 것**.
- 판(모양)은 새것이지만 구슬(데이터)은 원본과 같은 구슬.

<img src="images/np_reshape.png" alt="reshape는 값 순서를 그대로 두고 모양만 바꿈" width="759">

**실행 흐름**

1. 12개
2. 3×4 · 4×3 · 2×6
3. 바꾼 쪽 수정
4. 원본 확인
5. copy · flatten · ravel

**먼저 예측** · `r34 = base.reshape(3, 4)` 뒤 `r34[0, 0] = 99` 하면 `base[0]`은? — 답은 코드 실행 결과 아래에

In [8]:
base = np.arange(12)  # 값 12개 (0 ~ 11)


# 같은 12개를 세 가지 모양으로
r34 = base.reshape(3, 4)                                      # 12 = 3 × 4
r43 = base.reshape(4, 3)                                      # 12 = 4 × 3
r26 = base.reshape(2, 6)                                      # 12 = 2 × 6
print("3행 4열:\n", r34)                                      # 값 순서 그대로 채워짐
print("shape:", base.shape, r34.shape, r43.shape, r26.shape,  # 모양만 다르고 개수 같음
      "| size:", r34.size, r43.size, r26.size)


# reshape 결과를 고치면 원본 값도 바뀜 (모양만 다르고 데이터는 공유)
r34[0, 0] = 99                                                # 바꾼 쪽 첫 칸 수정
print("원본 base[0]:", base[0], "| 원본 shape:", base.shape,  # 원본 값 · 모양 확인
      "| 공유:", np.shares_memory(base, r34))
base[0] = 0                                                   # 되돌리기


# 원본과 끊으려면 copy · 1차원으로 펴기
r_copy = base.reshape(3, 4).copy()                               # 모양 바꾼 뒤 복사
print("copy 공유:", np.shares_memory(base, r_copy),              # 복사본은 공유 X
      "| flatten 공유:", np.shares_memory(base, r34.flatten()),  # 항상 복사
      "| ravel 공유:", np.shares_memory(base, r34.ravel()))      # 가능하면 공유

3행 4열:
 [[ 0  1  2  3]
 [ 4  5  6  7]
 [ 8  9 10 11]]
shape: (12,) (3, 4) (4, 3) (2, 6) | size: 12 12 12
원본 base[0]: 99 | 원본 shape: (12,) | 공유: True
copy 공유: False | flatten 공유: False | ravel 공유: True


**결과** · 예측 정답은 `99` — 모양은 따로지만 데이터를 공유

- `shape: (12,) (3, 4) (4, 3) (2, 6) | size: 12 12 12` — 모양이 달라도 size는 모두 12, 원본 shape는 그대로
- `원본 base[0]: 99 | 원본 shape: (12,) | 공유: True` — reshape 결과를 고치면 원본 값도 바뀜
- `copy 공유: False | flatten 공유: False | ravel 공유: True` — copy · flatten은 독립, ravel은 공유

**차이 · 언제 쓰나**

<table width="100%" style="width:100%; table-layout:auto">
<thead>
<tr>
<th style="text-align: left;">방법</th>
<th style="text-align: left;">차이</th>
<th style="text-align: left;">언제 쓰나</th>
</tr>
</thead>
<tbody>
<tr>
<td style="text-align: left;"><code>arr.reshape(3, 4)</code></td>
<td style="text-align: left;"><strong>새 모양</strong>, 원본 shape 유지, 데이터는 <strong>공유</strong></td>
<td style="text-align: left;">모양만 바꿔 쓸 때 (기본 선택)</td>
</tr>
<tr>
<td style="text-align: left;"><code>arr.reshape(3, 4).copy()</code></td>
<td style="text-align: left;">새 모양 + <strong>독립 데이터</strong></td>
<td style="text-align: left;">바꾼 쪽을 고쳐도 원본을 지켜야 할 때</td>
</tr>
<tr>
<td style="text-align: left;"><code>arr.flatten()</code></td>
<td style="text-align: left;">1차원 · <strong>항상 복사</strong></td>
<td style="text-align: left;">펼친 값을 따로 고칠 때</td>
</tr>
<tr>
<td style="text-align: left;"><code>arr.ravel()</code></td>
<td style="text-align: left;">1차원 · 가능하면 <strong>공유</strong> (빠름)</td>
<td style="text-align: left;">읽기만 할 때</td>
</tr>
</tbody>
</table>

> ⚠️ **주의**
>
> - "원본 유지"는 **모양**만 해당
> - reshape한 배열을 고치면 원본 **값**도 바뀜

> ✅ **한 줄 정리** · `reshape`는 원본의 모양은 그대로 둔 채 같은 데이터를 새 모양으로 보여 주므로, 따로 고치려면 `.copy()`한다.

**딥러닝 코드에서는**

- `x.reshape(batch_size, -1)` (이미지 → 1차원 특성)
- PyTorch는 `x.view(...)` · `x.reshape(...)`

### 2-4. `A` 모양 계약과 자동 계산 · `reshape(-1)` <sub>원본 08 + 09 · 강의 Part 3</sub>

> 🎯 **`reshape`는 언제 실패하며, `-1`은 무엇을 대신 계산하는가?**

💡 **쉽게 말하면**

- 구슬 12개는 15칸 판(5×3)에 꽂을 수 없음.
- `-1`은 "**이 칸 수는 네가 계산해**"라는 빈칸.

**실행 흐름**

1. (5, 3) 실패
2. (6, 2) 수정
3. -1로 행 · 열 자동
4. 1차원 펴기
5. -1 두 번 에러

**먼저 예측** · 값 24개를 `reshape(-1, 6)` 하면 shape는? — 답은 코드 실행 결과 아래에

In [9]:
# 12개 → (5, 3) = 15칸: 실패 예상
try:
    base.reshape(5, 3)                # 12개를 15칸에 (실패)
except ValueError as e:               # 모양 에러를 잡아
    print("ValueError:", e)           # size 12 ≠ 5 × 3
retry = base.reshape(6, 2)            # 12 = 6 × 2 로 한 곳만 수정
print("수정 후 shape:", retry.shape)  # 고친 모양 확인


# -1: 한 자리는 NumPy가 계산
base24 = np.arange(24)                                             # 값 24개
print("reshape(-1, 6) →", base24.reshape(-1, 6).shape)             # 24 ÷ 6 = 4행
print("reshape(4, -1) →", base24.reshape(4, -1).shape)             # 24 ÷ 4 = 6열
print("reshape(-1)    →", base24.reshape(4, 6).reshape(-1).shape)  # 1차원으로 펴기


# -1을 두 번 쓰면 에러
try:
    base24.reshape(-1, -1)   # 모르는 칸 2개 (실패)
except ValueError as e:      # 모양 에러를 잡아
    print("ValueError:", e)  # 에러 문장 그대로

ValueError: cannot reshape array of size 12 into shape (5,3)
수정 후 shape: (6, 2)
reshape(-1, 6) → (4, 6)
reshape(4, -1) → (4, 6)
reshape(-1)    → (24,)
ValueError: can only specify one unknown dimension


**결과** · 예측 정답은 `(4, 6)` — 24 ÷ 6 = 4

- `ValueError: cannot reshape array of size 12 into shape (5,3)` — 12개 ≠ 5 × 3 = 15칸 (모양 계약 오류)
- `수정 후 shape: (6, 2)` — 값 개수를 유지하고 모양만 한 곳 수정
- `reshape(4, -1) → (4, 6)` — 24 ÷ 4 = 6열
- `reshape(-1)    → (24,)` — `-1` 하나 = 1차원으로 펴기
- `ValueError: can only specify one unknown dimension` — 모르는 칸이 두 개면 계산 불가

**차이 · 언제 쓰나**

<table width="100%" style="width:100%; table-layout:auto">
<thead>
<tr>
<th style="text-align: left;">방법</th>
<th style="text-align: left;">차이</th>
<th style="text-align: left;">언제 쓰나</th>
</tr>
</thead>
<tbody>
<tr>
<td style="text-align: left;"><code>reshape(6, 2)</code></td>
<td style="text-align: left;">모든 축을 직접 지정</td>
<td style="text-align: left;">모양이 정해져 있을 때</td>
</tr>
<tr>
<td style="text-align: left;"><code>reshape(-1, 6)</code></td>
<td style="text-align: left;">한 축만 <strong>자동 계산</strong></td>
<td style="text-align: left;">샘플 수는 모르고 특성 수만 알 때</td>
</tr>
<tr>
<td style="text-align: left;"><code>reshape(-1)</code></td>
<td style="text-align: left;"><strong>1차원</strong>으로 펴기</td>
<td style="text-align: left;">모델 출력 → 1차원 예측값 목록</td>
</tr>
</tbody>
</table>

> ⚠️ **주의**
>
> - 나누어떨어지지 않으면(24개를 `reshape(-1, 5)`) `-1`이어도 에러

> ✅ **한 줄 정리** · `reshape`는 size가 같을 때만 되고, `-1`은 한 자리에만 써서 나머지를 NumPy가 계산하게 한다.

**딥러닝 코드에서는**

- `x.reshape(x.shape[0], -1)`
- 배치 수는 유지하고 나머지를 한 줄로 펴서 Linear 층에 넣기

---
## 3. 위치로 꺼내기

> **이 장의 흐름**
>
> - 2장에서 원하는 모양의 배열을 만들었다
> - 그 안에서 원하는 위치 · 구간 · 표의 칸을 꺼냄.
> - NumPy 슬라이싱은 리스트와 달리 원본과 연결됨

**파이프라인 위치** · ③ 전처리 · ⑤ 학습 — 학습/검증 나누기, 이미지 자르기, 최근 n일

### 3-1. `A` 위치로 꺼내기 · `a[i]` <sub>원본 10 · 강의 Part 4</sub>

> 🎯 **사람이 세는 n번째 값은 몇 번 위치에 있는가?**

💡 **쉽게 말하면**

- 배열은 **0번부터 번호가 붙은 사물함**.
- 7월은 일곱 번째지만 사물함 번호는 6.

**실행 흐름**

1. 월 · 방문자 길이 확인
2. 1월 · 7월
3. 뒤에서 세기
4. 값 vs 길이 1 배열
5. 범위 밖 에러

**먼저 예측** · 7월 방문자는 `visitors[7]`일까 `visitors[6]`일까? — 답은 코드 실행 결과 아래에

In [10]:
months = np.arange(1, 13)                                 # 1월 ~ 12월
visitors = np.array([1820, 1650, 2010, 2240, 2380, 2510,  # 월별 방문자 12개
                     2760, 2890, 2430, 2180, 1970, 2300])
print("길이가 같은가?", months.size == visitors.size)     # 짝지어 쓰기 전 확인


# 사람이 세는 n월 → 위치 n - 1
print("1월:", visitors[0], "| 7월:", visitors[6])  # 위치 = 월 - 1


# 뒤에서 세기
print("마지막:", visitors[-1], "| 뒤에서 두 번째:", visitors[-2])  # 음수 = 뒤에서


# 값 1개 vs 길이 1 배열
print("visitors[6] shape:", visitors[6].shape,  # 값 하나의 shape
      "| visitors[6:7]:", visitors[6:7],
      visitors[6:7].shape)                      # 숫자 → 값 · 슬라이스 → 배열


# 범위 밖 번호는 에러
try:
    visitors[12]             # 위치는 0 ~ 11
except IndexError as e:      # 범위 밖 에러 잡기
    print("IndexError:", e)  # 에러 문장 그대로

길이가 같은가? True
1월: 1820 | 7월: 2760
마지막: 2300 | 뒤에서 두 번째: 1970
visitors[6] shape: () | visitors[6:7]: [2760] (1,)
IndexError: index 12 is out of bounds for axis 0 with size 12


**결과** · 예측 정답은 `visitors[6]` — 사람이 세는 순서 - 1

- `길이가 같은가? True` — 두 배열을 짝지어 쓰기 전 확인
- `1월: 1820 | 7월: 2760` — 1월 = 위치 0, 7월 = 위치 6
- `마지막: 2300 | 뒤에서 두 번째: 1970` — -1 = 12월, -2 = 11월
- `visitors[6] shape: () | visitors[6:7]: [2760] (1,)` — 숫자 인덱스는 값 1개, 슬라이스는 배열
- `IndexError: index 12 is out of bounds for axis 0 with size 12` — 길이 12 → 마지막 위치는 11

**차이 · 언제 쓰나**

<table width="100%" style="width:100%; table-layout:auto">
<thead>
<tr>
<th style="text-align: left;">방법</th>
<th style="text-align: left;">차이</th>
<th style="text-align: left;">언제 쓰나</th>
</tr>
</thead>
<tbody>
<tr>
<td style="text-align: left;"><code>visitors[6]</code></td>
<td style="text-align: left;"><strong>앞</strong>에서 센 위치, 값 1개 (shape <code>()</code>)</td>
<td style="text-align: left;">n번째 값 (사람 기준 n - 1)</td>
</tr>
<tr>
<td style="text-align: left;"><code>visitors[-1]</code></td>
<td style="text-align: left;"><strong>뒤</strong>에서 센 위치</td>
<td style="text-align: left;">마지막 값 (길이를 몰라도 됨)</td>
</tr>
<tr>
<td style="text-align: left;"><code>visitors[6:7]</code></td>
<td style="text-align: left;">길이 1인 <strong>배열</strong> (shape <code>(1,)</code>)</td>
<td style="text-align: left;">결과를 배열로 유지해야 할 때</td>
</tr>
</tbody>
</table>

> ⚠️ **주의**
>
> - 마지막 값은 `a[len(a)]`가 아니라 `a[-1]` (`IndexError`)

> ✅ **한 줄 정리** · 사람이 세는 n번째는 위치 n - 1이고, 뒤에서는 -1부터 센다.

**딥러닝 코드에서는**

- `logits[0]` (배치 첫 샘플)
- `losses[-1]` (마지막 epoch 손실)

### 3-2. `A` 구간 자르기 · `a[start:stop:step]` <sub>원본 11 · 강의 Part 4</sub>

> 🎯 **3월부터 5월까지를 자르려면 stop에 무엇을 적는가?**

💡 **쉽게 말하면**

- `[2:5]`는 **2번 사물함부터 5번 앞까지**.
- stop 자리에 사람이 세는 마지막 번호(5월 → 5)를 그대로 적으면 됨.

**실행 흐름**

1. 3~5월 · 앞 4개 · 9월~끝 · 2칸 간격
2. stop 실수
3. 마지막 3개 · 뒤집기

**먼저 예측** · `visitors[2:4]`에 5월이 포함될까? — 답은 코드 실행 결과 아래에

In [11]:
# 3~5월 · 처음 4개월 · 9월~끝 · 두 달 간격
print("3~5월    :", visitors[2:5])   # 위치 2 ~ 4 (5 미포함)
print("처음 4개월:", visitors[:4])   # start 생략 → 0부터
print("9월~12월 :", visitors[8:])    # stop 생략 → 끝까지
print("두 달 간격:", visitors[::2])  # step 2


# 흔한 실수: stop에 5월 위치(4)를 쓰면 5월이 빠짐
print("[2:4] →", visitors[2:4])  # stop 4 → 5월 빠짐


# 마지막 3개 · 뒤집기
print("마지막 3개:", visitors[-3:], "| 뒤집어 앞 3개:", visitors[::-1][:3])  # 끝 구간 · 뒤집기

3~5월    : [2010 2240 2380]
처음 4개월: [1820 1650 2010 2240]
9월~12월 : [2430 2180 1970 2300]
두 달 간격: [1820 2010 2380 2760 2430 1970]
[2:4] → [2010 2240]
마지막 3개: [2180 1970 2300] | 뒤집어 앞 3개: [2300 1970 2180]


**결과** · 예측 정답은 포함 안 됨 — 위치 2 · 3(3월 · 4월)만

- `3~5월    : [2010 2240 2380]` — 위치 2 · 3 · 4 → 3개
- `9월~12월 : [2430 2180 1970 2300]` — stop 생략 → 끝까지
- `두 달 간격: [1820 2010 2380 2760 2430 1970]` — 1 · 3 · 5 · 7 · 9 · 11월
- `[2:4] → [2010 2240]` — stop에 5월 위치 4를 쓰면 5월이 빠짐
- `마지막 3개: [2180 1970 2300] | 뒤집어 앞 3개: [2300 1970 2180]` — 같은 3개, 순서만 반대

**차이 · 언제 쓰나**

<table width="100%" style="width:100%; table-layout:auto">
<thead>
<tr>
<th style="text-align: left;">방법</th>
<th style="text-align: left;">차이</th>
<th style="text-align: left;">언제 쓰나</th>
</tr>
</thead>
<tbody>
<tr>
<td style="text-align: left;"><code>a[start:stop]</code></td>
<td style="text-align: left;">start 포함, <strong>stop 미포함</strong></td>
<td style="text-align: left;">구간 꺼내기</td>
</tr>
<tr>
<td style="text-align: left;"><code>a[:n]</code> · <code>a[-n:]</code></td>
<td style="text-align: left;">앞 n개 · 마지막 n개</td>
<td style="text-align: left;">학습/검증 나누기, 최근 n일</td>
</tr>
<tr>
<td style="text-align: left;"><code>a[::step]</code></td>
<td style="text-align: left;">step 간격</td>
<td style="text-align: left;">n칸마다 고르기</td>
</tr>
<tr>
<td style="text-align: left;"><code>a[::-1]</code></td>
<td style="text-align: left;"><strong>뒤집기</strong></td>
<td style="text-align: left;">내림차순 만들기 (6-2)</td>
</tr>
</tbody>
</table>

> ⚠️ **주의**
>
> - NumPy 슬라이싱 결과는 복사본이 아님
> - 고치면 원본도 바뀜 (3-3)

> ✅ **한 줄 정리** · `[start:stop:step]`에서 stop은 포함되지 않으므로, 사람이 세는 마지막 번호를 stop에 적는다.

**딥러닝 코드에서는**

- `X_train, X_val = X[:800], X[800:]`
- `series[-30:]` (최근 30일)

### 3-3. `A` 뷰와 복사 · `copy()` <sub>원본 12 · 강의 Part 4</sub>

> 🎯 **슬라이싱한 배열을 고치면 원본도 바뀌는가?**

💡 **쉽게 말하면**

- NumPy 슬라이싱은 **원본을 들여다보는 창문(뷰)**.
- 창문 너머 물건을 옮기면 원본 방의 물건이 옮겨짐.
- `copy()`는 **사진을 찍어 따로 가져가는 것**.

**실행 흐름**

1. copy 사본 수정
2. 슬라이스(뷰) 수정
3. 뷰로 되돌리기
4. 리스트 슬라이싱과 비교

**먼저 예측** · `view = visitors[:3]` 뒤 `view[0] = 2000` 하면 `visitors[0]`은? — 답은 코드 실행 결과 아래에

In [12]:
# 방법 1: copy()로 만든 사본 수정
safe = visitors.copy()                                   # 독립된 새 배열
safe[0] = 2000                                           # 사본만 수정
print("[copy ] 사본:", safe[0], "| 원본:", visitors[0])  # 원본이 그대로인가


# 방법 2: 슬라이싱 결과(뷰) 수정
view = visitors[:3]                                      # 복사 X → 원본의 일부를 보는 창
view[0] = 2000                                           # 뷰를 통해 수정
print("[slice] 뷰  :", view[0], "| 원본:", visitors[0])  # 원본도 바뀌었나


# 뷰로 되돌리면 원본도 되돌아감
view[0] = 1820                      # 뷰로 원래 값 복구
print("되돌린 원본:", visitors[0],  # 원본도 복구됐나
      "| view.base is visitors:", view.base is visitors)


# 리스트 슬라이싱은 복사본 (NumPy와 반대)
lst = visitors.tolist()                                    # 배열 → 리스트
part = lst[:3]                                             # 리스트 슬라이싱
part[0] = 9999                                             # 조각만 수정
print("[list ] 조각:", part[0], "| 원본 리스트:", lst[0])  # 리스트 원본은 그대로

[copy ] 사본: 2000 | 원본: 1820
[slice] 뷰  : 2000 | 원본: 2000
되돌린 원본: 1820 | view.base is visitors: True
[list ] 조각: 9999 | 원본 리스트: 1820


**결과** · 예측 정답은 `2000` — 뷰는 원본과 연결

- `[copy ] 사본: 2000 | 원본: 1820` — copy는 독립 → 원본 그대로
- `[slice] 뷰  : 2000 | 원본: 2000` — 슬라이스는 뷰 → 원본도 바뀜
- `되돌린 원본: 1820 | view.base is visitors: True` — 뷰의 `.base`가 원본 → 연결 확인
- `[list ] 조각: 9999 | 원본 리스트: 1820` — Python 리스트 슬라이싱은 복사 → 원본 그대로

**차이 · 언제 쓰나**

<table width="100%" style="width:100%; table-layout:auto">
<thead>
<tr>
<th style="text-align: left;">방법</th>
<th style="text-align: left;">차이</th>
<th style="text-align: left;">언제 쓰나</th>
</tr>
</thead>
<tbody>
<tr>
<td style="text-align: left;"><code>a[:3]</code> 슬라이싱</td>
<td style="text-align: left;"><strong>뷰</strong> (원본과 연결, 고치면 원본도 바뀜)</td>
<td style="text-align: left;">읽기만 하거나, 원본을 일부러 고칠 때</td>
</tr>
<tr>
<td style="text-align: left;"><code>a[:3].copy()</code> · <code>a.copy()</code></td>
<td style="text-align: left;"><strong>독립 복사본</strong></td>
<td style="text-align: left;">원본을 지키며 수정할 때 (기본 선택)</td>
</tr>
<tr>
<td style="text-align: left;"><code>a[[0, 1, 2]]</code> · <code>a[mask]</code></td>
<td style="text-align: left;">fancy · mask는 <strong>자동 복사</strong></td>
<td style="text-align: left;">고른 결과를 따로 다룰 때 (6-1)</td>
</tr>
<tr>
<td style="text-align: left;">Python <code>list[:3]</code></td>
<td style="text-align: left;">리스트 슬라이싱은 <strong>복사</strong></td>
<td style="text-align: left;">NumPy와 반대라 습관 주의</td>
</tr>
</tbody>
</table>

> ⚠️ **주의**
>
> - Python 리스트와 반대
> - 슬라이싱한 데이터를 전처리하다 원본까지 망가뜨리는 실수가 흔함

> ✅ **한 줄 정리** · NumPy 슬라이싱은 원본과 연결된 뷰이므로, 원본을 지키려면 `.copy()`한다.

**딥러닝 코드에서는**

- `x_aug = x.copy()` 후 증강
- PyTorch는 `tensor.clone()`
- `Dataset`에서 원본 이미지를 직접 고치지 않기

### 3-4. `A` 표에서 한 칸 · `a[행, 열]` <sub>원본 13 · 강의 Part 5</sub>

> 🎯 **표에서 원하는 한 칸은 어떤 순서로 지정하는가?**

💡 **쉽게 말하면**

- 2차원 배열은 **엑셀 표**.
- 주소는 항상 **[몇 번째 줄, 몇 번째 칸]** 순서.

**실행 흐름**

1. 행 이름 · 열 이름 확인
2. [행, 열] 3칸
3. [1][2] 두 단계 방식
4. [:][1] 함정

**먼저 예측** · `sold[:][1]`은 1번 열일까 1번 행일까? — 답은 코드 실행 결과 아래에

In [13]:
branches = np.array(["강남", "홍대", "판교"])                   # 행 이름
timeslots = np.array(["아침", "점심", "저녁", "야간"])          # 열 이름
print("sold shape:", sold.shape, "→ 행 = 지점", branches.size,  # 행 · 열 개수 = 이름 개수
      "/ 열 = 시간대", timeslots.size)


# [행, 열] 순서로 한 칸 꺼내기
print("홍대 저녁:", sold[1, 2])  # 행 1 · 열 2
print("판교 점심:", sold[2, 1])  # 행 2 · 열 1
print("강남 야간:", sold[0, 3])  # 행 0 · 열 3


# 같은 값을 두 단계로 · 순서를 바꾸면 다른 값
print("sold[1][2]:", sold[1][2], "| sold[2, 1]:", sold[2, 1])  # 에러 없이 조용히 다른 값


# 두 단계 방식의 함정: [:][1]은 열이 아니라 행
print("sold[:, 1]:", sold[:, 1],    # 1번 열 (점심)
      "| sold[:][1]:", sold[:][1])  # [:] 전체 → [1] 1번 행

sold shape: (3, 4) → 행 = 지점 3 / 열 = 시간대 4
홍대 저녁: 94
판교 점심: 102
강남 야간: 25
sold[1][2]: 94 | sold[2, 1]: 102
sold[:, 1]: [ 88  75 102] | sold[:][1]: [30 75 94 48]


**결과** · 예측 정답은 1번 **행** — `[:]`가 전체를 그대로 돌려준 뒤 `[1]`이 행을 고름

- `홍대 저녁: 94` — 행 1(홍대) · 열 2(저녁)
- `판교 점심: 102` — 행 2(판교) · 열 1(점심)
- `sold[1][2]: 94 | sold[2, 1]: 102` — 두 단계도 같은 값, 순서를 바꾸면 에러 없이 다른 값
- `sold[:, 1]: [ 88  75 102] | sold[:][1]: [30 75 94 48]` — `[:, 1]`은 점심 열, `[:][1]`은 홍대 행

**차이 · 언제 쓰나**

<table width="100%" style="width:100%; table-layout:auto">
<thead>
<tr>
<th style="text-align: left;">방법</th>
<th style="text-align: left;">차이</th>
<th style="text-align: left;">언제 쓰나</th>
</tr>
</thead>
<tbody>
<tr>
<td style="text-align: left;"><code>sold[1, 2]</code></td>
<td style="text-align: left;"><strong>한 번에</strong> 행 · 열 지정</td>
<td style="text-align: left;">2차원 한 칸 (기본 선택)</td>
</tr>
<tr>
<td style="text-align: left;"><code>sold[1][2]</code></td>
<td style="text-align: left;">행을 꺼낸 뒤 열 (두 단계), 값은 같음</td>
<td style="text-align: left;">쓰지 않음 (슬라이스와 섞으면 함정)</td>
</tr>
<tr>
<td style="text-align: left;"><code>sold[:, 1]</code></td>
<td style="text-align: left;">1번 <strong>열</strong> 전체</td>
<td style="text-align: left;">한 열(시간대) 꺼내기</td>
</tr>
<tr>
<td style="text-align: left;"><code>sold[:][1]</code></td>
<td style="text-align: left;"><code>[:]</code> 전체 → <code>[1]</code> 1번 <strong>행</strong></td>
<td style="text-align: left;">열을 의도했다면 버그</td>
</tr>
</tbody>
</table>

> ⚠️ **주의**
>
> - 행 · 열을 바꿔 써도 에러가 나지 않고 다른 값이 조용히 나옴
> - shape와 행 · 열 이름 먼저 확인

> ✅ **한 줄 정리** · 2차원 한 칸은 `배열[행, 열]` 한 벌로 꺼내고, `[행][열]` 두 단계 방식은 쓰지 않는다.

**딥러닝 코드에서는**

- `img[y, x]` (픽셀 하나, 행 = 세로 y)
- `probs[i, label]` (i번 샘플의 정답 클래스 확률)

### 3-5. `A` 직사각형 · 행 · 열 전체 · `a[:, j]` <sub>원본 14 + 15 · 강의 Part 5</sub>

> 🎯 **표의 일부를 자를 때 결과는 언제 1차원이 되고 언제 2차원으로 남는가?**

💡 **쉽게 말하면**

- 숫자 하나로 고르면 그 방향은 **한 줄로 납작해져** 사라지고, `:` · 슬라이스로 고르면 **두께가 남음**.

<img src="images/np_index2d.png" alt="숫자로 고른 축은 사라지고 슬라이스로 고른 축은 남음" width="793">

**실행 흐름**

1. 행 · 열 모두 슬라이스
2. 행 하나 · 열 하나
3. 슬라이스로 축 유지

**먼저 예측** · `sold[:, 3]`과 `sold[:, 3:4]`의 shape는? — 답은 코드 실행 결과 아래에

In [14]:
# 직사각형: 행 슬라이스 + 열 슬라이스
rect = sold[1:3, 1:3]        # 홍대·판교 × 점심·저녁
print("직사각형:\n", rect)   # 잘라낸 영역
print("shape:", rect.shape)  # 2행 2열인지


# 행 전체 · 열 전체: 숫자 하나로 고른 축은 사라짐
print("강남 전체:", sold[0, :], "| shape", sold[0, :].shape)  # 행 하나 · 열 전부
print("야간 전체:", sold[:, 3], "| shape", sold[:, 3].shape)  # 행 전부 · 열 하나
print("점심~저녁 shape:", sold[:, 1:3].shape)                 # 행 전부 · 열 슬라이스


# 슬라이스로 고르면 축이 남음
print("sold[:, 3:4] shape:", sold[:, 3:4].shape)  # (3, 1) 2차원 유지

직사각형:
 [[ 75  94]
 [102  47]]
shape: (2, 2)
강남 전체: [42 88 61 25] | shape (4,)
야간 전체: [25 48 12] | shape (3,)
점심~저녁 shape: (3, 2)
sold[:, 3:4] shape: (3, 1)


**결과** · 예측 정답은 `(3,)` vs `(3, 1)`

- `shape: (2, 2)` — 행 1:3 → 2개, 열 1:3 → 2개
- `강남 전체: [42 88 61 25] | shape (4,)` — 행을 숫자로 → 행 축 사라짐
- `야간 전체: [25 48 12] | shape (3,)` — 열을 숫자로 → 열 축 사라짐
- `점심~저녁 shape: (3, 2)` — 열을 슬라이스로 → 2차원 유지
- `sold[:, 3:4] shape: (3, 1)` — 한 열이어도 슬라이스면 2차원

**차이 · 언제 쓰나**

<table width="100%" style="width:100%; table-layout:auto">
<thead>
<tr>
<th style="text-align: left;">방법</th>
<th style="text-align: left;">차이</th>
<th style="text-align: left;">언제 쓰나</th>
</tr>
</thead>
<tbody>
<tr>
<td style="text-align: left;"><code>sold[0, :]</code></td>
<td style="text-align: left;">숫자로 고른 행 축이 <strong>사라짐</strong> → <code>(4,)</code></td>
<td style="text-align: left;">한 행을 1차원으로</td>
</tr>
<tr>
<td style="text-align: left;"><code>sold[:, 3]</code></td>
<td style="text-align: left;">열 축이 사라짐 → <code>(3,)</code></td>
<td style="text-align: left;">한 열을 1차원으로</td>
</tr>
<tr>
<td style="text-align: left;"><code>sold[:, 3:4]</code></td>
<td style="text-align: left;">슬라이스 → 축 <strong>유지</strong> <code>(3, 1)</code></td>
<td style="text-align: left;">2차원 모양을 지켜야 할 때 (7-1 broadcasting)</td>
</tr>
<tr>
<td style="text-align: left;"><code>sold[1:3, 1:3]</code></td>
<td style="text-align: left;">양쪽 슬라이스 → <strong>직사각형</strong></td>
<td style="text-align: left;">표의 일부 영역 · 이미지 자르기</td>
</tr>
</tbody>
</table>

> ⚠️ **주의**
>
> - "판교까지"는 stop에 판교 위치 2가 아니라 3 (stop 미포함)

> ✅ **한 줄 정리** · 숫자 하나로 고른 축은 사라지고 슬라이스로 고른 축은 남으므로, 결과 shape를 보고 판단한다.

**딥러닝 코드에서는**

- `img[:, 10:50, 10:50]` (CHW 이미지 가운데 자르기)
- `X[:, 0]` (첫 번째 특성 열)

---
## 4. 조건으로 꺼내기

> **이 장의 흐름**
>
> - 3장은 위치를 알 때였다
> - 실제 질문은 "2400명이 넘은 달"처럼 조건만 알려 줌.
> - 조건
> - True/False
> - 꺼내기의 두 단계

**파이프라인 위치** · ③ 전처리 · ⑥ 평가 — 이상치 제거, 클래스별 샘플, 점수 기준 필터

### 4-1. `A` 조건으로 꺼내기 · `a[mask]` <sub>원본 16 · 강의 Part 6</sub>

> 🎯 **위치를 모르고 조건만 알 때 값을 어떻게 꺼내는가?**

💡 **쉽게 말하면**

- mask는 **O · X가 적힌 출석부**.
- 같은 순서의 다른 명단(월)에 출석부를 대면 O인 사람만 남음.

<img src="images/np_mask.png" alt="조건 → True/False → True 자리만 꺼내기" width="773">

**실행 흐름**

1. 조건식
2. True/False 확인
3. 대괄호에 넣기 (월 · 방문자)
4. 검증 · 개수 · 위치

**먼저 예측** · `visitors > 2400`의 결과는 숫자일까? — 답은 코드 실행 결과 아래에

In [15]:
# 1단계: 조건식 → True/False 배열
mask = visitors > 2400  # 칸마다 True/False
print("mask:", mask)    # 조건 결과를 먼저 눈으로
print("dtype:", mask.dtype, "| shape 같음:", mask.shape == visitors.shape)  # bool · 같은 길이


# 2단계: mask를 대괄호에 넣어 True 위치만 꺼내기
print("2400명 초과한 달:", months[mask])   # 조건은 visitors, 꺼내는 건 months
print("그 달의 방문자 :", visitors[mask])  # True 자리의 값


# 검증 · 개수 · 위치
print("모두 2400 초과?", np.all(visitors[mask] > 2400),  # 꺼낸 값 검증
      "| 개수:", mask.sum(),                             # True = 1로 합계
      "| 위치:", np.where(mask)[0])                      # True의 위치

mask: [False False False False False  True  True  True  True False False False]
dtype: bool | shape 같음: True
2400명 초과한 달: [6 7 8 9]
그 달의 방문자 : [2510 2760 2890 2430]
모두 2400 초과? True | 개수: 4 | 위치: [5 6 7 8]


**결과** · 예측 정답은 숫자가 아니라 같은 길이 12의 True/False 배열

- `mask: [False False False False False  True  True  True  True False False False]` — 6~9월만 True, 길이 12
- `2400명 초과한 달: [6 7 8 9]` — 조건은 방문자로 걸고 꺼내는 건 월
- `그 달의 방문자 : [2510 2760 2890 2430]` — True 위치의 값만
- `모두 2400 초과? True | 개수: 4 | 위치: [5 6 7 8]` — True를 1로 세어 sum = 개수, where = 위치

**차이 · 언제 쓰나**

<table width="100%" style="width:100%; table-layout:auto">
<thead>
<tr>
<th style="text-align: left;">방법</th>
<th style="text-align: left;">차이</th>
<th style="text-align: left;">언제 쓰나</th>
</tr>
</thead>
<tbody>
<tr>
<td style="text-align: left;"><code>visitors &gt; 2400</code></td>
<td style="text-align: left;"><strong>True/False 배열</strong> (같은 shape)</td>
<td style="text-align: left;">조건을 먼저 눈으로 확인</td>
</tr>
<tr>
<td style="text-align: left;"><code>visitors[mask]</code></td>
<td style="text-align: left;">True 위치의 <strong>값</strong></td>
<td style="text-align: left;">조건에 맞는 데이터 꺼내기</td>
</tr>
<tr>
<td style="text-align: left;"><code>np.where(mask)[0]</code></td>
<td style="text-align: left;">True의 <strong>위치 번호</strong></td>
<td style="text-align: left;">몇 번째인지 필요할 때</td>
</tr>
<tr>
<td style="text-align: left;"><code>mask.sum()</code> · <code>np.all()</code> · <code>np.any()</code></td>
<td style="text-align: left;">개수 · 전부인가 · 하나라도인가</td>
<td style="text-align: left;">조건 검증</td>
</tr>
</tbody>
</table>

> ⚠️ **주의**
>
> - 다른 배열에 mask를 쓸 때는 길이가 같아야 함
> - 짝이 어긋나면 엉뚱한 값

> ✅ **한 줄 정리** · 조건식으로 True/False 배열(mask)을 만든 뒤 `배열[mask]`로 True 위치의 값만 꺼낸다.

**딥러닝 코드에서는**

- `X[y == 1]` (양성 샘플만)
- `boxes[scores > 0.5]` (점수 0.5를 넘는 탐지 박스만)

### 4-2. `A` 조건 묶기 · `&` · `|` · `~` <sub>원본 17 · 강의 Part 6</sub>

> 🎯 **조건 두 개를 묶을 때 왜 `and` · `or` 대신 `&` · `|`를 쓰는가?**

💡 **쉽게 말하면**

- `and`는 **질문 하나에 예 · 아니오 하나**만 답하는데, 배열은 12개에 한꺼번에 묻기 때문에 **칸마다 답하는** `&`가 필요.

**실행 흐름**

1. & 범위
2. | 양 끝
3. ~ 반대
4. and 에러
5. 괄호 없음 에러

**먼저 예측** · `visitors[visitors >= 2000 & visitors <= 2500]`은 동작할까? — 답은 코드 실행 결과 아래에

In [16]:
# 두 조건 모두: & (각 조건은 괄호로)
stable = visitors[(visitors >= 2000) & (visitors <= 2500)]    # 둘 다 True인 칸
print("2000 이상 2500 이하:", stable, "| 개수", stable.size)  # 값 · 개수


# 둘 중 하나: |
extreme = visitors[(visitors < 1900) | (visitors > 2800)]  # 하나라도 True인 칸
print("1900 미만 또는 2800 초과:", extreme)                # 양 끝 값


# 반대: ~
print("2400 초과가 아닌 달:",                   # 반대 조건 개수
      visitors[~(visitors > 2400)].size, "개")  # True ↔ False 뒤집기


# and는 배열에 쓸 수 없음
try:
    visitors[(visitors >= 2000) and (visitors <= 2500)]  # and를 배열에 (실패)
except ValueError as e:                                  # 모호함 에러 잡기
    print("and → ValueError:", e)                        # 에러 문장 그대로


# 괄호를 빼면 & 가 먼저 계산되어 에러
try:
    visitors[visitors >= 2000 & visitors <= 2500]  # 2000 & visitors 를 먼저 계산
except ValueError:                                 # 괄호 없는 식의 에러
    print("괄호 없음 → ValueError")                # 실패했음을 표시

2000 이상 2500 이하: [2010 2240 2380 2430 2180 2300] | 개수 6
1900 미만 또는 2800 초과: [1820 1650 2890]
2400 초과가 아닌 달: 8 개
and → ValueError: The truth value of an array with more than one element is ambiguous. Use a.any() or a.all()
괄호 없음 → ValueError


**결과** · 예측 정답은 에러 — `&`가 비교보다 먼저 계산되므로 괄호 필수

- `2000 이상 2500 이하: [2010 2240 2380 2430 2180 2300] | 개수 6` — 두 조건 모두 True인 6개월
- `1900 미만 또는 2800 초과: [1820 1650 2890]` — 1월 · 2월 · 8월
- `2400 초과가 아닌 달: 8 개` — 12 - 4 = 8 (4-1의 반대)
- `and → ValueError: The truth value of an array with more than one element is ambiguous.` — 배열 전체를 참 · 거짓 하나로 정할 수 없음
- `괄호 없음 → ValueError` — `2000 & visitors`가 먼저 계산됨

**차이 · 언제 쓰나**

<table width="100%" style="width:100%; table-layout:auto">
<thead>
<tr>
<th style="text-align: left;">방법</th>
<th style="text-align: left;">차이</th>
<th style="text-align: left;">언제 쓰나</th>
</tr>
</thead>
<tbody>
<tr>
<td style="text-align: left;"><code>&amp;</code> vs <code>and</code></td>
<td style="text-align: left;"><code>&amp;</code>는 <strong>원소마다</strong> 그리고, <code>and</code>는 값 하나만 판단</td>
<td style="text-align: left;">배열은 <code>&amp;</code> (<code>and</code>는 ValueError)</td>
</tr>
<tr>
<td style="text-align: left;"><code>&#124;</code> vs <code>or</code></td>
<td style="text-align: left;"><code>&#124;</code>는 원소마다 또는</td>
<td style="text-align: left;">배열은 <code>&#124;</code></td>
</tr>
<tr>
<td style="text-align: left;"><code>~</code> vs <code>not</code></td>
<td style="text-align: left;"><code>~</code>는 원소마다 True ↔ False</td>
<td style="text-align: left;">배열은 <code>~</code></td>
</tr>
<tr>
<td style="text-align: left;"><code>(a &gt;= 1) &amp; (a &lt;= 5)</code></td>
<td style="text-align: left;"><strong>괄호 필수</strong> (<code>&amp;</code>가 비교보다 먼저 계산)</td>
<td style="text-align: left;">조건 2개 이상일 때 항상</td>
</tr>
</tbody>
</table>

> ⚠️ **주의**
>
> - 조건마다 괄호가 없으면 에러 또는 다른 뜻 · 항상 `(a > 1) & (a < 5)` 형태

> ✅ **한 줄 정리** · 배열 조건은 `&` · `|` · `~`로 묶고, 각 조건은 반드시 괄호로 감싼다.

**딥러닝 코드에서는**

- `df[(df.age > 20) & (df.label == 1)]` (Pandas도 같은 규칙) · `valid = (x >= 0) & (x < W)`

### 4-3. `B` 빈 결과 · `size == 0` <sub>원본 18 · 강의 Part 6</sub>

> 🎯 **조건에 맞는 값이 하나도 없으면 에러가 나는가?**

💡 **쉽게 말하면**

- 검색 결과 0건과 같음.
- **에러 대신 빈 상자**가 오고, 빈 상자의 평균을 내면 `nan`(숫자 아님)이 조용히 퍼짐.

**실행 흐름**

1. 빈 결과 shape · size
2. 빈 배열 평균(nan + 경고)
3. size로 판단

**먼저 예측** · `visitors[visitors > 5000]`의 결과는? — 답은 코드 실행 결과 아래에

In [17]:
# 조건에 맞는 값이 없으면
no_match = visitors[visitors > 5000]  # 맞는 값이 없는 조건
print("결과:", no_match, "| shape:", no_match.shape, "| size:", no_match.size)  # 빈 배열 모양


# 빈 배열의 평균 → 경고 + nan
with warnings.catch_warnings(record=True) as caught:       # 경고를 화면 대신 목록에 모음
    warnings.simplefilter("always")                        # 같은 경고도 매번 기록
    avg = no_match.mean()                                  # 빈 배열 평균 (경고 발생)
print("빈 배열 평균:", avg, "| 경고:", caught[0].message)  # nan · 경고 문장


# 비었는지는 size로 판단
if no_match.size == 0:                            # 개수 0 = 비었음
    print("해당 조건을 만족하는 달이 없습니다.")  # 계산 대신 안내

결과: [] | shape: (0,) | size: 0
빈 배열 평균: nan | 경고: Mean of empty slice
해당 조건을 만족하는 달이 없습니다.


**결과** · 예측 정답은 에러 없이 빈 배열 `[]`, shape `(0,)`

- `결과: [] | shape: (0,) | size: 0` — 오류가 아니라 빈 배열
- `빈 배열 평균: nan | 경고: Mean of empty slice` — nan + 경고 → 계산이 이어지면 결과 전체가 nan
- `해당 조건을 만족하는 달이 없습니다.` — `size == 0`으로 미리 걸러냄

**차이 · 언제 쓰나**

<table width="100%" style="width:100%; table-layout:auto">
<thead>
<tr>
<th style="text-align: left;">방법</th>
<th style="text-align: left;">차이</th>
<th style="text-align: left;">언제 쓰나</th>
</tr>
</thead>
<tbody>
<tr>
<td style="text-align: left;"><code>arr.size == 0</code></td>
<td style="text-align: left;">원소 <strong>개수</strong>로 판단</td>
<td style="text-align: left;">빈 결과 확인 (기본 선택, 몇 차원이든 동작)</td>
</tr>
<tr>
<td style="text-align: left;"><code>len(arr) == 0</code></td>
<td style="text-align: left;">첫 축 길이로 판단</td>
<td style="text-align: left;">1차원일 때만</td>
</tr>
<tr>
<td style="text-align: left;"><code>if arr:</code></td>
<td style="text-align: left;">원소가 여러 개면 <strong>ValueError</strong></td>
<td style="text-align: left;">배열에는 쓰지 않음</td>
</tr>
<tr>
<td style="text-align: left;"><code>np.any(mask)</code></td>
<td style="text-align: left;">조건을 만족하는 값이 <strong>하나라도</strong> 있는가</td>
<td style="text-align: left;">꺼내기 전에 미리 확인</td>
</tr>
</tbody>
</table>

> ⚠️ **주의**
>
> - `nan`이 한 번 섞이면 합계 · 평균이 전부 `nan`
> - 조건으로 꺼낸 뒤 `size` 확인

> ✅ **한 줄 정리** · 조건 결과가 비어도 에러가 아니라 빈 배열이므로, 계산 전에 `size == 0`인지 확인한다.

**딥러닝 코드에서는**

- 손실이 `nan`이 되는 흔한 원인
- 빈 배치 · 0으로 나누기 · `np.isnan(x).any()`로 점검

---
## 5. 한꺼번에 계산

> **이 장의 흐름**
>
> - 4장까지는 꺼내기였다
> - 꺼낸 값을 반복문 없이 합계 · 평균 · 원소별로 계산.
> - 강의 Part 0의 처음 질문(가격 × 수량)의 답

**파이프라인 위치** · ③ 전처리 · ⑥ 평가 — 정규화, 손실 · 정확도 계산

### 5-1. `A` 통계와 위치 · `sum()` · `argmax()` <sub>원본 19 · 강의 Part 7</sub>

> 🎯 **가장 큰 값이 아니라 가장 큰 값이 **몇 월**인지 알려면?**

💡 **쉽게 말하면**

- `max`는 **1등의 점수**, `argmax`는 **1등의 번호표**.
- 번호표가 있어야 이름 · 월 같은 다른 정보를 찾음.

**실행 흐름**

1. 합계 · 평균 · 표준편차 · 최소 · 최대
2. argmax · argmin
3. 월 찾기
4. 내장 sum 함정

**먼저 예측** · `visitors.argmax()`는 2890일까 7일까? — 답은 코드 실행 결과 아래에

In [18]:
# 기본 통계
print("합계:", visitors.sum(), "| 평균:", np.round(visitors.mean(), 1),  # 합계 · 평균
      "| 표준편차:", np.round(visitors.std(), 1))
print("최소:", visitors.min(), "| 최대:", visitors.max())                # 가장 작은 · 큰 값


# 값이 아니라 위치: argmax · argmin
peak = visitors.argmax()                      # 최댓값의 위치
low = visitors.argmin()                       # 최솟값의 위치
print("최대 위치:", peak, "→", months[peak],  # 위치로 월 꺼내기
      "월 | 최소 위치:", low, "→", months[low], "월")


# 2차원에서 Python 내장 sum과의 차이
print("sum(sold):", sum(sold), "| sold.sum():", sold.sum())  # 내장 sum은 행끼리 더함

합계: 27140 | 평균: 2261.7 | 표준편차: 350.4
최소: 1650 | 최대: 2890
최대 위치: 7 → 8 월 | 최소 위치: 1 → 2 월
sum(sold): [127 265 202  85] | sold.sum(): 679


**결과** · 예측 정답은 `7` — 값이 아니라 위치 (8월)

- `합계: 27140 | 평균: 2261.7 | 표준편차: 350.4` — 평균 = 27140 ÷ 12 = 2261.67 → 소수 1자리
- `최소: 1650 | 최대: 2890` — 가장 적은 값 · 가장 많은 값
- `최대 위치: 7 → 8 월 | 최소 위치: 1 → 2 월` — 위치 7 = 8월, 위치 1 = 2월
- `sum(sold): [127 265 202  85] | sold.sum(): 679` — 내장 sum은 행끼리 더해 열 합계 4개, 메서드는 전체 하나

**차이 · 언제 쓰나**

<table width="100%" style="width:100%; table-layout:auto">
<thead>
<tr>
<th style="text-align: left;">방법</th>
<th style="text-align: left;">차이</th>
<th style="text-align: left;">언제 쓰나</th>
</tr>
</thead>
<tbody>
<tr>
<td style="text-align: left;"><code>a.max()</code> vs <code>a.argmax()</code></td>
<td style="text-align: left;">최댓<strong>값</strong> vs 최댓값의 <strong>위치</strong></td>
<td style="text-align: left;">위치로 다른 배열(월 · 이름)을 꺼낼 때 <code>arg</code></td>
</tr>
<tr>
<td style="text-align: left;"><code>a.sum()</code> · <code>np.sum(a)</code></td>
<td style="text-align: left;">같은 결과 (메서드형 · 함수형)</td>
<td style="text-align: left;">취향, 리스트에도 쓰려면 <code>np.sum</code></td>
</tr>
<tr>
<td style="text-align: left;">Python <code>sum(a)</code></td>
<td style="text-align: left;">2차원이면 <strong>행끼리 더함</strong> (전체 합 X)</td>
<td style="text-align: left;">배열에는 쓰지 않음</td>
</tr>
<tr>
<td style="text-align: left;"><code>np.round(a.mean(), 1)</code></td>
<td style="text-align: left;">평균을 자릿수 맞춰 표시</td>
<td style="text-align: left;">보고용 숫자</td>
</tr>
</tbody>
</table>

> ⚠️ **주의**
>
> - `arg`가 붙으면 값이 아니라 위치 (`argmax` · `argmin` · `argsort`)

> ✅ **한 줄 정리** · `max()`는 값, `argmax()`는 위치를 돌려주며, 위치로 다른 배열의 정보를 꺼낸다.

**딥러닝 코드에서는**

- `pred = logits.argmax(axis=1)`
- 점수가 가장 큰 **클래스 번호**가 예측값

### 5-2. `A` 원소별 계산 · `a * b` <sub>원본 20 · 강의 Part 0 · Part 7</sub>

> 🎯 **처음 질문 · 메뉴별 매출과 전체 매출을 반복문 없이 한 줄로 계산하려면?**

💡 **쉽게 말하면**

- 같은 모양의 두 배열은 **같은 자리끼리 짝을 지어** 계산.
- 숫자 하나는 **모든 자리에 똑같이** 적용.

<img src="images/np_elementwise.png" alt="같은 위치끼리 곱한 뒤 합계" width="790">

**실행 흐름**

1. 가격 × 잔 수 · 합계
2. 비율 · 10% 인상
3. 리스트 방식과 비교

**먼저 예측** · `price * 1.1`의 dtype은? — 답은 코드 실행 결과 아래에

In [19]:
menu = np.array(["아메리카노", "라떼", "콜드브루", "녹차라떼", "레모네이드"])  # 메뉴 이름 5개
price = np.array([4500, 5000, 5500, 5300, 6000])                               # 같은 순서의 가격


# 같은 shape끼리: 같은 위치끼리 계산
revenue = price * cups          # 1-2의 cups와 위치별 곱
total = revenue.sum()           # 전체 매출 하나
print("메뉴별 매출:", revenue)  # 위치별 곱 결과
print("전체 매출  :", total)    # 합계


# 배열 × 숫자: 모든 원소에 같은 계산
share = np.round(revenue / total * 100, 1)         # 모든 칸 ÷ 같은 합계
new_price = price * 1.1                            # 정수 × 실수 → 실수
print("매출 비율(%):", share)                      # 비율 확인
print("10% 인상가 :", new_price, new_price.dtype)  # 실수로 바뀐 자료형


# 리스트로 같은 계산을 하면
rev_list = [p * c                                             # 같은 계산을 반복문으로
            for p, c in zip(price.tolist(), cups.tolist())]   # 짝지어 반복
print("리스트 방식과 같은가:", rev_list == revenue.tolist())  # 두 방법 결과 비교

메뉴별 매출: [1440000  900000  797500  318000  570000]
전체 매출  : 4025500
매출 비율(%): [35.8 22.4 19.8  7.9 14.2]
10% 인상가 : [4950. 5500. 6050. 5830. 6600.] float64
리스트 방식과 같은가: True


**결과** · 예측 정답은 `float64` — 정수 × 실수 → 실수

- `메뉴별 매출: [1440000  900000  797500  318000  570000]` — 4500 × 320 = 1440000부터 위치별 곱
- `전체 매출  : 4025500` — 5개 합계
- `매출 비율(%): [35.8 22.4 19.8  7.9 14.2]` — 아메리카노가 35.8%로 가장 큼
- `10% 인상가 : [4950. 5500. 6050. 5830. 6600.] float64` — 결과가 실수로 바뀜
- `리스트 방식과 같은가: True` — 같은 결과를 zip 반복문 없이 한 줄로

**차이 · 언제 쓰나**

<table width="100%" style="width:100%; table-layout:auto">
<thead>
<tr>
<th style="text-align: left;">방법</th>
<th style="text-align: left;">차이</th>
<th style="text-align: left;">언제 쓰나</th>
</tr>
</thead>
<tbody>
<tr>
<td style="text-align: left;"><code>a * b</code> (같은 shape)</td>
<td style="text-align: left;"><strong>같은 위치끼리</strong> 계산</td>
<td style="text-align: left;">가격 × 수량 · 이미지 - 평균</td>
</tr>
<tr>
<td style="text-align: left;"><code>a * 1.1</code> (배열 × 숫자)</td>
<td style="text-align: left;"><strong>모든 원소</strong>에 같은 계산</td>
<td style="text-align: left;">단위 변환 · 일괄 인상</td>
</tr>
<tr>
<td style="text-align: left;"><code>zip</code> + 리스트 컴프리헨션</td>
<td style="text-align: left;">반복문으로 같은 결과, 길고 느림</td>
<td style="text-align: left;">배열이 아닌 리스트만 있을 때</td>
</tr>
<tr>
<td style="text-align: left;"><code>np.dot(a, b)</code></td>
<td style="text-align: left;">곱한 뒤 <strong>더해 값 1개</strong></td>
<td style="text-align: left;">총매출처럼 합계 하나만 필요할 때 (7-2)</td>
</tr>
</tbody>
</table>

> ⚠️ **주의**
>
> - 두 배열의 shape가 다르면 에러 또는 의도와 다른 계산 (7-1)
> - 계산 전 shape 확인

> ✅ **한 줄 정리** · 같은 shape의 배열은 같은 위치끼리, 배열과 숫자는 모든 원소에 계산되어 반복문이 필요 없다.

**딥러닝 코드에서는**

- `x_norm = (x - mean) / std` · `loss = ((pred - y) ** 2).mean()` (MSE)

---
## 6. 고르기 · 정렬 · 축

> **이 장의 흐름**
>
> - 5장은 전체 계산이었다
> - 떨어진 위치 고르기, 순위 매기기, 2차원에서 행마다 · 열마다 계산하기

**파이프라인 위치** · ⑤ 학습 · ⑥ 평가 — 배치 고르기, Top-k 예측, 샘플별 · 특성별 통계

### 6-1. `A` 떨어진 위치 고르기 · `a[[i, j]]` <sub>원본 21 · 강의 Support</sub>

> 🎯 **떨어져 있는 여러 위치를 한 번에 꺼내려면?**

💡 **쉽게 말하면**

- **번호표 목록**을 건네면 그 번호의 사물함만 열어 줌.
- 목록 순서대로 · 같은 번호를 여러 번 적어도 됨.

**실행 흐름**

1. 위치 배열 [2, 5, 8]
2. 순서 바꾸기 · 반복
3. 슬라이싱과 비교
4. 복사본 확인

**먼저 예측** · `visitors[[8, 2, 5]]`는 위치 순서대로 정렬되어 나올까? — 답은 코드 실행 결과 아래에

In [20]:
# 떨어진 위치 여러 개: 3월 · 6월 · 9월
picked = np.array([2, 5, 8])                                 # 위치 배열
print("월:", months[picked], "| 방문자:", visitors[picked])  # 같은 위치로 두 배열


# 순서 바꾸기 · 같은 위치 반복
print("순서대로:", visitors[[8, 2, 5]], "| 반복:", visitors[[0, 0, 0]])  # 적은 순서 · 반복


# 규칙적인 간격이면 슬라이싱도 가능
print("슬라이싱 [2:9:3]:", visitors[2:9:3])  # 같은 간격이면 슬라이싱


# fancy indexing 결과는 복사본
f = visitors[picked]                                                       # 고른 결과 받기
f[0] = 0                                                                   # 결과만 수정
print("원본 3월:", visitors[2], "| 공유:", np.shares_memory(f, visitors))  # 원본 그대로인가

월: [3 6 9] | 방문자: [2010 2510 2430]
순서대로: [2430 2010 2510] | 반복: [1820 1820 1820]
슬라이싱 [2:9:3]: [2010 2510 2430]
원본 3월: 2010 | 공유: False


**결과** · 예측 정답은 아니오 — 적은 순서(9월 · 3월 · 6월) 그대로

- `월: [3 6 9] | 방문자: [2010 2510 2430]` — 위치 2 · 5 · 8 = 3 · 6 · 9월
- `순서대로: [2430 2010 2510] | 반복: [1820 1820 1820]` — 적은 순서대로, 같은 위치 반복 가능
- `슬라이싱 [2:9:3]: [2010 2510 2430]` — 간격이 일정하면 슬라이싱과 같은 결과
- `원본 3월: 2010 | 공유: False` — fancy 결과를 고쳐도 원본 그대로 (복사본)

**차이 · 언제 쓰나**

<table width="100%" style="width:100%; table-layout:auto">
<thead>
<tr>
<th style="text-align: left;">방법</th>
<th style="text-align: left;">차이</th>
<th style="text-align: left;">언제 쓰나</th>
</tr>
</thead>
<tbody>
<tr>
<td style="text-align: left;">슬라이싱 <code>a[2:9:3]</code></td>
<td style="text-align: left;"><strong>규칙적인 간격</strong>만, 뷰</td>
<td style="text-align: left;">일정 간격</td>
</tr>
<tr>
<td style="text-align: left;">fancy <code>a[[2, 5, 8]]</code></td>
<td style="text-align: left;"><strong>아무 위치 · 순서 · 반복</strong>, 복사</td>
<td style="text-align: left;">골라 둔 번호 목록으로 꺼낼 때 (배치 · 분할)</td>
</tr>
<tr>
<td style="text-align: left;">mask <code>a[a &gt; 2400]</code></td>
<td style="text-align: left;"><strong>조건</strong>으로 고름, 복사</td>
<td style="text-align: left;">값 기준으로 고를 때 (4-1)</td>
</tr>
</tbody>
</table>

> ⚠️ **주의**
>
> - `visitors[2, 5, 8]`(대괄호 하나)은 3차원 주소로 읽혀 에러
> - 위치 목록은 `visitors[[2, 5, 8]]`

> ✅ **한 줄 정리** · 떨어진 위치 여러 개는 위치 배열을 대괄호에 넣어 꺼내며, 결과는 복사본이다.

**딥러닝 코드에서는**

- `X[batch_idx]` (섞은 번호로 배치 꺼내기)
- `X[train_idx]` (9-5 분할)

### 6-2. `B` 정렬 · `np.sort()` · `.sort()` <sub>원본 22 · 강의 Support</sub>

> 🎯 **`np.sort()`와 `.sort()`는 무엇이 다른가?**

💡 **쉽게 말하면**

- `np.sort`는 **정렬한 복사본을 새로 출력**해 주고, `.sort()`는 **원본 자체를 줄 세우고 빈손(None)으로 돌아옴**.

**실행 흐름**

1. np.sort
2. 원본 확인
3. .sort() 반환값 · 사본
4. 내림차순
5. 반환값 저장 실수

**먼저 예측** · `a = a.sort()` 뒤 `a`는? — 답은 코드 실행 결과 아래에

In [21]:
made = np.array([182, 240, 165, 258, 203])  # 바리스타 5명의 하루 제조 잔 수


# np.sort: 정렬된 새 배열 → 원본 유지
print("np.sort :", np.sort(made), "| 원본:", made)  # 새 배열 · 원본 비교


# .sort(): 원본을 직접 정렬 → 반환값 None
copy_made = made.copy()                                 # 원본 대신 사본으로
returned = copy_made.sort()                             # 제자리 정렬 · 반환값 받기
print(".sort() 반환:", returned, "| 사본:", copy_made)  # 반환값 None 확인


# 내림차순: 정렬 후 뒤집기
print("내림차순:", np.sort(made)[::-1])  # 정렬 → 뒤집기


# 흔한 실수: .sort() 결과를 다시 저장
mistake = made.copy()       # 실수 재현용 사본
mistake = mistake.sort()    # 정렬은 됐지만 변수에는 None 저장
print("mistake:", mistake)  # 배열 대신 None

np.sort : [165 182 203 240 258] | 원본: [182 240 165 258 203]
.sort() 반환: None | 사본: [165 182 203 240 258]
내림차순: [258 240 203 182 165]
mistake: None


**결과** · 예측 정답은 `None` — `.sort()`의 반환값이 None

- `np.sort : [165 182 203 240 258] | 원본: [182 240 165 258 203]` — 새 배열, 원본 그대로
- `.sort() 반환: None | 사본: [165 182 203 240 258]` — 사본 자체가 정렬되고 반환은 None
- `내림차순: [258 240 203 182 165]` — 오름차순 → `[::-1]` 뒤집기
- `mistake: None` — 정렬된 배열을 잃어버림

**차이 · 언제 쓰나**

<table width="100%" style="width:100%; table-layout:auto">
<thead>
<tr>
<th style="text-align: left;">방법</th>
<th style="text-align: left;">차이</th>
<th style="text-align: left;">언제 쓰나</th>
</tr>
</thead>
<tbody>
<tr>
<td style="text-align: left;"><code>np.sort(a)</code></td>
<td style="text-align: left;">정렬된 <strong>새 배열</strong>, 원본 유지</td>
<td style="text-align: left;">원본이 필요할 때 (기본 선택)</td>
</tr>
<tr>
<td style="text-align: left;"><code>a.sort()</code></td>
<td style="text-align: left;"><strong>원본을 정렬</strong>, 반환값 <code>None</code></td>
<td style="text-align: left;">원본이 더 필요 없고 메모리를 아낄 때</td>
</tr>
<tr>
<td style="text-align: left;"><code>np.sort(a)[::-1]</code></td>
<td style="text-align: left;">오름차순 정렬 → <strong>뒤집기</strong></td>
<td style="text-align: left;">내림차순 (전용 옵션 없음)</td>
</tr>
<tr>
<td style="text-align: left;">Python <code>sorted(a)</code></td>
<td style="text-align: left;">결과가 <strong>리스트</strong></td>
<td style="text-align: left;">배열에는 쓰지 않음</td>
</tr>
</tbody>
</table>

> ⚠️ **주의**
>
> - `a = a.sort()`는 배열을 None으로 덮어씀
> - 다음 줄부터 `AttributeError`

> ✅ **한 줄 정리** · `np.sort()`는 새 배열을 돌려주고 `.sort()`는 원본을 바꾸며 None을 돌려주므로, 결과를 저장할 땐 `np.sort()`를 쓴다.

**딥러닝 코드에서는**

- `np.sort(scores)[::-1][:5]`
- 상위 5개 점수 값 확인 (위치가 필요하면 6-3 `argsort`)

### 6-3. `A` 순위 위치 · `argsort()` <sub>원본 23 · 강의 Support</sub>

> 🎯 **점수 순서로 이름까지 함께 정렬하려면?**

💡 **쉽게 말하면**

- `argsort`는 **줄 서는 순서표**(몇 번이 1등, 몇 번이 2등).
- 이 순서표를 이름 명단에 대면 이름도 같은 순서로 섬.

**실행 흐름**

1. argsort 위치
2. 값 확인
3. 뒤집어 상위 3
4. 하위 3
5. np.sort(이름)과 비교

**먼저 예측** · `np.argsort([182, 240, 165, 258, 203])`의 첫 값은? — 답은 코드 실행 결과 아래에

In [22]:
baristas = np.array(["민준", "서연", "도윤", "하윤", "지호"])  # made와 같은 순서의 이름


# argsort: 정렬 순서대로의 '위치'
order = np.argsort(made)                              # 작은 값의 위치부터
print("오름차순 위치:", order, "→ 값:", made[order])  # 위치로 꺼내면 정렬된 값


# 뒤집고 앞 3개 → 상위 3명 (위치를 이름 배열에도 적용)
top3 = np.argsort(made)[::-1][:3]                         # 큰 값 위치 3개
print("상위 3명:", baristas[top3], made[top3])            # 같은 위치를 이름에도
print("하위 3명:", baristas[order[:3]], made[order[:3]])  # 오름차순 앞 3개


# np.sort는 이름을 함께 옮기지 못함
print("np.sort(baristas):", np.sort(baristas))  # 이름 가나다순 → 잔 수와 짝이 끊김

오름차순 위치: [2 0 4 1 3] → 값: [165 182 203 240 258]
상위 3명: ['하윤' '서연' '지호'] [258 240 203]
하위 3명: ['도윤' '민준' '지호'] [165 182 203]
np.sort(baristas): ['도윤' '민준' '서연' '지호' '하윤']


**결과** · 예측 정답은 `2` — 가장 작은 165가 위치 2

- `오름차순 위치: [2 0 4 1 3] → 값: [165 182 203 240 258]` — 위치 배열로 꺼내면 정렬된 값
- `상위 3명: ['하윤' '서연' '지호'] [258 240 203]` — 뒤집고 앞 3개 → 이름과 잔 수가 짝 유지
- `하위 3명: ['도윤' '민준' '지호'] [165 182 203]` — 뒤집지 않고 앞 3개
- `np.sort(baristas): ['도윤' '민준' '서연' '지호' '하윤']` — 이름만 가나다순 → 잔 수와 짝이 끊김

**차이 · 언제 쓰나**

<table width="100%" style="width:100%; table-layout:auto">
<thead>
<tr>
<th style="text-align: left;">방법</th>
<th style="text-align: left;">차이</th>
<th style="text-align: left;">언제 쓰나</th>
</tr>
</thead>
<tbody>
<tr>
<td style="text-align: left;"><code>np.sort(made)</code></td>
<td style="text-align: left;">정렬된 <strong>값</strong></td>
<td style="text-align: left;">값만 필요할 때</td>
</tr>
<tr>
<td style="text-align: left;"><code>np.argsort(made)</code></td>
<td style="text-align: left;">정렬 순서대로의 <strong>위치</strong></td>
<td style="text-align: left;">이름 · 라벨 등 다른 배열을 같은 순서로 (순위)</td>
</tr>
<tr>
<td style="text-align: left;"><code>argsort()[::-1][:k]</code></td>
<td style="text-align: left;">내림차순 위치 앞 k개</td>
<td style="text-align: left;">상위 k개 (top-k 예측)</td>
</tr>
<tr>
<td style="text-align: left;"><code>made.argmax()</code></td>
<td style="text-align: left;">1등 위치 <strong>하나</strong></td>
<td style="text-align: left;">가장 큰 것 하나만 (예측 클래스)</td>
</tr>
</tbody>
</table>

> ⚠️ **주의**
>
> - `[::-1][:3]`(뒤집고 앞 3개 = 상위)와 `[:3][::-1]`(하위 3개를 뒤집음)은 다른 결과

> ✅ **한 줄 정리** · 이름 등 다른 배열을 함께 정렬할 때는 `argsort()`로 위치를 정렬해 양쪽에 적용한다.

**딥러닝 코드에서는**

- `top5 = probs.argsort()[::-1][:5]` → `class_names[top5]` (Top-5 예측 클래스 이름)

### 6-4. `A` 계산 방향 · `axis` <sub>원본 24 · 강의 Support</sub>

> 🎯 **`axis=0`과 `axis=1`은 각각 무엇마다 결과를 내는가?**

💡 **쉽게 말하면**

- `axis`는 **접어서 없앨 방향**.
- axis=0은 위아래(행)를 접어 **열마다** 하나, axis=1은 좌우(열)를 접어 **행마다** 하나.

<img src="images/np_axis.png" alt="axis=0은 열마다 · axis=1은 행마다" width="796">

**실행 흐름**

1. axis=0 · 1 · 없음 합계
2. 평균 · argmax · 정렬
3. keepdims

**먼저 예측** · 지점(행) 3곳별 합계는 `axis=0`일까 `axis=1`일까? — 답은 코드 실행 결과 아래에

In [23]:
# 시간대별(결과 4개) · 지점별(결과 3개) 합계
print("axis=0 (열마다):", sold.sum(axis=0),  # 시간대별 합계
      "| 개수", sold.sum(axis=0).size)       # 행 축이 사라짐
print("axis=1 (행마다):", sold.sum(axis=1),  # 지점별 합계
      "| 개수", sold.sum(axis=1).size)       # 열 축이 사라짐
print("axis 없음 (전체):", sold.sum())       # 전체 하나


# 평균 · 최대 위치 · 정렬도 같은 규칙
print("지점별 평균:", sold.mean(axis=1))                    # 평균도 같은 규칙
print("시간대별 1등 지점:", branches[sold.argmax(axis=0)])  # 열마다 최대 위치 → 지점 이름
print("행마다 정렬:\n", np.sort(sold, axis=1))              # 행 안에서만 정렬


# keepdims: 사라질 축을 길이 1로 남김
print("keepdims shape:", sold.sum(axis=1, keepdims=True).shape)  # 사라질 축을 1로 남김

axis=0 (열마다): [127 265 202  85] | 개수 4
axis=1 (행마다): [216 247 216] | 개수 3
axis 없음 (전체): 679
지점별 평균: [54.   61.75 54.  ]
시간대별 1등 지점: ['판교' '판교' '홍대' '홍대']
행마다 정렬:
 [[ 25  42  61  88]
 [ 30  48  75  94]
 [ 12  47  55 102]]
keepdims shape: (3, 1)


**결과** · 예측 정답은 `axis=1` — 결과가 3개(행마다) 나와야 하므로

- `axis=0 (열마다): [127 265 202  85] | 개수 4` — 시간대 4개 → 점심이 265로 최다
- `axis=1 (행마다): [216 247 216] | 개수 3` — 지점 3개 → 홍대가 247로 최다
- `axis 없음 (전체): 679` — 전체 하나, 127 + 265 + 202 + 85 = 679
- `지점별 평균: [54.   61.75 54.  ]` — 216 ÷ 4 = 54, 247 ÷ 4 = 61.75
- `시간대별 1등 지점: ['판교' '판교' '홍대' '홍대']` — 열마다 최대 위치 → 지점 이름
- `keepdims shape: (3, 1)` — 사라질 축을 길이 1로 남김 → 7-1에서 바로 사용

**차이 · 언제 쓰나**

<table width="100%" style="width:100%; table-layout:auto">
<thead>
<tr>
<th style="text-align: left;">방법</th>
<th style="text-align: left;">차이</th>
<th style="text-align: left;">언제 쓰나</th>
</tr>
</thead>
<tbody>
<tr>
<td style="text-align: left;"><code>axis=0</code></td>
<td style="text-align: left;">행 축이 사라짐 → <strong>열마다</strong> 결과</td>
<td style="text-align: left;">시간대별 · 특성(열)별 통계</td>
</tr>
<tr>
<td style="text-align: left;"><code>axis=1</code></td>
<td style="text-align: left;">열 축이 사라짐 → <strong>행마다</strong> 결과</td>
<td style="text-align: left;">지점별 · 샘플(행)별 통계</td>
</tr>
<tr>
<td style="text-align: left;">axis 생략</td>
<td style="text-align: left;"><strong>전체</strong> 하나</td>
<td style="text-align: left;">전체 합계 · 평균</td>
</tr>
<tr>
<td style="text-align: left;"><code>keepdims=True</code></td>
<td style="text-align: left;">사라질 축을 <strong>길이 1</strong>로 남김</td>
<td style="text-align: left;">결과로 원본을 다시 나눌 때 (7-1)</td>
</tr>
</tbody>
</table>

> ⚠️ **주의**
>
> - "axis=0 = 행"으로 외우면 거꾸로 헷갈림
> - **원하는 결과가 몇 개인지**로 판단

> ✅ **한 줄 정리** · `axis`에 적은 축이 사라지므로, 결과 개수가 열 수면 `axis=0`, 행 수면 `axis=1`을 쓴다.

**딥러닝 코드에서는**

- `x.mean(axis=0)` (특성별 평균)
- `logits.argmax(axis=1)` (샘플마다 예측 클래스)
- 이미지 `mean(axis=(0, 1))` (9-1)

---
## 7. shape 맞추기

> **이 장의 흐름**
>
> - 6장 axis로 2차원 계산을 했다
> - shape가 다른 배열끼리 계산하는 규칙(broadcasting)과 신경망의 핵심 계산(행렬 곱)

**파이프라인 위치** · ④ 모델 — 층 하나의 계산 `X @ W + b`, shape 오류 해결

### 7-1. `A` 다른 shape끼리 계산 · `broadcasting` <sub>원본 25 · 강의 Preview</sub>

> 🎯 **shape가 다른 배열끼리는 언제 계산되고 언제 에러가 나는가?**

💡 **쉽게 말하면**

- 작은 배열을 **복사기로 늘려** 큰 배열에 맞추는 것.
- 단, 오른쪽 끝부터 비교해 **길이가 같거나 1**인 축만 늘릴 수 있음.

<img src="images/np_broadcast.png" alt="(4,) · (3, 1)은 늘어나고 (3,)는 실패" width="808">

**실행 흐름**

1. (3, 4) + (4,) 성공
2. (3, 4) + (3,) 실패
3. (3, 1)로 세워 성공
4. 행 합계로 나누기

**먼저 예측** · `(3, 4) + (3,)`은 성공할까? — 답은 코드 실행 결과 아래에

In [24]:
# 시간대마다 다른 보너스 (4,) → 마지막 축 4 = 4
time_bonus = np.array([1, 2, 3, 4])                                       # 시간대 4개 보너스
print(sold.shape, "+", time_bonus.shape, "→", (sold + time_bonus).shape,  # 결과 모양
      "| 첫 행:", (sold + time_bonus)[0])                                 # 모든 행에 [1, 2, 3, 4]


# 지점마다 다른 보너스 (3,) → 마지막 축 4 ≠ 3
branch_bonus = np.array([10, 20, 30])  # 지점 3개 보너스
try:
    sold + branch_bonus                # (3, 4) + (3,) 시도
except ValueError as e:                # 모양 불일치 에러
    print("ValueError:", e)            # 에러 문장 그대로


# (3,) → (3, 1)로 세우면 행마다 같은 값
column_bonus = branch_bonus.reshape(3, 1)   # 세로로 세우기
print(sold.shape, "+", column_bonus.shape,  # 행마다 다른 값
      "→ 첫 열:", (sold + column_bonus)[:, 0])


# 응용: 각 행을 그 행의 합으로 나눠 비율 만들기
ratio = sold / sold.sum(axis=1, keepdims=True)  # (3, 4) / (3, 1)
print("행마다 비율 합:", ratio.sum(axis=1))     # 각 행 합이 1인가

(3, 4) + (4,) → (3, 4) | 첫 행: [43 90 64 29]
ValueError: operands could not be broadcast together with shapes (3,4) (3,) 
(3, 4) + (3, 1) → 첫 열: [52 50 85]
행마다 비율 합: [1. 1. 1.]


**결과** · 예측 정답은 실패 — 오른쪽 끝 4 ≠ 3

- `(3, 4) + (4,) → (3, 4) | 첫 행: [43 90 64 29]` — 42 + 1 = 43, 88 + 2 = 90, 61 + 3 = 64, 25 + 4 = 29
- `ValueError: operands could not be broadcast together with shapes (3,4) (3,)` — 오른쪽 끝 4와 3이 다름
- `(3, 4) + (3, 1) → 첫 열: [52 50 85]` — 42 + 10 = 52, 30 + 20 = 50, 55 + 30 = 85 (행마다 다른 값)
- `행마다 비율 합: [1. 1. 1.]` — (3, 4) ÷ (3, 1) → 각 행이 합 1인 비율

**차이 · 언제 쓰나**

<table width="100%" style="width:100%; table-layout:auto">
<thead>
<tr>
<th style="text-align: left;">방법</th>
<th style="text-align: left;">차이</th>
<th style="text-align: left;">언제 쓰나</th>
</tr>
</thead>
<tbody>
<tr>
<td style="text-align: left;">같은 shape <code>(3, 4) + (3, 4)</code></td>
<td style="text-align: left;">같은 위치끼리</td>
<td style="text-align: left;">기본 원소별 계산</td>
</tr>
<tr>
<td style="text-align: left;"><code>(3, 4) + (4,)</code></td>
<td style="text-align: left;">모든 <strong>행</strong>에 같은 벡터 → 열마다 다른 값</td>
<td style="text-align: left;">열(특성)별 보정 · 채널별 평균 빼기</td>
</tr>
<tr>
<td style="text-align: left;"><code>(3, 4) + (3, 1)</code></td>
<td style="text-align: left;">모든 <strong>열</strong>에 같은 값 → 행마다 다른 값</td>
<td style="text-align: left;">행(샘플)별 보정 · 행 합계로 나누기</td>
</tr>
<tr>
<td style="text-align: left;"><code>(3, 4) + 숫자</code></td>
<td style="text-align: left;">모든 원소에 같은 값</td>
<td style="text-align: left;">일괄 덧셈 · 정규화 <code>/ 255</code></td>
</tr>
</tbody>
</table>

> ⚠️ **주의**
>
> - 값 개수가 같아도(3개) 방향이 다르면 에러
> - `(3,)`와 `(3, 1)`은 다른 모양

> ✅ **한 줄 정리** · shape를 오른쪽 끝부터 비교해 각 축이 같거나 1이면 계산되며, 방향은 `reshape`로 맞춘다.

**딥러닝 코드에서는**

- `(x - mean) / std` (채널별 평균 `(3,)`을 이미지 `(H, W, 3)`에)
- `X @ W + b` (편향 `b`가 배치 전체에)

### 7-2. `A` 내적 · 행렬 곱 · 전치 · `np.dot()` · `@` · `.T` <sub>원본 26 · 강의 Preview</sub>

> 🎯 **`*`와 `np.dot` · `@`는 결과가 어떻게 다른가?**

💡 **쉽게 말하면**

- `*`는 **같은 자리끼리 곱한 영수증 여러 장**, `dot`은 **그 영수증을 모두 더한 합계 한 장**.
- 행렬 곱은 이 합계를 칸마다 반복.

<img src="images/np_matmul.png" alt="(2, 3) @ (3, 2) → (2, 2)" width="793">

**실행 흐름**

1. * vs dot
2. (2, 3) @ (3, 2)
3. 손계산 검증
4. 안쪽 불일치 에러 · .T
5. 3차원

**먼저 예측** · `(2, 3) @ (3, 2)`의 결과 shape는? — 답은 코드 실행 결과 아래에

In [25]:
price_vec = np.array([4500, 5000, 5500])  # 메뉴 3개 가격
cups_vec = np.array([10, 8, 6])           # 같은 순서의 잔 수


# * 는 결과 3개, dot은 곱한 뒤 더해 1개
print("원소별 곱:", price_vec * cups_vec, "| 내적:", np.dot(price_vec, cups_vec))  # 3개 vs 1개


# 행렬 곱: (2, 3) @ (3, 2) → (2, 2)
sales_count = np.array([[10, 8, 6],          # 강남: 아메리카노 · 라떼 · 콜드브루 잔 수
                        [12, 5, 9]])         # 홍대
price_cost = np.array([[4500, 1200],         # 메뉴별 [판매가, 원가]
                       [5000, 1500],
                       [5500, 1800]])
result = sales_count @ price_cost            # np.dot(A, B)와 같음
print("결과 shape:", result.shape)           # 바깥 숫자 2, 2
print("지점별 [총매출, 총원가]:\n", result)  # 칸마다 내적 결과
print("강남 총매출 손계산:", 10 * 4500 + 8 * 5000 + 6 * 5500)  # [0, 0] 검산


# 안쪽 숫자가 다르면 에러 → .T로 맞추기
try:
    sales_count @ sales_count                                # (2, 3) @ (2, 3) → 안쪽 3 ≠ 2
except ValueError:                                           # 안쪽 불일치 에러
    print("(2, 3) @ (2, 3) → ValueError")                    # 실패했음을 표시
print("전치:", sales_count.shape, "→", sales_count.T.shape,  # 전치로 안쪽 맞추기
      "| A @ A.T:", (sales_count @ sales_count.T).shape)     # (2, 3) @ (3, 2)


# 3차원 배열: (깊이, 행, 열)
cube = np.arange(8).reshape(2, 2, 2)                                          # 2×2 표 2장
print("cube ndim:", cube.ndim, "| shape:", cube.shape, "| size:", cube.size)  # 3차원 속성

원소별 곱: [45000 40000 33000] | 내적: 118000
결과 shape: (2, 2)
지점별 [총매출, 총원가]:
 [[118000  34800]
 [128500  38100]]
강남 총매출 손계산: 118000
(2, 3) @ (2, 3) → ValueError
전치: (2, 3) → (3, 2) | A @ A.T: (2, 2)
cube ndim: 3 | shape: (2, 2, 2) | size: 8


**결과** · 예측 정답은 `(2, 2)` — 안쪽 3끼리 맞고 바깥 2, 2가 남음

- `원소별 곱: [45000 40000 33000] | 내적: 118000` — 45000 + 40000 + 33000 = 118000
- `결과 shape: (2, 2)` — (지점 2, [매출 · 원가] 2)
- `[128500  38100]]` — 홍대 매출 12 × 4500 + 5 × 5000 + 9 × 5500 = 128500
- `강남 총매출 손계산: 118000` — 결과 [0, 0] = 강남 행 · 판매가 열의 내적
- `(2, 3) @ (2, 3) → ValueError` — 안쪽 3 ≠ 2
- `전치: (2, 3) → (3, 2) | A @ A.T: (2, 2)` — `.T`로 안쪽을 맞추면 계산 가능
- `cube ndim: 3 | shape: (2, 2, 2) | size: 8` — 3차원 = 2차원 표가 여러 장

**차이 · 언제 쓰나**

<table width="100%" style="width:100%; table-layout:auto">
<thead>
<tr>
<th style="text-align: left;">방법</th>
<th style="text-align: left;">차이</th>
<th style="text-align: left;">언제 쓰나</th>
</tr>
</thead>
<tbody>
<tr>
<td style="text-align: left;"><code>a * b</code></td>
<td style="text-align: left;">원소별 곱, 결과 <strong>여러 개</strong></td>
<td style="text-align: left;">같은 위치끼리 계산</td>
</tr>
<tr>
<td style="text-align: left;"><code>np.dot(a, b)</code> (1차원)</td>
<td style="text-align: left;">곱한 뒤 더한 <strong>값 1개</strong> (내적)</td>
<td style="text-align: left;">가중합 · 총매출</td>
</tr>
<tr>
<td style="text-align: left;"><code>A @ B</code> · <code>np.matmul</code></td>
<td style="text-align: left;"><strong>행렬 곱</strong>, 안쪽 길이가 같아야 함</td>
<td style="text-align: left;">신경망 한 층 <code>X @ W</code> (기본 선택)</td>
</tr>
<tr>
<td style="text-align: left;"><code>A.T</code></td>
<td style="text-align: left;">행 ↔ 열 (모든 축 역순)</td>
<td style="text-align: left;">안쪽 shape를 맞출 때</td>
</tr>
</tbody>
</table>

> ⚠️ **주의**
>
> - `*`는 행렬 곱이 아님
> - 신경망 층은 `@` (shape 오류가 안 나도 `*`면 다른 계산)

> ✅ **한 줄 정리** · `*`는 원소별 곱, `dot` · `@`는 곱한 뒤 더하는 계산이며, 행렬 곱은 안쪽 길이가 같아야 한다.

**딥러닝 코드에서는**

- `out = X @ W + b`
- `(배치, 입력)` @ `(입력, 출력)` → `(배치, 출력)`
- PyTorch `nn.Linear`가 이 계산

---
## 8. 종합 실습 · 신메뉴 수익 분석

> **이 장의 흐름**
>
> - 1 ~ 7장 연산을 한 문제에 모두 사용
> - 강의 Final Challenge(4상품)와 연습문제 27이 같은 단계라 연습문제 하나로 통합

**파이프라인 위치** · ③ 전처리(원소별 계산) → ⑥ 평가(조건 · 순위 · 위치)

> 🎯 **메뉴별 이익 · 순위 · 관리 대상을 반복문 없이 배열 연산만으로 찾으려면?**

💡 **쉽게 말하면**

- 가격 · 원가 · 판매량은 **같은 순서로 줄 선 세 명단**.
- 같은 자리끼리 빼고 곱해 이익을 만들고, 조건 · 순위표(argsort)를 이름 명단에 대면 답이 나옴.

<table width="100%" style="width:100%; table-layout:auto">
<thead>
<tr>
<th style="text-align: left;">단계</th>
<th style="text-align: left;">하는 일</th>
<th style="text-align: left;">쓰는 연산</th>
</tr>
</thead>
<tbody>
<tr>
<td style="text-align: left;">0</td>
<td style="text-align: left;">네 배열의 shape가 같은지 확인</td>
<td style="text-align: left;">1-2 <code>shape</code></td>
</tr>
<tr>
<td style="text-align: left;">1 ~ 3</td>
<td style="text-align: left;">개당 이익 → 총이익 → 전체 총이익</td>
<td style="text-align: left;">5-2 원소별 계산 · 5-1 <code>sum()</code></td>
</tr>
<tr>
<td style="text-align: left;">4</td>
<td style="text-align: left;">총이익 30만 이상인 메뉴 이름</td>
<td style="text-align: left;">4-1 mask</td>
</tr>
<tr>
<td style="text-align: left;">5</td>
<td style="text-align: left;">총이익이 높은 순서의 이름 · 금액</td>
<td style="text-align: left;">6-3 <code>argsort()[::-1]</code></td>
</tr>
<tr>
<td style="text-align: left;">6</td>
<td style="text-align: left;">총이익이 가장 낮은 메뉴</td>
<td style="text-align: left;">5-1 <code>argmin()</code></td>
</tr>
<tr>
<td style="text-align: left;">7</td>
<td style="text-align: left;">이익률(%) · 결과 검증</td>
<td style="text-align: left;">5-2 배열 ÷ 배열 · <code>assert</code></td>
</tr>
</tbody>
</table>

**먼저 예측** · 30만 이상 메뉴는 몇 개이고, 말차라떼의 총이익은 얼마인가? — 답은 코드 실행 결과 아래에

In [26]:
new_menu = np.array(["말차라떼", "흑임자라떼", "딸기크림", "밤라떼"])  # 메뉴 이름 4개
sell_price = np.array([6000, 6500, 7000, 5800])                        # 판매가
cost_price = np.array([2200, 2600, 3100, 2000])                        # 원가
sold_count = np.array([140, 95, 210, 60])                              # 판매량


# 0단계: 계산 전 shape 확인
assert (new_menu.shape == sell_price.shape  # 네 배열 길이가 같은가
        == cost_price.shape == sold_count.shape)
print("메뉴 수:", new_menu.size)            # 메뉴 개수


# 1~3단계: 개당 이익 → 총이익 → 전체 총이익
unit_profit = sell_price - cost_price                      # 5-2 같은 위치끼리 빼기
total_profit = unit_profit * sold_count                    # 5-2 같은 위치끼리 곱하기
grand_total = total_profit.sum()                           # 5-1 합계
print("개당 이익:", unit_profit)                           # 판매가 - 원가
print("총이익   :", total_profit, "| 전체:", grand_total)  # 메뉴별 · 전체

메뉴 수: 4
개당 이익: [3800 3900 3900 3800]
총이익   : [532000 370500 819000 228000] | 전체: 1949500


In [27]:
# 4단계: 총이익 30만 이상 메뉴
high = new_menu[total_profit >= 300000]          # 4-1 조건은 이익, 꺼내는 건 이름
print("30만 이상:", high, "| 개수:", high.size)  # 이름 · 개수


# 5단계: 총이익 높은 순서
order = np.argsort(total_profit)[::-1]                     # 6-3 위치 정렬 → 뒤집기
print("높은 순서:", new_menu[order], total_profit[order])  # 이름과 금액 같은 순서


# 6단계: 가장 낮은 메뉴
worst = total_profit.argmin()                              # 5-1 값이 아니라 위치
print("가장 낮음:", new_menu[worst], total_profit[worst])  # 위치로 이름 · 금액


# 7단계: 이익률 · 결과 검증
margin = np.round(unit_profit / sell_price * 100, 1)  # 판매가 대비 이익(%)
print("이익률(%):", margin)                           # 효율 비교
assert grand_total == 1949500 and high.size == 3      # 답 검증

30만 이상: ['말차라떼' '흑임자라떼' '딸기크림'] | 개수: 3
높은 순서: ['딸기크림' '말차라떼' '흑임자라떼' '밤라떼'] [819000 532000 370500 228000]
가장 낮음: 밤라떼 228000
이익률(%): [63.3 60.  55.7 65.5]


**결과** · 예측 정답은 `3`개 · `532000` (개당 이익 3800 × 140 = 532000)

- `개당 이익: [3800 3900 3900 3800]` — 판매가 - 원가, 같은 위치끼리
- `총이익   : [532000 370500 819000 228000] | 전체: 1949500` — 532000 + 370500 + 819000 + 228000 = 1949500
- `30만 이상: ['말차라떼' '흑임자라떼' '딸기크림'] | 개수: 3` — 밤라떼(228000)만 제외
- `높은 순서: ['딸기크림' '말차라떼' '흑임자라떼' '밤라떼'] [819000 532000 370500 228000]` — 이름과 금액이 같은 순서
- `가장 낮음: 밤라떼 228000` — argmin 위치 3 → 이름
- `이익률(%): [63.3 60.  55.7 65.5]` — 딸기크림은 총이익 1위지만 이익률 최하위, 밤라떼는 이익률 1위

<img src="images/np_profit.png" alt="총이익과 이익률 비교" width="988">

**해석**

- 딸기크림: 총이익 1위(819000)지만 원가가 가장 높아 이익률 최하위(55.7%) → **원가 관리 우선**
- 밤라떼: 이익률 1위(65.5%)인데 판매량이 60잔뿐 → **판매 늘리기 우선**

**차이 · 언제 쓰나**

<table width="100%" style="width:100%; table-layout:auto">
<thead>
<tr>
<th style="text-align: left;">방법</th>
<th style="text-align: left;">차이</th>
<th style="text-align: left;">언제 쓰나</th>
</tr>
</thead>
<tbody>
<tr>
<td style="text-align: left;"><code>total_profit.min()</code></td>
<td style="text-align: left;">가장 낮은 <strong>금액</strong></td>
<td style="text-align: left;">얼마인지만 필요할 때</td>
</tr>
<tr>
<td style="text-align: left;"><code>total_profit.argmin()</code></td>
<td style="text-align: left;">가장 낮은 <strong>위치</strong> → 이름까지 꺼냄</td>
<td style="text-align: left;">어떤 메뉴인지 필요할 때 (기본 선택)</td>
</tr>
<tr>
<td style="text-align: left;">총이익 순위</td>
<td style="text-align: left;">판매량까지 반영한 <strong>규모</strong></td>
<td style="text-align: left;">매출 기여가 큰 메뉴 찾기</td>
</tr>
<tr>
<td style="text-align: left;">이익률 순위</td>
<td style="text-align: left;">판매가 대비 <strong>효율</strong></td>
<td style="text-align: left;">가격 · 원가 조정 대상 찾기</td>
</tr>
</tbody>
</table>

> ⚠️ **주의**
>
> - 총이익만 보면 판매량이 많은 메뉴가 항상 위
> - 이익률과 함께 봐야 판단이 치우치지 않음

> ✅ **한 줄 정리** · 같은 shape의 배열로 계산하고, mask로 거르고, `argsort` · `argmin`으로 위치를 찾아 이름을 꺼내면 분석이 끝난다.

**딥러닝 코드에서는** · 모델 평가도 같은 흐름

1. `correct = pred == y` (같은 위치끼리 비교)
2. `acc = correct.mean()` (True = 1로 평균 = 정확도)
3. `wrong = np.where(~correct)[0]` → `names[wrong]` (틀린 샘플 이름)

---
## 9. 딥러닝 데이터로 연결

> **이 장의 흐름**
>
> - 8장까지는 표 데이터였다
> - 프로젝트의 이미지 · 시계열을 모델 입력 모양으로 바꾸는 작업도 같은 NumPy 연산.
> - PyTorch 없이 실행되며, PyTorch에서는 이름만 바뀐 같은 함수를 씀

**파이프라인 위치** · ① 데이터 준비 → ③ 전처리 → ④ 모델 입력 — 이미지 `(N, C, H, W)` float32, 시계열 `(N, 시간, 특성)`

### 9-1. `A` 이미지 정규화 · `astype(np.float32) / 255` <sub>추가</sub>

> 🎯 **0~255 정수 이미지를 모델에 넣기 전에 왜 실수 0~1로 바꾸는가?**

💡 **쉽게 말하면**

- `uint8`은 **0~255만 적히는 주행거리계**.
- 255 다음은 0으로 돌아가므로, 계산 전에 넉넉한 실수 계기판으로 옮기고 0~1로 줄임.

**실행 흐름**

1. (2, 4, 3) uint8 이미지
2. + 10 넘침
3. float32 ÷ 255
4. ÷ 255만 했을 때 dtype
5. 채널별 표준화

**먼저 예측** · uint8 값 `250`에 `10`을 더하면? — 답은 코드 실행 결과 아래에

In [28]:
# 가짜 이미지: 높이 2 × 너비 4 × 채널 3(RGB), 0~255 정수
img = np.array([[[255, 0, 0], [0, 255, 0], [0, 0, 255], [255, 255, 255]],  # (2, 4, 3) RGB
                [[0, 0, 0], [128, 128, 128], [250, 10, 60], [30, 200, 90]]],
               dtype=np.uint8)
print("shape:", img.shape, "| dtype:", img.dtype,                          # 이미지 기본 속성
      "| 범위:", img.min(), "~", img.max())


# uint8 그대로 더하면 255를 넘는 값이 0부터 다시 돌아감
print("uint8 + 10:", img[1, 2],  # 넘침 전 · 후 비교
      "→", (img + 10)[1, 2])     # 250 + 10 = 260 → 260 - 256 = 4


# 실수로 바꾼 뒤 255로 나누기 → 0~1
x = img.astype(np.float32) / 255                             # uint8 → float32 → 0~1
print("정규화:", x.dtype, "| 범위:", x.min(), "~", x.max())  # 0 ~ 1 실수인가
print("/ 255만 하면:", (img / 255).dtype)                    # 자동으로 float64 (메모리 2배)


# 채널별 표준화: (x - 채널 평균) / 채널 표준편차
mean = x.mean(axis=(0, 1))                             # 높이 · 너비 축을 없앰 → 채널별 (3,)
std = x.std(axis=(0, 1))                               # 채널별 표준편차
z = (x - mean) / std                                   # (2, 4, 3) - (3,) → 7-1 broadcasting
print("채널 평균 shape:", mean.shape,                  # 채널 3개 통계
      "| 표준화 후 채널 평균 ≈ 0:",
      np.allclose(z.mean(axis=(0, 1)), 0, atol=1e-6))  # 거의 0인지

shape: (2, 4, 3) | dtype: uint8 | 범위: 0 ~ 255
uint8 + 10: [250  10  60] → [ 4 20 70]
정규화: float32 | 범위: 0.0 ~ 1.0
/ 255만 하면: float64
채널 평균 shape: (3,) | 표준화 후 채널 평균 ≈ 0: True


**결과** · 예측 정답은 `4` — 260 - 256 (에러 없이 넘침)

- `shape: (2, 4, 3) | dtype: uint8 | 범위: 0 ~ 255` — 높이 2 · 너비 4 · 채널 3, 이미지 파일의 기본 형태
- `uint8 + 10: [250  10  60] → [ 4 20 70]` — 250 + 10 = 260 → 260 - 256 = 4로 넘침, 나머지는 정상
- `정규화: float32 | 범위: 0.0 ~ 1.0` — 0 → 0.0, 255 → 1.0
- `/ 255만 하면: float64` — 자동 float64 → 메모리 2배, 모델은 float32를 기대
- `채널 평균 shape: (3,) | 표준화 후 채널 평균 ≈ 0: True` — 축 (0, 1)을 없애 채널별 통계 → broadcasting으로 빼기

**차이 · 언제 쓰나**

<table width="100%" style="width:100%; table-layout:auto">
<thead>
<tr>
<th style="text-align: left;">방법</th>
<th style="text-align: left;">차이</th>
<th style="text-align: left;">언제 쓰나</th>
</tr>
</thead>
<tbody>
<tr>
<td style="text-align: left;"><code>img + 10</code> (uint8)</td>
<td style="text-align: left;">255를 넘으면 <strong>0부터 다시</strong> (조용한 오류)</td>
<td style="text-align: left;">쓰지 않음 → 먼저 실수로 변환</td>
</tr>
<tr>
<td style="text-align: left;"><code>img / 255</code></td>
<td style="text-align: left;">0~1 실수, 자동 <strong>float64</strong></td>
<td style="text-align: left;">빠르게 확인할 때</td>
</tr>
<tr>
<td style="text-align: left;"><code>img.astype(np.float32) / 255</code></td>
<td style="text-align: left;">0~1 <strong>float32</strong></td>
<td style="text-align: left;">모델 입력 (기본 선택)</td>
</tr>
<tr>
<td style="text-align: left;"><code>(x - mean) / std</code></td>
<td style="text-align: left;">평균 0 · 표준편차 1 (표준화)</td>
<td style="text-align: left;">사전학습 모델 입력 · 학습 안정화</td>
</tr>
</tbody>
</table>

> ⚠️ **주의**
>
> - uint8 상태로 더하기 · 빼기 · 평균 빼기를 하면 에러 없이 값이 넘침
> - **먼저 `astype(np.float32)`**

> ✅ **한 줄 정리** · 이미지는 `astype(np.float32) / 255`로 0~1 실수로 바꾼 뒤 계산하고, 필요하면 채널별로 표준화한다.

**딥러닝 코드에서는**

- `transforms.ToTensor()`가 이 작업(uint8 → float32 · ÷ 255 · HWC → CHW)을 한 번에 함
- `Normalize(mean, std)`는 채널별 표준화

### 9-2. `A` 채널 축 옮기기 · `transpose()` <sub>추가</sub>

> 🎯 **이미지의 채널 축을 맨 앞으로 옮길 때 `reshape`를 쓰면 왜 안 되는가?**

💡 **쉽게 말하면**

- `transpose`는 **책장의 방향만 돌리는 것**(같은 책 · 같은 칸), `reshape`는 **책을 순서대로 빼서 새 책장에 다시 꽂는 것**이라 엉뚱한 칸에 들어감.

<img src="images/np_hwc_chw.png" alt="HWC → CHW: transpose와 reshape의 차이" width="808">

**실행 흐름**

1. transpose(2, 0, 1)
2. 빨강 채널 확인
3. reshape와 비교
4. .T · moveaxis
5. 되돌리기

**먼저 예측** · `img.reshape(3, 2, 4)`의 0번이 빨강 채널일까? — 답은 코드 실행 결과 아래에

In [29]:
# HWC (높이, 너비, 채널) → CHW (채널, 높이, 너비)
chw = img.transpose(2, 0, 1)  # 새 자리에 '원래 축 번호': 채널(2) · 높이(0) · 너비(1)
print("HWC:", img.shape, "→ CHW:", chw.shape)  # 축 순서 변화


# 빨강 채널이 한 장으로 모였는지 확인
print("chw[0] (빨강):\n", chw[0])                                   # 첫 채널 = 빨강 한 장
print("원본 빨강과 같은가:", np.array_equal(chw[0], img[:, :, 0]))  # 값이 그대로인가


# reshape는 shape만 같고 값이 뒤섞임
wrong = img.reshape(3, 2, 4)    # 값을 앞에서부터 순서대로 다시 채움
print("reshape:", wrong.shape,  # reshape 결과 비교
      "| 빨강과 같은가:", np.array_equal(wrong[0], img[:, :, 0]))


# .T는 축 순서를 전부 뒤집음 · moveaxis는 축 하나 이동
print(".T:", img.T.shape,  # 모든 축 역순
      "| moveaxis(img, -1, 0):", np.moveaxis(img, -1, 0).shape)


# 되돌리기: CHW → HWC
back = chw.transpose(1, 2, 0)  # 높이(1) · 너비(2) · 채널(0)
print("되돌리기:", back.shape, "| 원본과 같은가:", np.array_equal(back, img))  # 왕복 확인

HWC: (2, 4, 3) → CHW: (3, 2, 4)
chw[0] (빨강):
 [[255   0   0 255]
 [  0 128 250  30]]
원본 빨강과 같은가: True
reshape: (3, 2, 4) | 빨강과 같은가: False
.T: (3, 4, 2) | moveaxis(img, -1, 0): (3, 2, 4)
되돌리기: (2, 4, 3) | 원본과 같은가: True


**결과** · 예측 정답은 아님 — shape만 같고 값이 뒤섞임

- `HWC: (2, 4, 3) → CHW: (3, 2, 4)` — 채널이 맨 앞으로
- `[[255   0   0 255]` — chw[0] 첫 줄 = 첫 행 픽셀 4개의 빨강 값
- `원본 빨강과 같은가: True` — `img[:, :, 0]`과 일치
- `reshape: (3, 2, 4) | 빨강과 같은가: False` — shape는 같지만 그림이 망가짐 (에러 없음)
- `.T: (3, 4, 2) | moveaxis(img, -1, 0): (3, 2, 4)` — .T는 (C, W, H)로 높이 · 너비까지 뒤바뀜
- `되돌리기: (2, 4, 3) | 원본과 같은가: True` — transpose(1, 2, 0)으로 원래대로

**차이 · 언제 쓰나**

<table width="100%" style="width:100%; table-layout:auto">
<thead>
<tr>
<th style="text-align: left;">방법</th>
<th style="text-align: left;">차이</th>
<th style="text-align: left;">언제 쓰나</th>
</tr>
</thead>
<tbody>
<tr>
<td style="text-align: left;"><code>img.transpose(2, 0, 1)</code></td>
<td style="text-align: left;"><strong>축 순서</strong>를 바꿈, 값 위치 유지</td>
<td style="text-align: left;">HWC → CHW (기본 선택)</td>
</tr>
<tr>
<td style="text-align: left;"><code>np.moveaxis(img, -1, 0)</code></td>
<td style="text-align: left;">축 <strong>하나</strong>를 옮김, 같은 결과</td>
<td style="text-align: left;">채널 축만 앞으로 보낼 때</td>
</tr>
<tr>
<td style="text-align: left;"><code>img.reshape(3, 2, 4)</code></td>
<td style="text-align: left;">shape만 같고 값이 <strong>뒤섞임</strong></td>
<td style="text-align: left;">축 순서 변경에는 절대 X</td>
</tr>
<tr>
<td style="text-align: left;"><code>img.T</code></td>
<td style="text-align: left;"><strong>모든 축 역순</strong> → <code>(C, W, H)</code></td>
<td style="text-align: left;">2차원 행렬 전치에만</td>
</tr>
</tbody>
</table>

> ⚠️ **주의**
>
> - `reshape` · `.T`는 에러 없이 shape만 맞춰 줌
> - 축 순서 변경은 **항상 `transpose`**, 바꾼 뒤 채널 하나를 원본과 비교

> ✅ **한 줄 정리** · 축 순서를 바꿀 때는 `transpose`에 원래 축 번호를 새 순서로 적고, `reshape`는 쓰지 않는다.

**딥러닝 코드에서는**

- PIL · OpenCV 이미지는 `(H, W, C)`
- PyTorch 모델은 `(C, H, W)` → `img.transpose(2, 0, 1)`
- 시각화할 땐 `transpose(1, 2, 0)`

### 9-3. `A` 배치 축 추가 · `expand_dims()` <sub>추가</sub>

> 🎯 **이미지 한 장을 모델에 넣으려면 왜 축을 하나 더 붙여야 하는가?**

💡 **쉽게 말하면**

- 모델은 **여러 장이 든 묶음**(배치)만 받는 기계.
- 한 장이어도 **1장짜리 묶음**으로 포장해야 함.

**실행 흐름**

1. (3, 2, 4)
2. expand_dims
3. None 인덱싱
4. 흑백 채널 축
5. squeeze로 제거

**먼저 예측** · `(3, 2, 4)`에 `expand_dims(axis=0)`을 하면? — 답은 코드 실행 결과 아래에

In [30]:
# 이미지 한 장 (C, H, W) → 배치 1장 (1, C, H, W)
batch1 = np.expand_dims(chw, axis=0)                 # 0번 자리에 길이 1인 축 추가
print("한 장:", chw.shape, "→ 배치:", batch1.shape)  # 앞에 1이 붙음


# 같은 결과: None 인덱싱
print("chw[None]:", chw[None].shape,  # 짧은 표기 결과
      "| chw[np.newaxis]:", chw[np.newaxis].shape)


# 흑백 (H, W) → 채널 축 추가 (1, H, W)
gray = img.mean(axis=2)                                         # 채널 평균 → (2, 4)
print("흑백:", gray.shape, "→", np.expand_dims(gray, 0).shape)  # 채널 축 추가


# 반대: 길이 1인 축 제거
print("squeeze:", batch1.shape, "→", batch1.squeeze(0).shape)  # 길이 1 축 제거

한 장: (3, 2, 4) → 배치: (1, 3, 2, 4)
chw[None]: (1, 3, 2, 4) | chw[np.newaxis]: (1, 3, 2, 4)
흑백: (2, 4) → (1, 2, 4)
squeeze: (1, 3, 2, 4) → (3, 2, 4)


**결과** · 예측 정답은 `(1, 3, 2, 4)`

- `한 장: (3, 2, 4) → 배치: (1, 3, 2, 4)` — 0번 자리에 길이 1인 배치 축
- `chw[None]: (1, 3, 2, 4) | chw[np.newaxis]: (1, 3, 2, 4)` — 같은 결과의 짧은 표기
- `흑백: (2, 4) → (1, 2, 4)` — 채널 축이 없는 흑백 → 채널 1
- `squeeze: (1, 3, 2, 4) → (3, 2, 4)` — 반대 작업: 길이 1 축 제거

**차이 · 언제 쓰나**

<table width="100%" style="width:100%; table-layout:auto">
<thead>
<tr>
<th style="text-align: left;">방법</th>
<th style="text-align: left;">차이</th>
<th style="text-align: left;">언제 쓰나</th>
</tr>
</thead>
<tbody>
<tr>
<td style="text-align: left;"><code>np.expand_dims(x, 0)</code></td>
<td style="text-align: left;">지정 위치에 길이 1 축 추가</td>
<td style="text-align: left;">읽기 쉬움 (기본 선택)</td>
</tr>
<tr>
<td style="text-align: left;"><code>x[None]</code> · <code>x[np.newaxis]</code></td>
<td style="text-align: left;">같은 결과, 더 짧음</td>
<td style="text-align: left;">코드에서 자주 보는 형태</td>
</tr>
<tr>
<td style="text-align: left;"><code>x.reshape(1, 3, 2, 4)</code></td>
<td style="text-align: left;">모든 축을 직접 적음</td>
<td style="text-align: left;">shape를 이미 알 때</td>
</tr>
<tr>
<td style="text-align: left;"><code>x.squeeze(0)</code></td>
<td style="text-align: left;">길이 1 축 <strong>제거</strong> (반대 작업)</td>
<td style="text-align: left;">배치 1장 결과 → 한 장</td>
</tr>
</tbody>
</table>

> ⚠️ **주의**
>
> - `squeeze()`에 축 번호를 안 주면 길이 1인 축이 **전부** 사라짐
> - 배치 1 · 채널 1이 함께 없어질 수 있음

> ✅ **한 줄 정리** · 모델 입력은 배치 축이 필요하므로 한 장은 `np.expand_dims(x, 0)`으로 `(1, ...)`을 만든다.

**딥러닝 코드에서는**

- `model(x.unsqueeze(0))`
- PyTorch의 `unsqueeze(0)` = `expand_dims(x, 0)` · 결과는 `out.squeeze(0)`

### 9-4. `A` 배치 쌓기 · `stack()` · `concatenate()` <sub>추가</sub>

> 🎯 **같은 모양의 샘플 여러 개를 배치로 묶을 때 `stack`과 `concatenate` 중 무엇을 쓰는가?**

💡 **쉽게 말하면**

- `stack`은 **종이를 위로 쌓아 묶음 만들기**(장 수라는 새 방향이 생김), `concatenate`는 **종이를 옆으로 이어 붙이기**(방향은 그대로, 길이만 늘어남).

**실행 흐름**

1. (3, 2, 4) 4장
2. stack (4, 3, 2, 4)
3. concatenate (12, 2, 4)
4. 배치 + 배치
5. shape 다른 stack 에러

**먼저 예측** · `(3, 2, 4)` 4장을 `concatenate`하면 shape는? — 답은 코드 실행 결과 아래에

In [31]:
# 값이 0 · 1 · 2 · 3으로 채워진 (3, 2, 4) 이미지 4장
imgs = [np.full((3, 2, 4), i) for i in range(4)]  # 값 0 ~ 3인 4장


# stack: 새 축(0번)을 만들며 쌓기 → 배치
batch = np.stack(imgs)                                                 # 새 0번 축으로 쌓기
print("stack:", batch.shape, "| batch[2]의 값:", np.unique(batch[2]))  # 장 번호 = 0번 축


# concatenate: 기존 축(0번)을 따라 이어 붙이기 → 채널 12개로 섞임
joined = np.concatenate(imgs)        # 기존 0번 축으로 잇기
print("concatenate:", joined.shape)  # 채널이 12로 섞임


# concatenate가 맞는 경우: 배치 + 배치
more = np.concatenate([batch, batch[:2]])  # (4, ...) + (2, ...) → (6, ...)
print("배치 합치기:", more.shape)          # 4 + 2 = 6장


# stack은 shape가 모두 같아야 함
try:
    np.stack([imgs[0], np.zeros((3, 2, 5))])  # 크기가 다른 두 장
except ValueError as e:                       # 모양 불일치 에러
    print("ValueError:", e)                   # 에러 문장 그대로

stack: (4, 3, 2, 4) | batch[2]의 값: [2]
concatenate: (12, 2, 4)
배치 합치기: (6, 3, 2, 4)
ValueError: all input arrays must have the same shape


**결과** · 예측 정답은 `(12, 2, 4)` — 새 축이 없어 채널 축이 12로 늘어남

- `stack: (4, 3, 2, 4) | batch[2]의 값: [2]` — 새 0번 축 = 장 번호, 2번 장은 전부 2
- `concatenate: (12, 2, 4)` — 4장의 채널이 한 축에 섞임 → 배치가 아님
- `배치 합치기: (6, 3, 2, 4)` — 이미 배치인 것끼리는 concatenate (4 + 2)
- `ValueError: all input arrays must have the same shape` — stack은 모든 shape가 같아야 함

**차이 · 언제 쓰나**

<table width="100%" style="width:100%; table-layout:auto">
<thead>
<tr>
<th style="text-align: left;">방법</th>
<th style="text-align: left;">차이</th>
<th style="text-align: left;">언제 쓰나</th>
</tr>
</thead>
<tbody>
<tr>
<td style="text-align: left;"><code>np.stack(목록)</code></td>
<td style="text-align: left;"><strong>새 축</strong>을 만들며 쌓음 <code>(N, ...)</code></td>
<td style="text-align: left;">같은 shape 샘플 → 배치 (기본 선택)</td>
</tr>
<tr>
<td style="text-align: left;"><code>np.concatenate(목록)</code></td>
<td style="text-align: left;"><strong>기존 축</strong>을 따라 이어 붙임</td>
<td style="text-align: left;">배치 + 배치, 특성 열 합치기</td>
</tr>
<tr>
<td style="text-align: left;"><code>np.array(목록)</code></td>
<td style="text-align: left;"><code>stack(axis=0)</code>과 같은 결과</td>
<td style="text-align: left;">리스트를 한 번에 배열로</td>
</tr>
<tr>
<td style="text-align: left;"><code>np.vstack</code> · <code>np.hstack</code></td>
<td style="text-align: left;">2차원 기준 세로 · 가로로 붙이기</td>
<td style="text-align: left;">표에 행 · 열 추가</td>
</tr>
</tbody>
</table>

> ⚠️ **주의**
>
> - 이미지 크기가 하나라도 다르면 stack 에러
> - 묶기 전에 크기 통일 (resize)

> ✅ **한 줄 정리** · 샘플을 배치로 묶을 땐 새 축을 만드는 `stack`, 배치끼리 이어 붙일 땐 `concatenate`를 쓴다.

**딥러닝 코드에서는**

- `DataLoader`가 샘플을 `torch.stack`으로 묶어 배치를 만듦
- 배치별 예측 모으기는 `np.concatenate(all_preds)`

### 9-5. `A` 재현 가능한 난수 · `default_rng()` <sub>추가</sub>

> 🎯 **실행할 때마다 같은 난수 · 같은 데이터 분할을 얻으려면?**

💡 **쉽게 말하면**

- seed는 **카드 섞기 시작 위치**.
- 같은 위치에서 같은 방법으로 섞으면 언제나 같은 순서가 나옴.

**실행 흐름**

1. seed 42 정수
2. 다시 42로 같은 결과
3. 가짜 이미지 · 정규분포
4. permutation으로 분할

**먼저 예측** · seed 42로 두 번 만든 정수 5개는 같을까? — 답은 코드 실행 결과 아래에

In [32]:
# seed가 같으면 언제 실행해도 같은 난수
rng = np.random.default_rng(42)                                        # 이 작업 전용 난수 생성기
print("seed 42 :", rng.integers(0, 10, size=5))                        # 0 이상 10 미만 정수 5개
print("다시 42 :", np.random.default_rng(42).integers(0, 10, size=5))  # 새 생성기도 같은 값


# 가짜 이미지 배치 · 정규분포 값
fake = rng.integers(0, 256, size=(4, 2, 4, 3), dtype=np.uint8)  # (N, H, W, C)
noise = rng.normal(0, 1, size=1000)                             # 평균 0 · 표준편차 1
print("가짜 배치:", fake.shape, fake.dtype,                     # (N, H, W, C) 확인
      "| noise", noise.size, "개 평균:", np.round(noise.mean(), 2))


# 학습 · 검증 나누기: 번호를 섞어서 자르기
idx = np.random.default_rng(0).permutation(10)  # 0~9를 섞은 순서 (원본 번호는 그대로)
train_idx, val_idx = idx[:8], idx[8:]           # 8 : 2로 자르기
print("섞인 번호:", idx, "| train", train_idx.size, "/ val", val_idx.size)  # 분할 개수

seed 42 :

 [0 7 6 4 4]
다시 42 : [0 7 6 4 4]
가짜 배치: (4, 2, 4, 3) uint8 | noise 1000 개 평균: -0.03
섞인 번호: [4 6 2 7 3 5 9 0 8 1] | train 8 / val 2


**결과** · 예측 정답은 같음 — 같은 seed, 같은 순서의 요청

- `seed 42 : [0 7 6 4 4]` — 0 이상 10 미만 정수 5개
- `다시 42 : [0 7 6 4 4]` — 새 생성기도 seed가 같으면 같은 결과
- `가짜 배치: (4, 2, 4, 3) uint8 | noise 1000 개 평균: -0.03` — (N, H, W, C) 가짜 이미지, 정규분포 1000개의 평균은 0에 가까움
- `섞인 번호: [4 6 2 7 3 5 9 0 8 1] | train 8 / val 2` — 섞은 번호를 8 : 2로 자름 (01 문법 8장과 같은 방식)

**차이 · 언제 쓰나**

<table width="100%" style="width:100%; table-layout:auto">
<thead>
<tr>
<th style="text-align: left;">방법</th>
<th style="text-align: left;">차이</th>
<th style="text-align: left;">언제 쓰나</th>
</tr>
</thead>
<tbody>
<tr>
<td style="text-align: left;"><code>np.random.default_rng(seed)</code></td>
<td style="text-align: left;"><strong>이 작업 전용</strong> 생성기, 다른 코드 영향 X</td>
<td style="text-align: left;">새 코드 (기본 선택)</td>
</tr>
<tr>
<td style="text-align: left;"><code>np.random.seed()</code> + <code>np.random.rand()</code></td>
<td style="text-align: left;"><strong>전역</strong> 난수 (옛 방식)</td>
<td style="text-align: left;">오래된 코드를 읽을 때</td>
</tr>
<tr>
<td style="text-align: left;"><code>rng.permutation(n)</code></td>
<td style="text-align: left;">섞인 <strong>새 배열</strong> 반환</td>
<td style="text-align: left;">번호를 섞어 학습 · 검증 나누기</td>
</tr>
<tr>
<td style="text-align: left;"><code>rng.shuffle(arr)</code></td>
<td style="text-align: left;"><strong>원본을 직접</strong> 섞음, 반환 <code>None</code></td>
<td style="text-align: left;">원본 순서가 필요 없을 때</td>
</tr>
</tbody>
</table>

> ⚠️ **주의**
>
> - 같은 생성기에서 뽑는 **순서**가 바뀌면 결과도 바뀜
> - 분할용 생성기는 따로 (`default_rng(0)`)

> ✅ **한 줄 정리** · `np.random.default_rng(seed)`로 전용 생성기를 만들면 언제 실행해도 같은 난수와 같은 분할을 얻는다.

**딥러닝 코드에서는**

- `torch.manual_seed(42)`와 `np.random.default_rng(42)`를 함께 고정 → 실험 재현 · `X[perm[:n_train]]`로 분할

### 9-6. `A` 시계열 슬라이딩 윈도우 · `sliding_window_view()` <sub>추가</sub>

> 🎯 **시계열 한 줄을 "과거 n개 → 다음 값" 학습 데이터로 바꾸려면?**

💡 **쉽게 말하면**

- **돋보기 창을 한 칸씩 밀며** 3일치를 보고, 바로 다음 날을 정답으로 적는 것.

<img src="images/np_window.png" alt="창을 한 칸씩 밀어 입력과 정답 만들기" width="728">

**실행 흐름**

1. 10일 기온
2. 반복문 창 + 정답
3. sliding_window_view
4. 마지막 창 제외
5. (샘플, 시간, 특성)
6. reshape와 비교

**먼저 예측** · 길이 10을 창 3으로 자르면 (입력, 정답) 쌍은 몇 개? — 답은 코드 실행 결과 아래에

In [33]:
# 10일 기온 → 3일 보고 다음 날 맞히기
temps10 = np.array([20, 22, 21, 23, 25, 24, 26, 28, 27, 29])  # 10일 기온
window = 3                                                    # 3일씩 보고 예측


# 방법 1: 반복문 + 슬라이싱으로 창 만들기
X_loop = np.array([temps10[i:i + window]                    # 창마다 3개씩
                   for i in range(len(temps10) - window)])  # 창 7개
y = temps10[window:]                                        # 각 창 바로 다음 값 = 정답
print("X:", X_loop.shape, "| y:", y.shape, "| 첫 창:", X_loop[0], "→ 정답", y[0])  # 입력 · 정답 짝


# 방법 2: sliding_window_view (반복문 없이)
windows = sliding_window_view(temps10, window)     # 가능한 창 전부 → (8, 3)
X = windows[:-1]                                   # 마지막 창은 다음 값(정답)이 없음 → 제외
print("전체 창:", windows.shape, "→ X:", X.shape,  # 두 방법 비교
      "| 두 방법 같음:", np.array_equal(X, X_loop))


# RNN 입력 모양: (샘플, 시간, 특성)
X_rnn = np.expand_dims(X, -1).astype(np.float32)  # 특성 축 추가 → float32
print("RNN 입력:", X_rnn.shape, X_rnn.dtype)      # 3축 · float32


# reshape(-1, 3)은 겹치지 않게 자를 뿐 (창과 다름)
print("reshape 둘째 조각:", temps10[:9].reshape(-1, 3)[1], "| 창 둘째:", windows[1])  # 겹침 차이

X: (7, 3) | y: (7,) | 첫 창: [20 22 21] → 정답 23
전체 창: (8, 3) → X: (7, 3) | 두 방법 같음: True
RNN 입력: (7, 3, 1) float32
reshape 둘째 조각: [23 25 24] | 창 둘째: [22 21 23]


**결과** · 예측 정답은 `7`개 — 10 - 3 = 7

- `X: (7, 3) | y: (7,) | 첫 창: [20 22 21] → 정답 23` — 1~3일 → 4일
- `전체 창: (8, 3) → X: (7, 3) | 두 방법 같음: True` — 가능한 창 8개 중 정답이 없는 마지막 1개 제외
- `RNN 입력: (7, 3, 1) float32` — (샘플 7, 시간 3, 특성 1)
- `reshape 둘째 조각: [23 25 24] | 창 둘째: [22 21 23]` — reshape는 3칸씩 건너뜀, 창은 1칸씩 겹쳐 이동

**차이 · 언제 쓰나**

<table width="100%" style="width:100%; table-layout:auto">
<thead>
<tr>
<th style="text-align: left;">방법</th>
<th style="text-align: left;">차이</th>
<th style="text-align: left;">언제 쓰나</th>
</tr>
</thead>
<tbody>
<tr>
<td style="text-align: left;">반복문 + 슬라이싱</td>
<td style="text-align: left;">이해 쉬움, 데이터가 크면 느림</td>
<td style="text-align: left;">처음 구조를 확인할 때</td>
</tr>
<tr>
<td style="text-align: left;"><code>sliding_window_view(x, w)</code></td>
<td style="text-align: left;">반복문 없이 <strong>모든 창</strong> (뷰 · 읽기 전용)</td>
<td style="text-align: left;">긴 시계열 (기본 선택)</td>
</tr>
<tr>
<td style="text-align: left;"><code>x.reshape(-1, w)</code></td>
<td style="text-align: left;"><strong>겹치지 않게</strong> 자름 (창이 아님)</td>
<td style="text-align: left;">일 단위 → 주 단위 묶기</td>
</tr>
<tr>
<td style="text-align: left;"><code>np.expand_dims(X, -1)</code></td>
<td style="text-align: left;">마지막에 특성 축 추가</td>
<td style="text-align: left;">RNN 입력 <code>(샘플, 시간, 특성)</code></td>
</tr>
</tbody>
</table>

> ⚠️ **주의**
>
> - 학습 · 검증은 **시간 순서대로** 나눔(앞 = 학습, 뒤 = 검증)
> - 섞으면 미래 값으로 과거를 맞히는 셈

> ✅ **한 줄 정리** · 시계열은 창을 한 칸씩 밀어 `X = 과거 n개`, `y = 다음 값`을 만들고 `(샘플, 시간, 특성)` 모양으로 넣는다.

**딥러닝 코드에서는**

- `nn.LSTM(..., batch_first=True)` 입력이 `(배치, 시간, 특성)`
- 정규화 평균 · 표준편차는 **학습 구간에서만** 계산

---
## 그래서 오늘 무엇을 배웠는가?

<table width="100%" style="width:100%; table-layout:auto">
<thead>
<tr>
<th style="text-align: left;">학습 목표</th>
<th style="text-align: left;">할 수 있게 된 것</th>
<th style="text-align: left;">확인 위치</th>
</tr>
</thead>
<tbody>
<tr>
<td style="text-align: left;">1. 배열과 shape · dtype</td>
<td style="text-align: left;"><code>list * 2</code>는 6개 · 배열은 3개, <code>sold</code>는 <code>(3, 4)</code> · 문자 섞이면 <code>&lt;U21</code></td>
<td style="text-align: left;">1-1 · 1-2 · 1-3</td>
</tr>
<tr>
<td style="text-align: left;">2. 뷰와 복사</td>
<td style="text-align: left;">reshape한 쪽을 고치면 <code>base[0]</code>이 99, 슬라이스를 고치면 원본도 2000</td>
<td style="text-align: left;">2-3 · 3-3</td>
</tr>
<tr>
<td style="text-align: left;">3. 조건 · 순위로 꺼내기</td>
<td style="text-align: left;"><code>&amp;</code>로 6개월, <code>argsort()[::-1][:3]</code>로 하윤 · 서연 · 지호</td>
<td style="text-align: left;">4-2 · 6-3</td>
</tr>
<tr>
<td style="text-align: left;">4. 결과 shape 미리 말하기</td>
<td style="text-align: left;"><code>axis=0</code> → <code>(4,)</code>, <code>(3, 4) + (3, 1)</code> → <code>(3, 4)</code>, <code>(2, 3) @ (3, 2)</code> → <code>(2, 2)</code></td>
<td style="text-align: left;">6-4 · 7-1 · 7-2</td>
</tr>
<tr>
<td style="text-align: left;">5. 딥러닝 입력 만들기</td>
<td style="text-align: left;"><code>(2, 4, 3)</code> uint8 → <code>(1, 3, 2, 4)</code> float32, 10일 기온 → <code>(7, 3, 1)</code></td>
<td style="text-align: left;">9장</td>
</tr>
</tbody>
</table>

### 주제별 한 줄 · 반드시 기억할 것

<table width="100%" style="width:100%; table-layout:auto">
<thead>
<tr>
<th style="text-align: center;">등급</th>
<th style="text-align: left;">주제</th>
<th style="text-align: left;">반드시 기억할 것</th>
<th style="text-align: left;">딥러닝 코드에서</th>
</tr>
</thead>
<tbody>
<tr>
<td style="text-align: center;"><code>A</code></td>
<td style="text-align: left;">리스트 vs 배열</td>
<td style="text-align: left;">계산은 배열, 연산은 원소마다</td>
<td style="text-align: left;"><code>np.array(data, dtype=np.float32)</code></td>
</tr>
<tr>
<td style="text-align: center;"><code>A</code></td>
<td style="text-align: left;">shape · dtype</td>
<td style="text-align: left;">계산 전에 항상 <code>shape</code> · <code>dtype</code> 출력</td>
<td style="text-align: left;"><code>print(x.shape, x.dtype)</code></td>
</tr>
<tr>
<td style="text-align: center;"><code>A</code></td>
<td style="text-align: left;">astype</td>
<td style="text-align: left;"><code>astype(int)</code>는 버림</td>
<td style="text-align: left;"><code>labels.astype(np.int64)</code></td>
</tr>
<tr>
<td style="text-align: center;"><code>A</code></td>
<td style="text-align: left;">arange · reshape</td>
<td style="text-align: left;">stop 미포함, size가 같아야, <code>-1</code> 자동</td>
<td style="text-align: left;"><code>x.reshape(len(x), -1)</code></td>
</tr>
<tr>
<td style="text-align: center;"><code>A</code></td>
<td style="text-align: left;">인덱싱 · 슬라이싱</td>
<td style="text-align: left;">0부터, stop 미포함, <strong>슬라이싱은 뷰</strong></td>
<td style="text-align: left;"><code>X[:800]</code>, <code>x.copy()</code></td>
</tr>
<tr>
<td style="text-align: center;"><code>A</code></td>
<td style="text-align: left;">2차원</td>
<td style="text-align: left;"><code>[행, 열]</code>, 숫자는 축 제거 · 슬라이스는 유지</td>
<td style="text-align: left;"><code>img[:, 10:50, 10:50]</code></td>
</tr>
<tr>
<td style="text-align: center;"><code>A</code></td>
<td style="text-align: left;">mask</td>
<td style="text-align: left;">조건 → <code>a[mask]</code>, <code>&amp;</code> · <code>~</code> + 괄호</td>
<td style="text-align: left;"><code>X[y == 1]</code></td>
</tr>
<tr>
<td style="text-align: center;"><code>A</code></td>
<td style="text-align: left;">통계 · argmax</td>
<td style="text-align: left;"><code>arg</code> = 위치</td>
<td style="text-align: left;"><code>logits.argmax(axis=1)</code></td>
</tr>
<tr>
<td style="text-align: center;"><code>A</code></td>
<td style="text-align: left;">fancy · argsort</td>
<td style="text-align: left;">위치 목록을 다른 배열에도 적용</td>
<td style="text-align: left;"><code>names[probs.argsort()[::-1][:5]]</code></td>
</tr>
<tr>
<td style="text-align: center;"><code>A</code></td>
<td style="text-align: left;">axis</td>
<td style="text-align: left;">적은 축이 사라짐, 결과 개수로 판단</td>
<td style="text-align: left;"><code>x.mean(axis=0)</code></td>
</tr>
<tr>
<td style="text-align: center;"><code>A</code></td>
<td style="text-align: left;">broadcasting · 행렬 곱</td>
<td style="text-align: left;">오른쪽 끝부터 같거나 1, <code>@</code>는 안쪽 일치</td>
<td style="text-align: left;"><code>X @ W + b</code></td>
</tr>
<tr>
<td style="text-align: center;"><code>A</code></td>
<td style="text-align: left;">종합</td>
<td style="text-align: left;">원소별 계산 → mask → argsort · argmin</td>
<td style="text-align: left;"><code>(pred == y).mean()</code></td>
</tr>
<tr>
<td style="text-align: center;"><code>A</code></td>
<td style="text-align: left;">이미지 입력</td>
<td style="text-align: left;">float32 ÷ 255 → <code>transpose(2, 0, 1)</code> → <code>expand_dims</code></td>
<td style="text-align: left;"><code>ToTensor()</code>, <code>unsqueeze(0)</code></td>
</tr>
<tr>
<td style="text-align: center;"><code>A</code></td>
<td style="text-align: left;">배치 · 난수</td>
<td style="text-align: left;">샘플은 <code>stack</code>, seed 고정 <code>default_rng</code></td>
<td style="text-align: left;"><code>DataLoader</code>, <code>manual_seed</code></td>
</tr>
<tr>
<td style="text-align: center;"><code>A</code></td>
<td style="text-align: left;">시계열</td>
<td style="text-align: left;">창 n개 → 다음 값, <code>(샘플, 시간, 특성)</code></td>
<td style="text-align: left;"><code>nn.LSTM(batch_first=True)</code></td>
</tr>
<tr>
<td style="text-align: center;"><code>B</code></td>
<td style="text-align: left;">zeros · ones</td>
<td style="text-align: left;">shape는 괄호로 묶기</td>
<td style="text-align: left;"><code>np.zeros(len(test_set))</code></td>
</tr>
<tr>
<td style="text-align: center;"><code>B</code></td>
<td style="text-align: left;">빈 결과</td>
<td style="text-align: left;"><code>size == 0</code> 확인, nan 주의</td>
<td style="text-align: left;"><code>np.isnan(loss)</code></td>
</tr>
<tr>
<td style="text-align: center;"><code>B</code></td>
<td style="text-align: left;">정렬</td>
<td style="text-align: left;"><code>np.sort</code> 새 배열 · <code>.sort()</code>는 None</td>
<td style="text-align: left;"><code>np.sort(scores)[::-1]</code></td>
</tr>
</tbody>
</table>

---
## 기억 확인 문제

위 내용을 **보지 않고** 답한 뒤, ✅ 한 줄 정리와 비교해 ○ (정확) · △ (일부) · × (모름)를 표시. △ · ×인 섹션만 다시 보기.

### 1. 질문에 답하기

<table width="100%" style="width:100%; table-layout:auto">
<thead>
<tr>
<th style="text-align: left;">섹션</th>
<th style="text-align: left;">🎯 질문</th>
</tr>
</thead>
<tbody>
<tr>
<td style="text-align: left;">1-1</td>
<td style="text-align: left;">리스트에 <code>* 2</code>를 하면 왜 가격이 2배가 되지 않는가?</td>
</tr>
<tr>
<td style="text-align: left;">1-2</td>
<td style="text-align: left;"><code>shape</code> · <code>size</code> · <code>len</code>은 무엇이 다른가?</td>
</tr>
<tr>
<td style="text-align: left;">1-4</td>
<td style="text-align: left;"><code>astype(int)</code>는 반올림인가, 버림인가?</td>
</tr>
<tr>
<td style="text-align: left;">2-3</td>
<td style="text-align: left;"><code>reshape</code>한 배열을 고치면 원본은 어떻게 되는가?</td>
</tr>
<tr>
<td style="text-align: left;">3-3</td>
<td style="text-align: left;">슬라이싱한 배열을 고치면 원본도 바뀌는가?</td>
</tr>
<tr>
<td style="text-align: left;">3-5</td>
<td style="text-align: left;"><code>sold[:, 3]</code>과 <code>sold[:, 3:4]</code>의 shape는 왜 다른가?</td>
</tr>
<tr>
<td style="text-align: left;">4-2</td>
<td style="text-align: left;">배열 조건을 묶을 때 왜 <code>and</code> 대신 <code>&amp;</code>를 쓰고 괄호를 치는가?</td>
</tr>
<tr>
<td style="text-align: left;">5-1</td>
<td style="text-align: left;"><code>max()</code>와 <code>argmax()</code>는 무엇이 다른가?</td>
</tr>
<tr>
<td style="text-align: left;">6-2</td>
<td style="text-align: left;"><code>np.sort()</code>와 <code>.sort()</code>는 무엇이 다른가?</td>
</tr>
<tr>
<td style="text-align: left;">6-3</td>
<td style="text-align: left;">점수 순서로 이름까지 정렬하려면?</td>
</tr>
<tr>
<td style="text-align: left;">6-4</td>
<td style="text-align: left;">행마다 합계를 내려면 axis를 몇으로 하는가?</td>
</tr>
<tr>
<td style="text-align: left;">7-1</td>
<td style="text-align: left;">broadcasting은 언제 성공하는가?</td>
</tr>
<tr>
<td style="text-align: left;">9-2</td>
<td style="text-align: left;">HWC → CHW를 <code>reshape</code>로 바꾸면 왜 안 되는가?</td>
</tr>
<tr>
<td style="text-align: left;">9-4</td>
<td style="text-align: left;">샘플을 배치로 묶을 때 <code>stack</code>과 <code>concatenate</code> 중 무엇을 쓰는가?</td>
</tr>
<tr>
<td style="text-align: left;">9-6</td>
<td style="text-align: left;">시계열을 학습 데이터로 바꾸는 방법은?</td>
</tr>
</tbody>
</table>

#### 내 답 · 이 셀을 더블클릭해서 작성

**1-1** · (여기에 답 작성) · ○ △ × 중 하나만 남기기

**1-2** · (여기에 답 작성) · ○ △ × 중 하나만 남기기

**1-4** · (여기에 답 작성) · ○ △ × 중 하나만 남기기

**2-3** · (여기에 답 작성) · ○ △ × 중 하나만 남기기

**3-3** · (여기에 답 작성) · ○ △ × 중 하나만 남기기

**3-5** · (여기에 답 작성) · ○ △ × 중 하나만 남기기

**4-2** · (여기에 답 작성) · ○ △ × 중 하나만 남기기

**5-1** · (여기에 답 작성) · ○ △ × 중 하나만 남기기

**6-2** · (여기에 답 작성) · ○ △ × 중 하나만 남기기

**6-3** · (여기에 답 작성) · ○ △ × 중 하나만 남기기

**6-4** · (여기에 답 작성) · ○ △ × 중 하나만 남기기

**7-1** · (여기에 답 작성) · ○ △ × 중 하나만 남기기

**9-2** · (여기에 답 작성) · ○ △ × 중 하나만 남기기

**9-4** · (여기에 답 작성) · ○ △ × 중 하나만 남기기

**9-6** · (여기에 답 작성) · ○ △ × 중 하나만 남기기

### 2. 코드로 답하기

강의자료 미니 실습을 그대로 가져온 문제. 아래 빈 셀에 직접 작성 후 실행.

1. `np.arange`로 **4부터 16까지 3씩** 증가하는 배열, `np.zeros`로 **3행 2열 정수** 0 배열 만들기
2. `bitcoin = np.array([970, 1180, 1072, 1348, 2286, 2481, 2875, 4703, 4339, 6468, 10234, 14156])`에서 사람이 세는 **7~9번째** 값 꺼내기
3. 1부터 100까지 정수에서 **짝수만** 골라 앞 5개와 개수 출력
4. `sales = np.array([[10, 12, 14, 16], [20, 18, 21, 25], [11, 13, 15, 17]])`(행 서울 · 부산 · 대전, 열 Q1 ~ Q4)에서 **서울 · 부산의 Q2 · Q3** 직사각형과 shape
5. `names = np.array(["John", "Mike", "Sarah", "Kate", "Samuel"])`, `scores = np.array([78, 95, 84, 98, 88])`에서 점수 **상위 3명**의 이름
6. (9장) seed 0으로 만든 `(32, 32, 3)` uint8 가짜 이미지를 모델 입력 **`(1, 3, 32, 32)` float32 · 0~1**로 바꾸기

In [34]:
# 직접 작성 · 코드 문제

### 3. 변형 문제 · 조건을 바꿔 다시 풀기

1. `np.arange(30).reshape(-1, 6)`의 shape를 먼저 말하고, `reshape(4, -1)`은 왜 실패하는지 확인
2. `sold`에서 시간대별 **평균**이 가장 높은 시간대 이름을 `timeslots`와 `argmax`로 구하기 (axis는?)
3. `x = np.arange(15).reshape(5, 3)`에서 **열 평균**을 빼면 shape는? **행 평균**을 빼려면 무엇이 필요한가?
4. 9-6에서 `window`를 **4**로 바꾸면 `X` · `y` · RNN 입력의 shape는?

In [35]:
# 직접 작성 · 변형 문제

### 4. 모범 답안 · 1 ~ 3을 다 푼 뒤에 보기

#### 질문 답

<table width="100%" style="width:100%; table-layout:auto">
<thead>
<tr>
<th style="text-align: left;">섹션</th>
<th style="text-align: left;">답</th>
</tr>
</thead>
<tbody>
<tr>
<td style="text-align: left;">1-1</td>
<td style="text-align: left;">리스트의 <code>*</code>는 이어 붙이기(반복)라서. 배열은 원소마다 곱함</td>
</tr>
<tr>
<td style="text-align: left;">1-2</td>
<td style="text-align: left;"><code>shape</code> 모양 · <code>size</code> 총 개수 · <code>len</code> 첫 축 길이</td>
</tr>
<tr>
<td style="text-align: left;">1-4</td>
<td style="text-align: left;">버림. 반올림은 <code>np.round()</code> 후 <code>astype(int)</code></td>
</tr>
<tr>
<td style="text-align: left;">2-3</td>
<td style="text-align: left;">원본 shape는 그대로지만 데이터를 공유해 원본 값도 바뀜. 끊으려면 <code>.copy()</code></td>
</tr>
<tr>
<td style="text-align: left;">3-3</td>
<td style="text-align: left;">바뀜(뷰). 원본을 지키려면 <code>.copy()</code></td>
</tr>
<tr>
<td style="text-align: left;">3-5</td>
<td style="text-align: left;">숫자 하나로 고른 축은 사라지고(<code>(3,)</code>), 슬라이스로 고른 축은 남음(<code>(3, 1)</code>)</td>
</tr>
<tr>
<td style="text-align: left;">4-2</td>
<td style="text-align: left;"><code>and</code>는 값 하나만 판단해 배열이면 에러, <code>&amp;</code>가 비교보다 먼저 계산되므로 괄호 필수</td>
</tr>
<tr>
<td style="text-align: left;">5-1</td>
<td style="text-align: left;"><code>max()</code>는 값, <code>argmax()</code>는 위치</td>
</tr>
<tr>
<td style="text-align: left;">6-2</td>
<td style="text-align: left;"><code>np.sort()</code>는 새 배열, <code>.sort()</code>는 원본 정렬 + <code>None</code> 반환</td>
</tr>
<tr>
<td style="text-align: left;">6-3</td>
<td style="text-align: left;"><code>np.argsort()</code>로 위치를 정렬해 이름 배열에도 적용 (내림차순은 <code>[::-1]</code>)</td>
</tr>
<tr>
<td style="text-align: left;">6-4</td>
<td style="text-align: left;"><code>axis=1</code> (적은 축이 사라짐 → 행마다 하나)</td>
</tr>
<tr>
<td style="text-align: left;">7-1</td>
<td style="text-align: left;">shape를 오른쪽 끝부터 비교해 각 축이 같거나 1일 때</td>
</tr>
<tr>
<td style="text-align: left;">9-2</td>
<td style="text-align: left;"><code>reshape</code>는 값을 순서대로 다시 채워 그림이 뒤섞임. <code>transpose(2, 0, 1)</code></td>
</tr>
<tr>
<td style="text-align: left;">9-4</td>
<td style="text-align: left;"><code>stack</code> (새 배치 축). 배치끼리 이어 붙일 때만 <code>concatenate</code></td>
</tr>
<tr>
<td style="text-align: left;">9-6</td>
<td style="text-align: left;">창을 한 칸씩 밀어 과거 n개 = 입력, 다음 값 = 정답 → <code>(샘플, 시간, 특성)</code>, 시간 순서대로 분할</td>
</tr>
</tbody>
</table>

#### 코드 문제 답

```python
# 1. arange · zeros
print(np.arange(4, 17, 3))  # [ 4  7 10 13 16]
print(np.zeros((3, 2), dtype=int).shape)  # (3, 2)

# 2. 사람 기준 7~9번째 → 위치 6 ~ 8 → [6:9]
bitcoin = np.array([970, 1180, 1072, 1348, 2286, 2481,
                    2875, 4703, 4339, 6468, 10234, 14156])
print(bitcoin[6:9])  # [2875 4703 4339]

# 3. 짝수 = 2로 나눈 나머지가 0
numbers = np.arange(1, 101)
evens = numbers[numbers % 2 == 0]
print(evens[:5], evens.size)  # [ 2  4  6  8 10] 50

# 4. 행 0:2 (서울 · 부산) · 열 1:3 (Q2 · Q3)
sales = np.array([[10, 12, 14, 16], [20, 18, 21, 25], [11, 13, 15, 17]])
rect = sales[0:2, 1:3]
print(rect.tolist(), rect.shape)  # [[12, 14], [18, 21]] (2, 2)

# 5. argsort → 뒤집기 → 앞 3개 → 이름에 적용
names = np.array(["John", "Mike", "Sarah", "Kate", "Samuel"])
scores = np.array([78, 95, 84, 98, 88])
print(names[np.argsort(scores)[::-1][:3]])  # ['Kate' 'Mike' 'Samuel']

# 6. float32 ÷ 255 → HWC → CHW → 배치 축
rng = np.random.default_rng(0)
img32 = rng.integers(0, 256, size=(32, 32, 3), dtype=np.uint8)
x = np.expand_dims((img32.astype(np.float32) / 255).transpose(2, 0, 1), 0)
print(x.shape, x.dtype)  # (1, 3, 32, 32) float32
print(x.min() >= 0, x.max() <= 1)  # True True
```

#### 변형 문제 답

<table width="100%" style="width:100%; table-layout:auto">
<thead>
<tr>
<th style="text-align: left;">문제</th>
<th style="text-align: left;">답</th>
</tr>
</thead>
<tbody>
<tr>
<td style="text-align: left;">1</td>
<td style="text-align: left;"><code>(5, 6)</code> · 30은 4로 나누어떨어지지 않아 <code>ValueError</code></td>
</tr>
<tr>
<td style="text-align: left;">2</td>
<td style="text-align: left;"><code>sold.mean(axis=0)</code> = <code>[42.33 88.33 67.33 28.33]</code> → <code>argmax()</code> 1 → <code>'점심'</code> (열마다 → <code>axis=0</code>)</td>
</tr>
<tr>
<td style="text-align: left;">3</td>
<td style="text-align: left;"><code>x - x.mean(axis=0)</code> → <code>(5, 3)</code> (끝자리 3 = 3) · 행 평균은 <code>keepdims=True</code>로 <code>(5, 1)</code>을 만들어야 함</td>
</tr>
<tr>
<td style="text-align: left;">4</td>
<td style="text-align: left;">창 10 - 4 = 6개 → <code>X</code> <code>(6, 4)</code> · <code>y</code> <code>(6,)</code> · RNN 입력 <code>(6, 4, 1)</code></td>
</tr>
</tbody>
</table>

---
## 오답 노트

틀린 문제는 정답을 베끼지 말고 **원인**을 적음. 다음 복습 때 이 표의 "재발 방지 질문"부터 스스로에게 물어보기.

<table width="100%" style="width:100%; table-layout:auto">
<thead>
<tr>
<th style="text-align: left;">틀린 내용</th>
<th style="text-align: left;">틀린 이유</th>
<th style="text-align: left;">올바른 원칙</th>
<th style="text-align: left;">재발 방지 질문</th>
</tr>
</thead>
<tbody>
<tr>
<td style="text-align: left;">(예) <code>sold[:, 3]</code>을 <code>(3, 1)</code>이라고 답함</td>
<td style="text-align: left;">슬라이스와 숫자 인덱스를 구분 안 함</td>
<td style="text-align: left;">숫자로 고른 축은 사라짐</td>
<td style="text-align: left;">이 축을 숫자로 골랐나, 범위로 골랐나?</td>
</tr>
<tr>
<td style="text-align: left;"></td>
<td style="text-align: left;"></td>
<td style="text-align: left;"></td>
<td style="text-align: left;"></td>
</tr>
<tr>
<td style="text-align: left;"></td>
<td style="text-align: left;"></td>
<td style="text-align: left;"></td>
<td style="text-align: left;"></td>
</tr>
</tbody>
</table>

---
## 면접 30초 답변

**Q1. Python 리스트 대신 NumPy 배열을 쓰는 이유는?**
> - 배열은 한 가지 자료형의 숫자를 연속된 메모리에 담아, 반복문 없이 원소별 계산과 통계를 한 줄로 빠르게 처리합니다.
> - 리스트의 `* 2`가 이어 붙이기인 것과 달리 배열은 모든 원소에 곱하기가 적용되고, Pandas와 PyTorch 텐서도 같은 구조라서 데이터 처리의 기본이 됩니다.

**Q2. NumPy에서 뷰와 복사의 차이는?**
> - 슬라이싱과 `reshape`는 데이터를 복사하지 않고 원본을 보는 뷰를 돌려주므로, 결과를 고치면 원본도 바뀝니다.
> - 반면 fancy indexing과 boolean mask는 복사본을 만듭니다.
> - 원본을 보존해야 하는 전처리에서는 `.copy()`를 명시합니다.

**Q3. broadcasting 규칙과 shape 오류가 났을 때 대처는?**
> 두 shape를 오른쪽 끝부터 비교해 각 축의 길이가 같거나 한쪽이 1이면, 길이 1인 쪽이 늘어나 계산됩니다. 오류가 나면 먼저 두 배열의 shape를 출력하고, `reshape(-1, 1)`이나 `keepdims=True`, `expand_dims`로 방향을 맞춥니다.

**Q4. 이미지를 PyTorch 모델에 넣기 전에 NumPy에서 어떤 변환을 하나요?**
> 파일에서 읽은 이미지는 `(H, W, C)` uint8이므로 `astype(np.float32) / 255`로 0~1 실수로 바꾸고, `transpose(2, 0, 1)`로 `(C, H, W)`가 되게 축 순서를 바꾼 뒤, 한 장이면 `expand_dims`로 배치 축을 붙입니다. 축 순서를 `reshape`로 바꾸면 값이 뒤섞이므로 반드시 `transpose`를 씁니다.

**Q5. 시계열 데이터를 학습 데이터로 만드는 방법은?**
> - 길이 n의 창을 한 칸씩 밀면서 과거 n개를 입력, 바로 다음 값을 정답으로 만들고 `(샘플, 시간, 특성)` 모양으로 맞춥니다.
> - 학습과 검증은 시간 순서대로 나누고, 정규화 통계도 학습 구간에서만 계산해 미래 정보가 섞이지 않게 합니다.

---
## 최종 요약표

<table width="100%" style="width:100%; table-layout:auto">
<thead>
<tr>
<th style="text-align: left;">함수·문법</th>
<th style="text-align: left;">소속</th>
<th style="text-align: left;">역할</th>
<th style="text-align: left;">코드에서 사용한 위치</th>
</tr>
</thead>
<tbody>
<tr>
<td style="text-align: left;"><code>np.array()</code></td>
<td style="text-align: left;">numpy</td>
<td style="text-align: left;">리스트 → 배열</td>
<td style="text-align: left;">1-1 <code>price_array = np.array(price_list)</code></td>
</tr>
<tr>
<td style="text-align: left;"><code>.ndim</code> · <code>.shape</code> · <code>.size</code></td>
<td style="text-align: left;">ndarray 속성</td>
<td style="text-align: left;">축 개수 · 모양 · 총 개수</td>
<td style="text-align: left;">1-2 <code>arr.ndim</code> · <code>arr.shape</code> · <code>arr.size</code></td>
</tr>
<tr>
<td style="text-align: left;"><code>.dtype</code> · <code>dtype=</code></td>
<td style="text-align: left;">ndarray 속성 · 생성 인자</td>
<td style="text-align: left;">자료형 확인 · 지정</td>
<td style="text-align: left;">1-3 <code>print(arr, &quot;→&quot;, arr.dtype)</code></td>
</tr>
<tr>
<td style="text-align: left;"><code>.astype()</code></td>
<td style="text-align: left;">ndarray 메서드</td>
<td style="text-align: left;">자료형을 바꾼 새 배열</td>
<td style="text-align: left;">1-4 <code>temps_int = temps.astype(int)</code></td>
</tr>
<tr>
<td style="text-align: left;"><code>np.round()</code> · <code>np.floor()</code></td>
<td style="text-align: left;">numpy</td>
<td style="text-align: left;">반올림 · 내림</td>
<td style="text-align: left;">1-4 <code>print(&quot;반올림 후 변환:&quot;, np.round(temps).astype(int))</code></td>
</tr>
<tr>
<td style="text-align: left;"><code>len()</code> · <code>type()</code></td>
<td style="text-align: left;">Python 내장 함수</td>
<td style="text-align: left;">첫 축 길이 · 자료형</td>
<td style="text-align: left;">1-2 <code>len(arr)</code> · 1-1 <code>type(price_list).__name__</code></td>
</tr>
<tr>
<td style="text-align: left;"><code>np.arange()</code> · <code>np.linspace()</code></td>
<td style="text-align: left;">numpy</td>
<td style="text-align: left;">간격 · 개수로 수열 만들기</td>
<td style="text-align: left;">2-1 <code>seq_a = np.arange(10)</code></td>
</tr>
<tr>
<td style="text-align: left;"><code>np.zeros()</code> · <code>np.ones()</code> · <code>np.full()</code> · <code>np.zeros_like()</code></td>
<td style="text-align: left;">numpy</td>
<td style="text-align: left;">값을 채운 배열</td>
<td style="text-align: left;">2-2 <code>booking = np.zeros((4, 3), dtype=int)</code></td>
</tr>
<tr>
<td style="text-align: left;"><code>.reshape()</code> · <code>-1</code></td>
<td style="text-align: left;">ndarray 메서드</td>
<td style="text-align: left;">모양 바꾸기 (데이터 공유) · 자동 계산</td>
<td style="text-align: left;">2-3 <code>r34 = base.reshape(3, 4)</code></td>
</tr>
<tr>
<td style="text-align: left;"><code>.copy()</code> · <code>.flatten()</code> · <code>.ravel()</code></td>
<td style="text-align: left;">ndarray 메서드</td>
<td style="text-align: left;">복사 · 1차원 펴기</td>
<td style="text-align: left;">2-3 <code>r_copy = base.reshape(3, 4).copy()</code></td>
</tr>
<tr>
<td style="text-align: left;"><code>np.shares_memory()</code></td>
<td style="text-align: left;">numpy</td>
<td style="text-align: left;">데이터 공유 여부 확인</td>
<td style="text-align: left;">2-3 <code>np.shares_memory(base, r34)</code></td>
</tr>
<tr>
<td style="text-align: left;"><code>a[i]</code> · <code>a[-i]</code></td>
<td style="text-align: left;">Python 기본 문법</td>
<td style="text-align: left;">위치로 값 하나</td>
<td style="text-align: left;">3-1 <code>visitors[6]</code> · <code>visitors[-1]</code></td>
</tr>
<tr>
<td style="text-align: left;"><code>a[start:stop:step]</code></td>
<td style="text-align: left;">Python 기본 문법</td>
<td style="text-align: left;">구간 꺼내기 (NumPy는 뷰)</td>
<td style="text-align: left;">3-2 <code>visitors[2:5]</code> · <code>visitors[::2]</code></td>
</tr>
<tr>
<td style="text-align: left;"><code>a[행, 열]</code> · <code>a[:, j]</code></td>
<td style="text-align: left;">NumPy 인덱싱</td>
<td style="text-align: left;">2차원 칸 · 행 · 열</td>
<td style="text-align: left;">3-4 <code>sold[1, 2]</code> · 3-5 <code>sold[:, 3]</code></td>
</tr>
<tr>
<td style="text-align: left;"><code>.copy()</code> · <code>.base</code></td>
<td style="text-align: left;">ndarray 메서드 · 속성</td>
<td style="text-align: left;">복사 · 뷰의 원본 확인</td>
<td style="text-align: left;">3-3 <code>safe = visitors.copy()</code> · <code>view.base is visitors</code></td>
</tr>
<tr>
<td style="text-align: left;"><code>.tolist()</code></td>
<td style="text-align: left;">ndarray 메서드</td>
<td style="text-align: left;">배열 → 리스트</td>
<td style="text-align: left;">3-3 <code>lst = visitors.tolist()</code></td>
</tr>
<tr>
<td style="text-align: left;"><code>&gt;</code> · <code>&gt;=</code> · <code>==</code></td>
<td style="text-align: left;">비교 연산자</td>
<td style="text-align: left;">원소별 True/False 배열</td>
<td style="text-align: left;">4-1 <code>mask = visitors &gt; 2400</code></td>
</tr>
<tr>
<td style="text-align: left;"><code>a[mask]</code></td>
<td style="text-align: left;">NumPy 인덱싱</td>
<td style="text-align: left;">True 위치의 값</td>
<td style="text-align: left;">4-1 <code>months[mask]</code> · <code>visitors[mask]</code></td>
</tr>
<tr>
<td style="text-align: left;"><code>&amp;</code> · <code>&#124;</code> · <code>~</code></td>
<td style="text-align: left;">NumPy 연산자</td>
<td style="text-align: left;">원소별 그리고 · 또는 · 반대</td>
<td style="text-align: left;">4-2 <code>(visitors &gt;= 2000) &amp; (visitors &lt;= 2500)</code> · <code>(visitors &lt; 1900) &#124; (visitors &gt; 2800)</code> · <code>~(visitors &gt; 2400)</code></td>
</tr>
<tr>
<td style="text-align: left;"><code>np.all()</code> · <code>np.any()</code> · <code>np.where()</code></td>
<td style="text-align: left;">numpy</td>
<td style="text-align: left;">전부인가 · 하나라도인가 · 위치</td>
<td style="text-align: left;">4-1 <code>np.all(visitors[mask] &gt; 2400)</code> · <code>np.where(mask)[0]</code></td>
</tr>
<tr>
<td style="text-align: left;"><code>warnings.catch_warnings()</code></td>
<td style="text-align: left;">warnings</td>
<td style="text-align: left;">경고를 잡아 확인</td>
<td style="text-align: left;">4-3 <code>with warnings.catch_warnings(record=True) as caught:</code></td>
</tr>
<tr>
<td style="text-align: left;"><code>.sum()</code> · <code>.mean()</code> · <code>.std()</code> · <code>.min()</code> · <code>.max()</code></td>
<td style="text-align: left;">ndarray 메서드</td>
<td style="text-align: left;">기본 통계</td>
<td style="text-align: left;">5-1 <code>visitors.sum()</code> · <code>visitors.mean()</code> · <code>visitors.std()</code></td>
</tr>
<tr>
<td style="text-align: left;"><code>.argmax()</code> · <code>.argmin()</code></td>
<td style="text-align: left;">ndarray 메서드</td>
<td style="text-align: left;">최대 · 최소의 위치</td>
<td style="text-align: left;">5-1 <code>peak = visitors.argmax()</code></td>
</tr>
<tr>
<td style="text-align: left;"><code>+</code> · <code>-</code> · <code>*</code> · <code>/</code> (배열끼리 · 숫자)</td>
<td style="text-align: left;">NumPy 연산</td>
<td style="text-align: left;">원소별 계산</td>
<td style="text-align: left;">5-2 <code>revenue = price * cups</code> · <code>revenue / total * 100</code></td>
</tr>
<tr>
<td style="text-align: left;"><code>zip()</code> · <code>sum()</code></td>
<td style="text-align: left;">Python 내장 함수</td>
<td style="text-align: left;">리스트 방식 비교 · 행끼리 합 (배열엔 X)</td>
<td style="text-align: left;">5-2 <code>zip(price.tolist(), cups.tolist())</code> · 5-1 <code>sum(sold)</code></td>
</tr>
<tr>
<td style="text-align: left;"><code>a[[i, j, k]]</code></td>
<td style="text-align: left;">NumPy 인덱싱 (fancy)</td>
<td style="text-align: left;">위치 목록으로 고르기 (복사)</td>
<td style="text-align: left;">6-1 <code>visitors[[8, 2, 5]]</code></td>
</tr>
<tr>
<td style="text-align: left;"><code>np.sort()</code> · <code>.sort()</code></td>
<td style="text-align: left;">numpy · ndarray 메서드</td>
<td style="text-align: left;">새 배열로 정렬 · 원본 정렬</td>
<td style="text-align: left;">6-2 <code>np.sort(made)</code> · <code>copy_made.sort()</code></td>
</tr>
<tr>
<td style="text-align: left;"><code>np.argsort()</code></td>
<td style="text-align: left;">numpy</td>
<td style="text-align: left;">정렬 순서대로의 위치</td>
<td style="text-align: left;">6-3 <code>order = np.argsort(made)</code></td>
</tr>
<tr>
<td style="text-align: left;"><code>axis=</code> · <code>keepdims=</code></td>
<td style="text-align: left;">통계 함수 인자</td>
<td style="text-align: left;">계산 방향 · 축 유지</td>
<td style="text-align: left;">6-4 <code>sold.sum(axis=0)</code> · <code>sold.sum(axis=1, keepdims=True)</code></td>
</tr>
<tr>
<td style="text-align: left;">broadcasting</td>
<td style="text-align: left;">NumPy 규칙</td>
<td style="text-align: left;">다른 shape를 자동으로 맞춤 (오른쪽 끝부터 같거나 1)</td>
<td style="text-align: left;">7-1 <code>sold + column_bonus</code> (<code>(3, 4) + (3, 1)</code>)</td>
</tr>
<tr>
<td style="text-align: left;"><code>np.dot()</code> · <code>@</code> · <code>np.matmul()</code></td>
<td style="text-align: left;">numpy · 연산자</td>
<td style="text-align: left;">내적 · 행렬 곱</td>
<td style="text-align: left;">7-2 <code>np.dot(price_vec, cups_vec)</code> · <code>sales_count @ price_cost</code></td>
</tr>
<tr>
<td style="text-align: left;"><code>.T</code></td>
<td style="text-align: left;">ndarray 속성</td>
<td style="text-align: left;">전치 (모든 축 역순)</td>
<td style="text-align: left;">7-2 <code>sales_count.T.shape</code></td>
</tr>
<tr>
<td style="text-align: left;"><code>.astype(np.float32)</code> · <code>/ 255</code></td>
<td style="text-align: left;">ndarray 메서드 · 연산</td>
<td style="text-align: left;">uint8 → 0~1 실수</td>
<td style="text-align: left;">9-1 <code>x = img.astype(np.float32) / 255</code></td>
</tr>
<tr>
<td style="text-align: left;"><code>.transpose()</code> · <code>np.moveaxis()</code></td>
<td style="text-align: left;">ndarray 메서드 · numpy</td>
<td style="text-align: left;">축 순서 바꾸기 (HWC ↔ CHW)</td>
<td style="text-align: left;">9-2 <code>chw = img.transpose(2, 0, 1)</code></td>
</tr>
<tr>
<td style="text-align: left;"><code>np.expand_dims()</code> · <code>[None]</code> · <code>.squeeze()</code></td>
<td style="text-align: left;">numpy · 인덱싱 · ndarray 메서드</td>
<td style="text-align: left;">길이 1 축 추가 · 제거</td>
<td style="text-align: left;">9-3 <code>np.expand_dims(chw, axis=0)</code> · <code>chw[None]</code> · <code>batch1.squeeze(0)</code></td>
</tr>
<tr>
<td style="text-align: left;"><code>np.stack()</code> · <code>np.concatenate()</code></td>
<td style="text-align: left;">numpy</td>
<td style="text-align: left;">새 축으로 쌓기 · 기존 축으로 잇기</td>
<td style="text-align: left;">9-4 <code>batch = np.stack(imgs)</code></td>
</tr>
<tr>
<td style="text-align: left;"><code>np.random.default_rng()</code> · <code>.integers()</code> · <code>.normal()</code> · <code>.permutation()</code></td>
<td style="text-align: left;">numpy.random</td>
<td style="text-align: left;">재현 가능한 난수 · 번호 섞기</td>
<td style="text-align: left;">9-5 <code>rng.integers(0, 10, size=5)</code> · <code>rng.normal(0, 1, size=1000)</code> · <code>.permutation(10)</code></td>
</tr>
<tr>
<td style="text-align: left;"><code>sliding_window_view()</code></td>
<td style="text-align: left;">numpy.lib.stride_tricks</td>
<td style="text-align: left;">시계열 창 만들기</td>
<td style="text-align: left;">9-6 <code>windows = sliding_window_view(temps10, window)</code></td>
</tr>
<tr>
<td style="text-align: left;"><code>np.array_equal()</code> · <code>np.allclose()</code> · <code>np.unique()</code></td>
<td style="text-align: left;">numpy</td>
<td style="text-align: left;">같은가 · 거의 같은가 · 고유값</td>
<td style="text-align: left;">9-2 <code>np.array_equal(chw[0], img[:, :, 0])</code> · 9-1 <code>np.allclose(...)</code> · 9-4 <code>np.unique(batch[2])</code></td>
</tr>
</tbody>
</table>

### 할 것 · 하지 말 것

<table width="100%" style="width:100%; table-layout:auto">
<thead>
<tr>
<th style="text-align: left;">✅ 할 것</th>
<th style="text-align: left;">⚠️ 하지 말 것</th>
</tr>
</thead>
<tbody>
<tr>
<td style="text-align: left;">계산 전 <code>shape</code> · <code>dtype</code> 출력</td>
<td style="text-align: left;"><code>a = a.sort()</code></td>
</tr>
<tr>
<td style="text-align: left;">원본을 지킬 땐 <code>.copy()</code></td>
<td style="text-align: left;">슬라이싱 결과를 원본 걱정 없이 수정</td>
</tr>
<tr>
<td style="text-align: left;">조건마다 괄호 + <code>&amp;</code></td>
<td style="text-align: left;">배열에 <code>and</code> · <code>or</code></td>
</tr>
<tr>
<td style="text-align: left;">이미지는 <code>astype(np.float32)</code> 먼저</td>
<td style="text-align: left;">uint8 상태로 더하기 · 빼기</td>
</tr>
<tr>
<td style="text-align: left;">축 순서는 <code>transpose</code></td>
<td style="text-align: left;">축 순서를 <code>reshape</code>로 바꾸기</td>
</tr>
<tr>
<td style="text-align: left;">난수는 <code>default_rng(seed)</code></td>
<td style="text-align: left;">시계열을 무작위로 섞어 분할</td>
</tr>
</tbody>
</table>

### 헷갈리는 짝 한눈에 보기

<table width="100%" style="width:100%; table-layout:auto">
<thead>
<tr>
<th style="text-align: left;">짝</th>
<th style="text-align: left;">차이</th>
<th style="text-align: left;">결론</th>
</tr>
</thead>
<tbody>
<tr>
<td style="text-align: left;"><code>list * 2</code> vs <code>array * 2</code></td>
<td style="text-align: left;">이어 붙이기 vs 원소별 계산</td>
<td style="text-align: left;">계산은 배열</td>
</tr>
<tr>
<td style="text-align: left;"><code>shape</code> vs <code>size</code> vs <code>len</code></td>
<td style="text-align: left;">모양 vs 총 개수 vs 첫 축 길이</td>
<td style="text-align: left;">모양 확인은 <code>shape</code></td>
</tr>
<tr>
<td style="text-align: left;"><code>astype(int)</code> vs <code>np.round</code></td>
<td style="text-align: left;">버림 vs 반올림</td>
<td style="text-align: left;">반올림이 필요하면 <code>np.round</code> 먼저</td>
</tr>
<tr>
<td style="text-align: left;"><code>arange</code> vs <code>linspace</code></td>
<td style="text-align: left;">간격 지정 · stop 미포함 vs 개수 지정 · stop 포함</td>
<td style="text-align: left;">간격이면 <code>arange</code></td>
</tr>
<tr>
<td style="text-align: left;"><code>reshape</code> vs 원본</td>
<td style="text-align: left;">새 모양 · 데이터 공유 vs 원래 모양</td>
<td style="text-align: left;">따로 고치려면 <code>.copy()</code></td>
</tr>
<tr>
<td style="text-align: left;">슬라이싱 vs <code>copy()</code></td>
<td style="text-align: left;">뷰(원본 연결) vs 독립 복사</td>
<td style="text-align: left;">원본 보호는 <code>copy()</code></td>
</tr>
<tr>
<td style="text-align: left;"><code>sold[1, 2]</code> vs <code>sold[1][2]</code></td>
<td style="text-align: left;">한 번에 vs 두 단계</td>
<td style="text-align: left;"><code>[행, 열]</code></td>
</tr>
<tr>
<td style="text-align: left;"><code>sold[:, 3]</code> vs <code>sold[:, 3:4]</code></td>
<td style="text-align: left;">축 사라짐 <code>(3,)</code> vs 유지 <code>(3, 1)</code></td>
<td style="text-align: left;">모양 유지는 슬라이스</td>
</tr>
<tr>
<td style="text-align: left;"><code>&amp;</code> vs <code>and</code></td>
<td style="text-align: left;">원소마다 vs 값 하나 (배열이면 에러)</td>
<td style="text-align: left;">배열은 <code>&amp;</code> + 괄호</td>
</tr>
<tr>
<td style="text-align: left;"><code>max</code> vs <code>argmax</code></td>
<td style="text-align: left;">값 vs 위치</td>
<td style="text-align: left;">다른 배열을 꺼낼 땐 <code>argmax</code></td>
</tr>
<tr>
<td style="text-align: left;"><code>np.sort</code> vs <code>.sort()</code></td>
<td style="text-align: left;">새 배열 vs 원본 정렬 + <code>None</code></td>
<td style="text-align: left;">보통 <code>np.sort</code></td>
</tr>
<tr>
<td style="text-align: left;"><code>sort</code> vs <code>argsort</code></td>
<td style="text-align: left;">값 vs 위치</td>
<td style="text-align: left;">이름과 함께 정렬은 <code>argsort</code></td>
</tr>
<tr>
<td style="text-align: left;"><code>axis=0</code> vs <code>axis=1</code></td>
<td style="text-align: left;">열마다 vs 행마다</td>
<td style="text-align: left;">결과 개수로 판단</td>
</tr>
<tr>
<td style="text-align: left;"><code>*</code> vs <code>@</code></td>
<td style="text-align: left;">원소별 곱 vs 행렬 곱</td>
<td style="text-align: left;">신경망 층은 <code>@</code></td>
</tr>
<tr>
<td style="text-align: left;"><code>transpose</code> vs <code>reshape</code></td>
<td style="text-align: left;">축 순서 변경 vs 모양만 변경</td>
<td style="text-align: left;">HWC → CHW는 <code>transpose</code></td>
</tr>
<tr>
<td style="text-align: left;"><code>stack</code> vs <code>concatenate</code></td>
<td style="text-align: left;">새 축 vs 기존 축</td>
<td style="text-align: left;">샘플 → 배치는 <code>stack</code></td>
</tr>
<tr>
<td style="text-align: left;"><code>default_rng</code> vs <code>np.random.seed</code></td>
<td style="text-align: left;">전용 생성기 vs 전역 난수</td>
<td style="text-align: left;"><code>default_rng</code></td>
</tr>
<tr>
<td style="text-align: left;"><code>sliding_window_view</code> vs <code>reshape(-1, w)</code></td>
<td style="text-align: left;">겹치는 창 vs 겹치지 않는 조각</td>
<td style="text-align: left;">시계열 입력은 창</td>
</tr>
</tbody>
</table>

---
## 다음 복습일

<table width="100%" style="width:100%; table-layout:auto">
<thead>
<tr>
<th style="text-align: left;">언제</th>
<th style="text-align: left;">할 일</th>
<th style="text-align: left;">걸리는 시간</th>
</tr>
</thead>
<tbody>
<tr>
<td style="text-align: left;">내일</td>
<td style="text-align: left;">기억 확인 문제 1을 자료 없이 다시 답하고 오답 노트 갱신</td>
<td style="text-align: left;">15분</td>
</tr>
<tr>
<td style="text-align: left;">3일 뒤</td>
<td style="text-align: left;">변형 문제 4개를 새로 풀기 · 그림(축 · broadcasting)을 가리고 결과 shape 말하기</td>
<td style="text-align: left;">20분</td>
</tr>
<tr>
<td style="text-align: left;">7일 뒤</td>
<td style="text-align: left;">8장 종합 실습을 빈 셀에 다시 짜기 · 9장 이미지 · 시계열 변환을 손으로 적고 실행</td>
<td style="text-align: left;">30분</td>
</tr>
<tr>
<td style="text-align: left;">14일 뒤</td>
<td style="text-align: left;">오답 노트에 남은 문제만 다시 풀기 · Pandas · PyTorch 전에 3 · 6 · 7장 ✅ 줄만 읽기</td>
<td style="text-align: left;">15분</td>
</tr>
</tbody>
</table>

---
## 부록

### 이 노트북을 만든 방법

- 뼈대 · NumPy 강의자료 Part 0 ~ Final Challenge
- 예제 · 같은 Part 구조의 연습문제 27문제를 **중복 없이** 7개 대단원 · 24개 섹션에 넣음
- 더한 것 · 8장 종합 실습 · 9장 딥러닝 데이터로 연결(6개 섹션)
- 목적 · 이후 Pandas · PyTorch 코드의 배열 · 텐서를 읽기 위한 최소 NumPy

### 읽는 법

<table width="100%" style="width:100%; table-layout:auto">
<thead>
<tr>
<th style="text-align: left;">표시</th>
<th style="text-align: left;">의미</th>
</tr>
</thead>
<tbody>
<tr>
<td style="text-align: left;"><code>A</code> · <code>B</code></td>
<td style="text-align: left;"><strong>A</strong> 딥러닝 코드에 바로 나오는 필수 · <strong>B</strong> 알아두면 되는 것</td>
</tr>
<tr>
<td style="text-align: left;">🎯</td>
<td style="text-align: left;">이번 섹션에서 답해야 할 질문</td>
</tr>
<tr>
<td style="text-align: left;">💡</td>
<td style="text-align: left;">쉬운 설명 · 비유</td>
</tr>
<tr>
<td style="text-align: left;">⚠️</td>
<td style="text-align: left;">자주 하는 실수</td>
</tr>
<tr>
<td style="text-align: left;">✅</td>
<td style="text-align: left;">기억할 결론 = 🎯 질문의 답</td>
</tr>
</tbody>
</table>

**섹션 구성** · 🎯 질문 → 💡 쉬운 설명 → 실행 흐름 → 먼저 예측 → 코드 → 결과(예측 정답 · 출력 줄) → 차이 · 언제 쓰나 → ⚠️ 주의 → ✅ 한 줄 정리

**코드 주석** · 코드 위 = 이 단계에서 하는 일, 코드 옆 = 그 줄이 왜 필요한가 (줄마다 · 열 맞춤)

### 원본 → 섹션 대응표

<table width="100%" style="width:100%; table-layout:auto">
<thead>
<tr>
<th style="text-align: left;">강의자료</th>
<th style="text-align: left;">연습문제</th>
<th style="text-align: left;">섹션</th>
</tr>
</thead>
<tbody>
<tr>
<td style="text-align: left;">Part 0 리스트와 배열</td>
<td style="text-align: left;">01 · 02</td>
<td style="text-align: left;">1-1 · 1-2</td>
</tr>
<tr>
<td style="text-align: left;">Part 1 dtype</td>
<td style="text-align: left;">03 · 04</td>
<td style="text-align: left;">1-3 · 1-4</td>
</tr>
<tr>
<td style="text-align: left;">Part 2 arange · zeros · ones</td>
<td style="text-align: left;">05 · 06</td>
<td style="text-align: left;">2-1 · 2-2 (강의 미니 실습 → 10-2의 1번)</td>
</tr>
<tr>
<td style="text-align: left;">Part 3 reshape</td>
<td style="text-align: left;">07 · 08 + 09</td>
<td style="text-align: left;">2-3 · 2-4 (<strong>통합</strong>: 실패 원인과 <code>-1</code> 자동 계산)</td>
</tr>
<tr>
<td style="text-align: left;">Part 4 1차원 인덱싱 · 슬라이싱</td>
<td style="text-align: left;">10 · 11 · 12</td>
<td style="text-align: left;">3-1 ~ 3-3 (bitcoin 미니 실습 → 10-2의 2번)</td>
</tr>
<tr>
<td style="text-align: left;">Part 5 2차원</td>
<td style="text-align: left;">13 · 14 + 15</td>
<td style="text-align: left;">3-4 · 3-5 (<strong>통합</strong>: 직사각형과 행 · 열 전체, 매출표 미니 실습 → 10-2의 4번)</td>
</tr>
<tr>
<td style="text-align: left;">Part 6 조건</td>
<td style="text-align: left;">16 · 17 · 18</td>
<td style="text-align: left;">4-1 ~ 4-3 (짝수 미니 실습 → 10-2의 3번)</td>
</tr>
<tr>
<td style="text-align: left;">Part 7 계산</td>
<td style="text-align: left;">19 · 20</td>
<td style="text-align: left;">5-1 · 5-2 (Part 0의 처음 질문을 5-2에서 회수)</td>
</tr>
<tr>
<td style="text-align: left;">Support</td>
<td style="text-align: left;">21 · 22 · 23 · 24</td>
<td style="text-align: left;">6-1 ~ 6-4 (상위 3명 미니 실습 → 10-2의 5번)</td>
</tr>
<tr>
<td style="text-align: left;">Preview</td>
<td style="text-align: left;">25 · 26</td>
<td style="text-align: left;">7-1 · 7-2 (3차원 cube 포함)</td>
</tr>
<tr>
<td style="text-align: left;">Final Challenge</td>
<td style="text-align: left;">27</td>
<td style="text-align: left;">8 (<strong>통합</strong>: 강의 4상품 문제와 같은 단계라 연습문제 1개로)</td>
</tr>
<tr>
<td style="text-align: left;">(추가)</td>
<td style="text-align: left;">—</td>
<td style="text-align: left;">9-1 ~ 9-6 딥러닝 데이터 (PyTorch 없이 실행)</td>
</tr>
</tbody>
</table>

강의의 GDP · 지역 매출 예시는 같은 개념의 연습문제 데이터(카페)로 **한 번만** 다룸.

### 전체 지도

#### 단원 연결 · 왜 이 순서로 배우는가

<table width="100%" style="width:100%; table-layout:auto">
<thead>
<tr>
<th style="text-align: center;">순서</th>
<th style="text-align: left;">배우는 이유</th>
<th style="text-align: left;">대단원</th>
</tr>
</thead>
<tbody>
<tr>
<td style="text-align: center;">1</td>
<td style="text-align: left;">리스트로는 숫자 계산에 반복문이 필요하다</td>
<td style="text-align: left;">배열 기본</td>
</tr>
<tr>
<td style="text-align: center;">2</td>
<td style="text-align: left;">배열을 직접 다 적지 않고 만들고, 모양을 바꾼다</td>
<td style="text-align: left;">만들기 · 모양 바꾸기</td>
</tr>
<tr>
<td style="text-align: center;">3</td>
<td style="text-align: left;">만든 배열에서 원하는 위치를 꺼낸다</td>
<td style="text-align: left;">위치로 꺼내기</td>
</tr>
<tr>
<td style="text-align: center;">4</td>
<td style="text-align: left;">위치 대신 조건으로 꺼낸다</td>
<td style="text-align: left;">조건으로 꺼내기</td>
</tr>
<tr>
<td style="text-align: center;">5</td>
<td style="text-align: left;">꺼낸 값을 반복문 없이 계산한다</td>
<td style="text-align: left;">한꺼번에 계산</td>
</tr>
<tr>
<td style="text-align: center;">6</td>
<td style="text-align: left;">순위를 매기고 행 · 열 방향을 정해 계산한다</td>
<td style="text-align: left;">고르기 · 정렬 · 축</td>
</tr>
<tr>
<td style="text-align: center;">7</td>
<td style="text-align: left;">모양이 다른 배열끼리 · 행렬끼리 계산한다</td>
<td style="text-align: left;">shape 맞추기</td>
</tr>
<tr>
<td style="text-align: center;">8</td>
<td style="text-align: left;">1~7을 한 문제에 모두 쓴다</td>
<td style="text-align: left;">종합 실습</td>
</tr>
<tr>
<td style="text-align: center;">9</td>
<td style="text-align: left;">이미지 · 시계열을 모델 입력으로 바꾼다</td>
<td style="text-align: left;">딥러닝 데이터로 연결</td>
</tr>
</tbody>
</table>

#### ML/DL 공통 파이프라인 · 각 대단원이 쓰이는 곳

<table width="100%" style="width:100%; table-layout:auto">
<thead>
<tr>
<th style="text-align: left;">단계</th>
<th style="text-align: left;">하는 일</th>
<th style="text-align: left;">쓰이는 장</th>
</tr>
</thead>
<tbody>
<tr>
<td style="text-align: left;">① 데이터 준비</td>
<td style="text-align: left;">이미지 · 시계열을 배열로 읽기</td>
<td style="text-align: left;">1장 · <strong>9장</strong></td>
</tr>
<tr>
<td style="text-align: left;">② 불러오기</td>
<td style="text-align: left;">번호 섞기 · 배치 고르기</td>
<td style="text-align: left;">2장 · 6장 · <strong>9장</strong></td>
</tr>
<tr>
<td style="text-align: left;">③ 전처리</td>
<td style="text-align: left;">자료형 · 모양 맞추기, 자르기, 조건 필터, 정규화</td>
<td style="text-align: left;">1장 ~ 5장 · <strong>9장</strong></td>
</tr>
<tr>
<td style="text-align: left;">④ 모델</td>
<td style="text-align: left;">행렬 곱 <code>X @ W + b</code>, 입력 shape</td>
<td style="text-align: left;">7장 · <strong>9장</strong></td>
</tr>
<tr>
<td style="text-align: left;">⑤ 학습</td>
<td style="text-align: left;">학습 · 검증 나누기, 배치 묶기</td>
<td style="text-align: left;">3장 · <strong>9장</strong></td>
</tr>
<tr>
<td style="text-align: left;">⑥ 평가</td>
<td style="text-align: left;">예측 클래스(argmax), 정확도, Top-k</td>
<td style="text-align: left;">4장 ~ 6장 · 8장</td>
</tr>
<tr>
<td style="text-align: left;">⑦ 저장 · 기록</td>
<td style="text-align: left;">결과 배열 모으기</td>
<td style="text-align: left;">9장 (<code>concatenate</code>)</td>
</tr>
</tbody>
</table>

**굵은 9장** · 이미지 · 시계열 프로젝트의 ① ~ ⑤ 를 NumPy만으로 연결

### 전체 코드

1 ~ 9장 핵심 코드를 장별 셀로 모음. **위에서부터 이 셀들만 실행해도 전체가 동작**하며, 각 출력 앞 `[번호]`가 섹션.

In [36]:
# ============================================================
# NumPy 전체 정리 — 전체 코드
# ============================================================
import numpy as np                                       # 배열 계산 라이브러리
from numpy.lib.stride_tricks import sliding_window_view  # 시계열 창 (9-6)

In [37]:
# ---------- 1. 배열 기본 ----------
# 1-1 리스트 * 2 는 반복, 배열 * 2 는 원소별 계산
price_list = [4500, 5000, 5500]                           # 가격 3개 (리스트)
print("[1-1]", price_list * 2, np.array(price_list) * 2)  # 반복 vs 원소별

# 1-2 ndim · shape · size
cups = np.array([320, 180, 145, 60, 95])                                  # 메뉴별 잔 수
sold = np.array([[42, 88, 61, 25], [30, 75, 94, 48], [55, 102, 47, 12]])  # 지점 × 시간대
print("[1-2]", sold.ndim, sold.shape, sold.size)                          # 축 · 모양 · 개수

# 1-3 dtype은 정수 → 실수 → 문자열 방향으로 통일
print("[1-3]", np.array([10, 20, 30.5]).dtype, np.array([10, 20, "30"]).dtype)  # 자료형 통일

# 1-4 astype(int)은 버림, 반올림은 np.round 먼저
temps = np.array([21.7, 24.3, 19.8, 26.5])                      # 실수 온도
print("[1-4]", temps.astype(int), np.round(temps).astype(int))  # 버림 vs 반올림

[1-1] [4500, 5000, 5500, 4500, 5000, 5500] [ 9000 10000 11000]
[1-2] 2 (3, 4) 12
[1-3] float64 <U21
[1-4] [21 24 19 26] [22 24 20 26]


In [38]:
# ---------- 2. 만들기 · 모양 바꾸기 ----------
# 2-1 arange는 stop 미포함 · linspace는 개수 지정, stop 포함
print("[2-1]", np.arange(5, 51, 5)[-1], np.linspace(0, 1, 5))

# 2-2 zeros · ones: shape는 괄호로 묶어 전달
print("[2-2]", np.zeros((4, 3), dtype=int).shape, np.ones((2, 5)).dtype)  # 빈 표

# 2-3 reshape: 모양만 바꾸고 데이터는 공유
base = np.arange(12)                      # 값 12개
print("[2-3]", base.reshape(3, 4).shape,  # 모양 · 공유
      np.shares_memory(base, base.reshape(3, 4)))

# 2-4 -1은 한 자리만 자동 계산
print("[2-4]", np.arange(24).reshape(-1, 6).shape)  # -1 자동 계산

[2-1] 50 [0.   0.25 0.5  0.75 1.  ]
[2-2] (4, 3) float64
[2-3] (3, 4) True
[2-4] (4, 6)


In [39]:
# ---------- 3. 위치로 꺼내기 ----------
months = np.arange(1, 13)                                 # 1 ~ 12월
visitors = np.array([1820, 1650, 2010, 2240, 2380, 2510,  # 월별 방문자
                     2760, 2890, 2430, 2180, 1970, 2300])

# 3-1 인덱싱: 0부터, 뒤는 -1부터
print("[3-1]", visitors[0], visitors[6], visitors[-1])  # 1월 · 7월 · 12월

# 3-2 슬라이싱: stop 미포함
print("[3-2]", visitors[2:5], visitors[::2])  # 구간 · 간격

# 3-3 슬라이싱은 뷰 → 원본 보호는 copy()
print("[3-3]", np.shares_memory(visitors, visitors[:3]),  # 뷰 vs 복사
      np.shares_memory(visitors, visitors[:3].copy()))

# 3-4 2차원 한 칸: [행, 열]
print("[3-4]", sold[1, 2], sold[2, 1], sold[0, 3])  # [행, 열]

# 3-5 숫자 하나로 고른 축은 사라지고, 슬라이스로 고른 축은 남음
print("[3-5]", sold[1:3, 1:3].shape, sold[:, 3].shape, sold[:, 3:4].shape)  # 축 유지

[3-1] 1820 2760 2300
[3-2] [2010 2240 2380] [1820 2010 2380 2760 2430 1970]
[3-3] True False
[3-4] 94 102 25
[3-5] (2, 2) (3,) (3, 1)


In [40]:
# ---------- 4. 조건으로 꺼내기 ----------
# 4-1 mask: 조건 → True/False → 대괄호
mask = visitors > 2400                        # 조건 → True/False
print("[4-1]", months[mask], visitors[mask])  # 같은 mask로 두 배열

# 4-2 & · | (각 조건은 괄호로)
print("[4-2]", visitors[(visitors >= 2000) & (visitors <= 2500)].size,  # 조건 묶기
      visitors[(visitors < 1900) | (visitors > 2800)])

# 4-3 결과가 비면 size 0
print("[4-3]", visitors[visitors > 5000].size)  # 빈 결과 = 0

[4-1] [6 7 8 9] [2510 2760 2890 2430]
[4-2] 6 [1820 1650 2890]
[4-3] 0


In [41]:
# ---------- 5. 한꺼번에 계산 ----------
# 5-1 통계 · argmax(위치)
print("[5-1]", visitors.sum(), np.round(visitors.mean(), 1),  # 통계 · 위치
      months[visitors.argmax()])

# 5-2 같은 위치끼리 계산
price = np.array([4500, 5000, 5500, 5300, 6000])  # 메뉴별 가격
revenue = price * cups                            # 같은 위치끼리 곱
print("[5-2]", revenue, revenue.sum())            # 메뉴별 · 전체

[5-1] 27140 2261.7 8
[5-2] [1440000  900000  797500  318000  570000] 4025500


In [42]:
# ---------- 6. 고르기 · 정렬 · 축 ----------
# 6-1 fancy indexing: 위치 배열로 고르기
print("[6-1]", visitors[[2, 5, 8]])  # 3 · 6 · 9월

# 6-2 np.sort는 새 배열 · 내림차순은 [::-1]
made = np.array([182, 240, 165, 258, 203])          # 제조 잔 수
print("[6-2]", np.sort(made), np.sort(made)[::-1])  # 오름 · 내림

# 6-3 argsort → 위치 → 이름에도 적용
baristas = np.array(["민준", "서연", "도윤", "하윤", "지호"])  # 같은 순서 이름
print("[6-3]", baristas[np.argsort(made)[::-1][:3]])           # 상위 3명

# 6-4 axis=0 열마다 · axis=1 행마다
print("[6-4]", sold.sum(axis=0), sold.sum(axis=1))  # 열마다 · 행마다

[6-1] [2010 2510 2430]
[6-2] [165 182 203 240 258] [258 240 203 182 165]
[6-3] ['하윤' '서연' '지호']
[6-4] [127 265 202  85] [216 247 216]


In [43]:
# ---------- 7. shape 맞추기 ----------
# 7-1 broadcasting: 오른쪽 끝부터 같거나 1
print("[7-1]", (sold + np.array([1, 2, 3, 4]))[0],  # (4,) · (3, 1) 늘리기
      (sold + np.array([10, 20, 30]).reshape(3, 1))[:, 0])

# 7-2 내적 · 행렬 곱 · 전치
sales_count = np.array([[10, 8, 6], [12, 5, 9]])                   # 지점 × 메뉴
price_cost = np.array([[4500, 1200], [5000, 1500], [5500, 1800]])  # 메뉴 × [가격, 원가]
print("[7-2]", np.dot([4500, 5000, 5500], [10, 8, 6]),             # 내적 · 행렬 곱
      (sales_count @ price_cost).tolist(), sales_count.T.shape)

[7-1] [43 90 64 29] [52 50 85]
[7-2] 118000 [[118000, 34800], [128500, 38100]] (3, 2)


In [44]:
# ---------- 8. 종합 실습: 신메뉴 수익 분석 ----------
new_menu = np.array(["말차라떼", "흑임자라떼", "딸기크림", "밤라떼"])  # 신메뉴 4개
unit_profit = (np.array([6000, 6500, 7000, 5800])                      # 판매가 - 원가
               - np.array([2200, 2600, 3100, 2000]))
total_profit = unit_profit * np.array([140, 95, 210, 60])              # × 판매량
print("[8]", total_profit.sum(), new_menu[total_profit >= 300000],     # 합계 · 조건 · 순위
      new_menu[np.argsort(total_profit)[::-1]])

[8] 1949500 ['말차라떼' '흑임자라떼' '딸기크림'] ['딸기크림' '말차라떼' '흑임자라떼' '밤라떼']


In [45]:
# ---------- 9. 딥러닝 데이터로 연결 ----------
# 9-1 uint8 → float32 → 0~1
img = np.random.default_rng(0).integers(0, 256, size=(2, 4, 3), dtype=np.uint8)
x = img.astype(np.float32) / 255                     # 0 ~ 1 실수
print("[9-1]", x.dtype, x.min() >= 0, x.max() <= 1)  # 범위 확인

# 9-2 HWC → CHW 는 transpose
chw = x.transpose(2, 0, 1)                                     # HWC → CHW
print("[9-2]", chw.shape, np.array_equal(chw[0], x[:, :, 0]))  # 값 보존 확인

# 9-3 배치 축 추가
print("[9-3]", np.expand_dims(chw, 0).shape)  # 배치 축

# 9-4 stack(새 축) vs concatenate(기존 축)
print("[9-4]", np.stack([chw, chw]).shape, np.concatenate([chw, chw]).shape)  # 새 축 vs 기존 축

# 9-5 seed 고정 난수 · 번호 섞어 나누기
idx = np.random.default_rng(0).permutation(10)  # 섞은 번호
print("[9-5]", idx[:8].size, idx[8:].size)      # 8 : 2

# 9-6 슬라이딩 윈도우 → (샘플, 시간, 특성)
temps10 = np.array([20, 22, 21, 23, 25, 24, 26, 28, 27, 29])    # 10일 기온
X = sliding_window_view(temps10, 3)[:-1]                        # 정답 있는 창만
print("[9-6]", np.expand_dims(X, -1).shape, temps10[3:].shape)  # 입력 · 정답 모양

[9-1] float32 True True
[9-2] (3, 2, 4) True
[9-3] (1, 3, 2, 4)
[9-4] (2, 3, 2, 4) (6, 2, 4)
[9-5] 8 2
[9-6] (7, 3, 1) (7,)
